# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your_api_key",        # Get from your team dashboard
    team_name="your_team_name"     # Your exact team name
)

# Iteration 1
- 483.6284

In [ ]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — Complete Solution
================================================================================

TREE OF THOUGHTS STRATEGIC ANALYSIS
====================================

STEP 1: Literature & Context Search Findings
----------------------------------------------
• Healthcare cost prediction is dominated by gradient boosted trees (XGBoost,
  LightGBM, CatBoost) — these consistently win on tabular regression tasks.
• MAE loss is critical: it optimizes for median-like predictions rather than 
  mean, making it more robust to the right-skewed cost distribution.
• The data is NOT zero-inflated (min target = $306.88), so zero-inflated 
  regression (Tweedie, hurdle models) is unnecessary.
• PDF extraction via pdfplumber is purely CPU-based — no GPU/OCR needed.

STEP 2: Three Strategic Approaches
------------------------------------
Strategy A — "Tabular Baseline" (GBM on CSV features only)
  + Simple, fast, no PDF dependency
  + Prior cost has 0.849 correlation with target; sqrt(cost) reaches 0.901
  + Cross-validated MAE estimate: ~650-750
  - Ignores rich CPT code information trapped in PDFs
  - No insurance/ZIP3 features (only available in PDFs + patients.csv)

Strategy B — "Multimodal Feature Engineer" (GBM with PDF-extracted features)
  + Extracts CPT codes → procedure type counts, severity indicators
  + Extracts ZIP3 and Insurance from PDF headers (bonus features!)
  + Feature space grows from 4 to ~30+ engineered features
  + Estimated MAE improvement: 5-15% over baseline
  - Requires all PDFs (time to process ~4000 files, but fast with pdfplumber)

Strategy C — "Deep Regressor" (TabTransformer / Neural Net)
  + Could learn non-linear interactions automatically
  - Only 2000 training samples → deep learning will likely overfit
  - 8GB VRAM is sufficient but unnecessary — GBMs dominate at this scale
  - Slower iteration cycle, harder to debug
  - Performance ceiling lower than well-engineered GBM at n=2000

STEP 3: Critique & Selection
-------------------------------
┌──────────────────┬────────────┬───────────────┬─────────────────────┐
│     Criterion    │ Strategy A │  Strategy B   │    Strategy C        │
├──────────────────┼────────────┼───────────────┼─────────────────────┤
│ Feasibility      │ ★★★★★      │ ★★★★★         │ ★★★☆☆               │
│ MAE Suitability  │ ★★★☆☆      │ ★★★★★         │ ★★★☆☆               │
│ Perf. Ceiling    │ ★★★☆☆      │ ★★★★★         │ ★★★☆☆               │
│ Robustness       │ ★★★★★      │ ★★★★☆         │ ★★☆☆☆               │
└──────────────────┴────────────┴───────────────┴─────────────────────┘

WINNER: Strategy B — Multimodal Feature Engineer
Rationale: With only 2000 samples, this is a FEATURE ENGINEERING competition.
The key insight is that sqrt(prior_cost) alone gives r=0.901 with target.
Adding CPT-derived severity features + insurance/ZIP will push this further.
An ensemble of LightGBM + XGBoost + CatBoost, all with MAE/L1 loss, will
be extremely competitive.

STEP 4: Execution Plan
-----------------------
1. PDF Pipeline: pdfplumber (CPU-only, ~0.1s per PDF)
   → Extract: CPT codes, descriptions, quantities, amounts, ZIP3, insurance
   → Engineer: unique_codes, total_line_items, max_single_charge, 
     procedure_category_counts, severity_score, cost_concentration (HHI)
     
2. Feature Engineering:
   - sqrt/log transforms of cost features (linearize the relationship)
   - cost_per_visit ratio
   - condition × cost interaction features  
   - CPT category aggregations (E&M codes, procedures, lab, critical care)
   
3. Model: Ensemble of 3 GBMs with MAE/L1 loss
   - LightGBM (objective='mae')
   - XGBoost (objective='reg:absoluteerror')
   - CatBoost (loss_function='MAE')
   Weighted average with weights tuned on OOF predictions.
   
4. Validation: 5-fold stratified CV (stratified by primary_chronic + cost bins)

================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings
import json
from pathlib import Path
from collections import Counter

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    SEED = 42
    N_FOLDS = 5
    # Set this to your actual receipts_pdf directory path
    # e.g., r"C:\Users\Eric\AgentDS\healthcare\receipts_pdf"
    RECEIPTS_DIR = None  # Will auto-detect
    
    # Paths (update for your local setup)
    TRAIN_PATH = "ed_cost_train.csv"
    TEST_PATH = "ed_cost_test.csv"
    PATIENTS_PATH = "patients.csv"  # Optional: if you have it
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# MODULE 1: PDF EXTRACTION PIPELINE
# ============================================================================
def extract_receipt_features(pdf_path):
    """
    Extract structured features from a single receipt PDF.
    Uses pdfplumber (CPU-only, no GPU needed).
    
    Returns dict with:
    - zip3, insurance (metadata from header)
    - CPT code features (counts, categories, amounts)
    """
    try:
        import pdfplumber
    except ImportError:
        print("WARNING: pdfplumber not installed. Run: pip install pdfplumber")
        return None
    
    features = {}
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text()
            if not text:
                return None
            
            lines = text.split('\n')
            
            # --- Extract header metadata ---
            for line in lines:
                if 'ZIP3' in line:
                    zip_match = re.search(r'ZIP3:\s*(\d+)', line)
                    ins_match = re.search(r'Insurance:\s*(\w+)', line)
                    features['zip3'] = zip_match.group(1) if zip_match else 'unknown'
                    features['insurance'] = ins_match.group(1) if ins_match else 'unknown'
            
            # --- Extract line items (CPT codes) ---
            line_items = []
            for line in lines:
                match = re.match(
                    r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', 
                    line
                )
                if match:
                    code, desc, qty, unit_price, total = match.groups()
                    line_items.append({
                        'code': code,
                        'description': desc.strip(),
                        'qty': int(qty),
                        'unit_price': float(unit_price.replace(',', '')),
                        'line_total': float(total.replace(',', ''))
                    })
            
            # --- Engineer features from line items ---
            if line_items:
                codes = [item['code'] for item in line_items]
                amounts = [item['line_total'] for item in line_items]
                
                # Basic counts
                features['n_line_items'] = len(line_items)
                features['n_unique_codes'] = len(set(codes))
                
                # Amount statistics
                features['max_single_charge'] = max(amounts)
                features['min_single_charge'] = min(amounts)
                features['mean_charge'] = np.mean(amounts)
                features['std_charge'] = np.std(amounts) if len(amounts) > 1 else 0
                features['median_charge'] = np.median(amounts)
                
                # Cost concentration (Herfindahl-Hirschman Index)
                total = sum(amounts)
                if total > 0:
                    shares = [a / total for a in amounts]
                    features['cost_hhi'] = sum(s**2 for s in shares)
                    features['max_charge_pct'] = max(amounts) / total
                else:
                    features['cost_hhi'] = 1.0
                    features['max_charge_pct'] = 1.0
                
                # --- CPT Code Category Classification ---
                # E&M (Evaluation & Management) codes: 99xxx
                # Procedure codes: 10000-69999
                # Lab codes: 80000-89999
                # Critical care: 99291-99292
                # Observation: G0378-G0379
                
                cat_counts = {
                    'n_em_codes': 0,        # E&M visits
                    'n_procedure_codes': 0,  # Surgical/procedural
                    'n_lab_codes': 0,        # Laboratory
                    'n_critical_care': 0,    # Critical care
                    'n_observation': 0,      # Observation
                    'n_other_codes': 0       # Everything else
                }
                cat_costs = {
                    'cost_em': 0,
                    'cost_procedure': 0,
                    'cost_lab': 0,
                    'cost_critical_care': 0,
                    'cost_observation': 0,
                    'cost_other': 0
                }
                
                # Severity scoring based on E&M level
                em_levels = []
                
                for item in line_items:
                    code = item['code']
                    cost = item['line_total']
                    
                    if code.startswith('G037'):
                        cat_counts['n_observation'] += 1
                        cat_costs['cost_observation'] += cost
                    elif code in ('99291', '99292'):
                        cat_counts['n_critical_care'] += 1
                        cat_costs['cost_critical_care'] += cost
                    elif code.startswith('99'):
                        cat_counts['n_em_codes'] += 1
                        cat_costs['cost_em'] += cost
                        # Extract E&M level (last digit of 9928x codes)
                        if code.startswith('9928') and len(code) == 5:
                            em_levels.append(int(code[-1]))
                    elif code[0].isdigit():
                        code_num = int(code)
                        if 80000 <= code_num <= 89999:
                            cat_counts['n_lab_codes'] += 1
                            cat_costs['cost_lab'] += cost
                        elif 10000 <= code_num <= 69999:
                            cat_counts['n_procedure_codes'] += 1
                            cat_costs['cost_procedure'] += cost
                        else:
                            cat_counts['n_other_codes'] += 1
                            cat_costs['cost_other'] += cost
                    else:
                        cat_counts['n_other_codes'] += 1
                        cat_costs['cost_other'] += cost
                
                features.update(cat_counts)
                features.update(cat_costs)
                
                # E&M severity score
                features['max_em_level'] = max(em_levels) if em_levels else 0
                features['mean_em_level'] = np.mean(em_levels) if em_levels else 0
                
                # Has high-acuity indicators
                features['has_critical_care'] = int(cat_counts['n_critical_care'] > 0)
                features['has_intubation'] = int(any('31500' == item['code'] for item in line_items))
                features['has_observation'] = int(cat_counts['n_observation'] > 0)
                features['has_central_line'] = int(any('36556' == item['code'] for item in line_items))
                
                # Procedure diversity ratio
                features['code_diversity'] = features['n_unique_codes'] / features['n_line_items']
                
            else:
                # No line items extracted
                for key in ['n_line_items', 'n_unique_codes', 'max_single_charge',
                           'min_single_charge', 'mean_charge', 'std_charge', 
                           'median_charge', 'cost_hhi', 'max_charge_pct',
                           'n_em_codes', 'n_procedure_codes', 'n_lab_codes',
                           'n_critical_care', 'n_observation', 'n_other_codes',
                           'cost_em', 'cost_procedure', 'cost_lab',
                           'cost_critical_care', 'cost_observation', 'cost_other',
                           'max_em_level', 'mean_em_level',
                           'has_critical_care', 'has_intubation', 'has_observation',
                           'has_central_line', 'code_diversity']:
                    features[key] = 0
    
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
        return None
    
    return features


def extract_all_receipts(patient_ids, receipts_dir):
    """
    Extract features from all receipt PDFs.
    Returns a DataFrame indexed by patient_id.
    """
    if receipts_dir is None or not os.path.exists(receipts_dir):
        print(f"WARNING: Receipts directory not found: {receipts_dir}")
        print("Proceeding without PDF features.")
        return None
    
    all_features = {}
    missing = 0
    
    for i, pid in enumerate(patient_ids):
        pdf_path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(pdf_path):
            features = extract_receipt_features(pdf_path)
            if features:
                all_features[pid] = features
        else:
            missing += 1
        
        if (i + 1) % 500 == 0:
            print(f"  Processed {i+1}/{len(patient_ids)} PDFs...")
    
    if missing > 0:
        print(f"  WARNING: {missing}/{len(patient_ids)} PDFs not found")
    
    if all_features:
        df = pd.DataFrame.from_dict(all_features, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    
    return None


# ============================================================================
# MODULE 2: FEATURE ENGINEERING
# ============================================================================
def engineer_tabular_features(df, is_train=True):
    """
    Engineer features from the tabular CSV data.
    These are the core features that work WITHOUT PDFs.
    """
    feat = df.copy()
    
    # --- Encode primary_chronic ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    
    # --- Transform cost features (linearize relationship) ---
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['log_prior_cost'] = np.log1p(feat['prior_ed_cost_5y_usd'])
    feat['cbrt_prior_cost'] = np.cbrt(feat['prior_ed_cost_5y_usd'])
    
    # --- Ratio features ---
    feat['cost_per_visit'] = (
        feat['prior_ed_cost_5y_usd'] / feat['prior_ed_visits_5y'].clip(lower=1)
    )
    feat['sqrt_cost_per_visit'] = np.sqrt(feat['cost_per_visit'])
    feat['log_cost_per_visit'] = np.log1p(feat['cost_per_visit'])
    
    # --- Visit features ---
    feat['visits_squared'] = feat['prior_ed_visits_5y'] ** 2
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['visits_per_year'] = feat['prior_ed_visits_5y'] / 5.0
    
    # --- Interaction features ---
    feat['chronic_x_cost'] = feat['chronic_encoded'] * feat['prior_ed_cost_5y_usd']
    feat['chronic_x_visits'] = feat['chronic_encoded'] * feat['prior_ed_visits_5y']
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']
    feat['cost_x_visits'] = feat['prior_ed_cost_5y_usd'] * feat['prior_ed_visits_5y']
    
    # --- Polynomial features ---
    feat['cost_squared'] = feat['prior_ed_cost_5y_usd'] ** 2
    
    # --- Annualized cost projection (simple heuristic) ---
    # Prior 5y cost → extrapolate to 3y  
    feat['annual_cost'] = feat['prior_ed_cost_5y_usd'] / 5.0
    feat['projected_3y_simple'] = feat['annual_cost'] * 3.0
    
    # --- Categorical one-hot (for non-tree models) ---
    feat['is_HF'] = (feat['primary_chronic'] == 'HF').astype(int)
    feat['is_DiabetesComp'] = (feat['primary_chronic'] == 'DiabetesComp').astype(int)
    feat['is_Pneumonia'] = (feat['primary_chronic'] == 'Pneumonia').astype(int)
    
    return feat


def merge_pdf_features(feat_df, pdf_df):
    """Merge PDF-extracted features into the main feature DataFrame."""
    if pdf_df is None:
        return feat_df
    
    # Encode categorical PDF features
    if 'insurance' in pdf_df.columns:
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        pdf_df['insurance_encoded'] = pdf_df['insurance'].map(ins_map).fillna(-1)
        
    if 'zip3' in pdf_df.columns:
        # Use first digit of ZIP3 as region indicator
        pdf_df['zip_region'] = pdf_df['zip3'].astype(str).str[0].astype(int)
    
    # Drop raw categorical columns before merge
    cols_to_drop = ['insurance', 'zip3']
    pdf_numeric = pdf_df.drop(columns=[c for c in cols_to_drop if c in pdf_df.columns])
    
    merged = feat_df.merge(pdf_numeric, on='patient_id', how='left')
    
    return merged


# ============================================================================
# MODULE 3: MODEL TRAINING & ENSEMBLE
# ============================================================================
def get_feature_columns(df):
    """Get list of feature columns (everything except IDs, target, and raw strings)."""
    exclude = [
        'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
        # Raw string columns from patients.csv — we use encoded versions instead
        'sex', 'insurance', 'zip3',
        # Temp columns
        'cost_bin', 'strat_col',
    ]
    # Also exclude any remaining object/string columns LightGBM can't handle
    numeric_cols = [
        c for c in df.columns 
        if c not in exclude and df[c].dtype in ['int64', 'float64', 'int32', 'float32', 'bool']
    ]
    return numeric_cols


def train_lightgbm(X_train, y_train, X_val, y_val, feature_cols, fold_num=0):
    """Train LightGBM with MAE objective."""
    params = {
        'objective': 'mae',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': 6,
        'min_child_samples': 20,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'lambda_l1': 0.1,
        'lambda_l2': 0.1,
        'verbose': -1,
        'seed': Config.SEED + fold_num,
        'n_jobs': -1,
    }
    
    dtrain = lgb.Dataset(X_train[feature_cols], label=y_train)
    dval = lgb.Dataset(X_val[feature_cols], label=y_val, reference=dtrain)
    
    model = lgb.train(
        params, dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    return model


def train_xgboost(X_train, y_train, X_val, y_val, feature_cols, fold_num=0):
    """Train XGBoost with MAE objective."""
    params = {
        'objective': 'reg:absoluteerror',
        'eval_metric': 'mae',
        'learning_rate': 0.05,
        'max_depth': 6,
        'min_child_weight': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'seed': Config.SEED + fold_num,
        'nthread': -1,
        'verbosity': 0,
    }
    
    dtrain = xgb.DMatrix(X_train[feature_cols], label=y_train)
    dval = xgb.DMatrix(X_val[feature_cols], label=y_val)
    
    model = xgb.train(
        params, dtrain,
        num_boost_round=2000,
        evals=[(dval, 'val')],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    
    return model


def train_catboost(X_train, y_train, X_val, y_val, feature_cols, fold_num=0):
    """Train CatBoost with MAE objective."""
    model = CatBoostRegressor(
        loss_function='MAE',
        eval_metric='MAE',
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3,
        random_seed=Config.SEED + fold_num,
        verbose=0,
        early_stopping_rounds=50,
    )
    
    model.fit(
        X_train[feature_cols], y_train,
        eval_set=(X_val[feature_cols], y_val),
    )
    
    return model


def run_kfold_ensemble(train_df, test_df, feature_cols):
    """
    Run K-Fold cross-validation with LightGBM + XGBoost + CatBoost ensemble.
    Returns OOF predictions, test predictions, and CV score.
    """
    y = train_df['ed_cost_next3y_usd'].values
    
    # Stratified folds by chronic condition + cost quintile
    train_df['cost_bin'] = pd.qcut(y, q=5, labels=False)
    train_df['strat_col'] = (
        train_df['primary_chronic'].astype(str) + '_' + 
        train_df['cost_bin'].astype(str)
    )
    le = LabelEncoder()
    strat_labels = le.fit_transform(train_df['strat_col'])
    
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
    
    # Storage
    oof_lgb = np.zeros(len(train_df))
    oof_xgb = np.zeros(len(train_df))
    oof_cat = np.zeros(len(train_df))
    
    test_lgb = np.zeros(len(test_df))
    test_xgb = np.zeros(len(test_df))
    test_cat = np.zeros(len(test_df))
    
    fold_scores = []
    lgb_importances = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_df, strat_labels)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold + 1}/{Config.N_FOLDS}")
        print(f"{'='*60}")
        
        X_tr = train_df.iloc[train_idx]
        X_vl = train_df.iloc[val_idx]
        y_tr = y[train_idx]
        y_vl = y[val_idx]
        
        # --- LightGBM ---
        lgb_model = train_lightgbm(X_tr, y_tr, X_vl, y_vl, feature_cols, fold)
        oof_lgb[val_idx] = lgb_model.predict(X_vl[feature_cols])
        test_lgb += lgb_model.predict(test_df[feature_cols]) / Config.N_FOLDS
        lgb_importances.append(
            pd.Series(lgb_model.feature_importance(importance_type='gain'), 
                      index=feature_cols)
        )
        
        # --- XGBoost ---
        xgb_model = train_xgboost(X_tr, y_tr, X_vl, y_vl, feature_cols, fold)
        oof_xgb[val_idx] = xgb_model.predict(
            xgb.DMatrix(X_vl[feature_cols])
        )
        test_xgb += xgb_model.predict(
            xgb.DMatrix(test_df[feature_cols])
        ) / Config.N_FOLDS
        
        # --- CatBoost ---
        cat_model = train_catboost(X_tr, y_tr, X_vl, y_vl, feature_cols, fold)
        oof_cat[val_idx] = cat_model.predict(X_vl[feature_cols])
        test_cat += cat_model.predict(test_df[feature_cols]) / Config.N_FOLDS
        
        # --- Fold scores ---
        mae_lgb = mean_absolute_error(y_vl, oof_lgb[val_idx])
        mae_xgb = mean_absolute_error(y_vl, oof_xgb[val_idx])
        mae_cat = mean_absolute_error(y_vl, oof_cat[val_idx])
        
        # Simple average ensemble for this fold
        oof_ens = (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3
        mae_ens = mean_absolute_error(y_vl, oof_ens)
        
        fold_scores.append({
            'fold': fold + 1,
            'lgb': mae_lgb, 'xgb': mae_xgb, 'cat': mae_cat, 'ensemble': mae_ens
        })
        print(f"  LGB MAE:      {mae_lgb:.2f}")
        print(f"  XGB MAE:      {mae_xgb:.2f}")
        print(f"  CatBoost MAE: {mae_cat:.2f}")
        print(f"  Ensemble MAE: {mae_ens:.2f}")
    
    # --- Optimize ensemble weights on OOF ---
    print(f"\n{'='*60}")
    print("OPTIMIZING ENSEMBLE WEIGHTS")
    print(f"{'='*60}")
    
    best_mae = float('inf')
    best_weights = (1/3, 1/3, 1/3)
    
    for w1 in np.arange(0.1, 0.8, 0.05):
        for w2 in np.arange(0.1, 0.8 - w1, 0.05):
            w3 = 1.0 - w1 - w2
            if w3 < 0.05:
                continue
            oof_blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
            mae = mean_absolute_error(y, oof_blend)
            if mae < best_mae:
                best_mae = mae
                best_weights = (w1, w2, w3)
    
    print(f"Best weights: LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}")
    print(f"Best OOF MAE: {best_mae:.2f}")
    
    # Individual model OOF MAEs
    print(f"\nOverall OOF MAE — LGB:      {mean_absolute_error(y, oof_lgb):.2f}")
    print(f"Overall OOF MAE — XGB:      {mean_absolute_error(y, oof_xgb):.2f}")
    print(f"Overall OOF MAE — CatBoost: {mean_absolute_error(y, oof_cat):.2f}")
    print(f"Overall OOF MAE — Equal Avg: {mean_absolute_error(y, (oof_lgb+oof_xgb+oof_cat)/3):.2f}")
    
    # Final test predictions
    test_pred = (
        best_weights[0] * test_lgb + 
        best_weights[1] * test_xgb + 
        best_weights[2] * test_cat
    )
    
    # Clip predictions to reasonable range
    test_pred = np.clip(test_pred, 0, None)
    
    # Feature importance
    importance = pd.concat(lgb_importances, axis=1).mean(axis=1).sort_values(ascending=False)
    print(f"\nTop 15 Features (LightGBM gain):")
    for feat, imp in importance.head(15).items():
        print(f"  {feat:30s} {imp:.1f}")
    
    # Clean up temp columns
    train_df.drop(columns=['cost_bin', 'strat_col'], inplace=True, errors='ignore')
    
    return test_pred, best_mae, fold_scores


# ============================================================================
# MODULE 4: MAIN PIPELINE
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2: ED Cost Forecasting")
    print("Strategy B — Multimodal Feature Engineer + GBM Ensemble")
    print("=" * 70)
    
    # --- Detect file paths ---
    # Try multiple possible locations
    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.TRAIN_PATH = os.path.join(base, 'ed_cost_train.csv')
            Config.TEST_PATH = os.path.join(base, 'ed_cost_test.csv')
            Config.PATIENTS_PATH = os.path.join(base, 'patients.csv')
            # Check for receipts
            receipts_candidates = [
                os.path.join(base, 'receipts_pdf'),
                os.path.join(base, '..', 'receipts_pdf'),
            ]
            for rc in receipts_candidates:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
                    break
            break
    
    print(f"\nTrain: {Config.TRAIN_PATH}")
    print(f"Test:  {Config.TEST_PATH}")
    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")
    
    # --- Load data ---
    print("\n[1/5] Loading data...")
    train = pd.read_csv(Config.TRAIN_PATH)
    test = pd.read_csv(Config.TEST_PATH)
    
    print(f"  Train: {train.shape[0]} rows, {train.shape[1]} cols")
    print(f"  Test:  {test.shape[0]} rows, {test.shape[1]} cols")
    print(f"  Target stats: mean={train['ed_cost_next3y_usd'].mean():.1f}, "
          f"median={train['ed_cost_next3y_usd'].median():.1f}, "
          f"std={train['ed_cost_next3y_usd'].std():.1f}")
    
    # --- Load patients.csv if available ---
    patients_df = None
    if os.path.exists(Config.PATIENTS_PATH):
        patients_df = pd.read_csv(Config.PATIENTS_PATH)
        print(f"  Patients: {patients_df.shape[0]} rows, cols: {patients_df.columns.tolist()}")
        # Merge patient demographics
        train = train.merge(patients_df, on='patient_id', how='left')
        test = test.merge(patients_df, on='patient_id', how='left')
        
        # Encode patient features (create numeric versions)
        if 'sex' in train.columns:
            train['sex_encoded'] = (train['sex'] == 'M').astype(int)
            test['sex_encoded'] = (test['sex'] == 'M').astype(int)
        if 'age' in train.columns:
            train['age'] = train['age'].astype(float)
            test['age'] = test['age'].astype(float)
        if 'insurance' in train.columns:
            ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
            train['insurance_encoded'] = train['insurance'].map(ins_map).fillna(-1).astype(float)
            test['insurance_encoded'] = test['insurance'].map(ins_map).fillna(-1).astype(float)
        if 'zip3' in train.columns:
            train['zip_region'] = train['zip3'].astype(str).str[0].astype(float)
            test['zip_region'] = test['zip3'].astype(str).str[0].astype(float)
    
    # --- Extract PDF features ---
    print("\n[2/5] Extracting PDF features...")
    all_pids = sorted(set(train['patient_id'].tolist() + test['patient_id'].tolist()))
    pdf_features = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)
    
    if pdf_features is not None:
        print(f"  Extracted PDF features for {len(pdf_features)} patients")
        print(f"  PDF feature columns: {pdf_features.columns.tolist()}")
    
    # --- Feature engineering ---
    print("\n[3/5] Engineering features...")
    train_feat = engineer_tabular_features(train, is_train=True)
    test_feat = engineer_tabular_features(test, is_train=False)
    
    # Merge PDF features
    train_feat = merge_pdf_features(train_feat, pdf_features)
    test_feat = merge_pdf_features(test_feat, pdf_features)
    
    # Get feature columns
    feature_cols = get_feature_columns(train_feat)
    print(f"  Total features: {len(feature_cols)}")
    print(f"  Features: {feature_cols}")
    
    # Fill NaN (from missing PDFs or patients.csv)
    for col in feature_cols:
        if train_feat[col].isna().any():
            fill_val = train_feat[col].median()
            train_feat[col] = train_feat[col].fillna(fill_val)
        if test_feat[col].isna().any():
            fill_val = train_feat[col].median() if not train_feat[col].isna().all() else 0
            test_feat[col] = test_feat[col].fillna(fill_val)
    
    # Final safety: ensure all feature columns are numeric
    for col in feature_cols:
        train_feat[col] = pd.to_numeric(train_feat[col], errors='coerce').fillna(0)
        test_feat[col] = pd.to_numeric(test_feat[col], errors='coerce').fillna(0)
    
    # --- Train ensemble ---
    print("\n[4/5] Training ensemble...")
    test_pred, cv_mae, fold_scores = run_kfold_ensemble(
        train_feat, test_feat, feature_cols
    )
    
    # --- Generate submission ---
    print(f"\n[5/5] Generating submission...")
    submission = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    
    output_path = Config.OUTPUT_PATH
    submission.to_csv(output_path, index=False)
    print(f"  Saved: {output_path}")
    print(f"  Predictions: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    
    # --- Summary ---
    print(f"\n{'='*70}")
    print(f"FINAL RESULTS")
    print(f"{'='*70}")
    print(f"  CV MAE (optimized ensemble): {cv_mae:.2f}")
    print(f"  Submission rows: {len(submission)}")
    print(f"\nFold-by-fold breakdown:")
    for fs in fold_scores:
        print(f"  Fold {fs['fold']}: LGB={fs['lgb']:.2f}  XGB={fs['xgb']:.2f}  "
              f"CAT={fs['cat']:.2f}  Ensemble={fs['ensemble']:.2f}")
    
    return submission, cv_mae


# ============================================================================
# RUN
# ============================================================================
if __name__ == '__main__':
    submission, cv_mae = main()

# Iteration 2
- 475.2562

In [ ]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v2 (Anti-Overfitting)
================================================================================
Key changes from v1:
  1. PDF extraction uses PyMuPDF (fitz) with pdfplumber fallback
  2. Reduced feature set — removed redundant correlated transforms
  3. Stronger regularization across all models
  4. Fixed ensemble weights (equal 1/3) — no OOF weight optimization
  5. Added repeated K-fold for stability
  6. Cleaner feature selection: only features that generalize
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings
from collections import Counter

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, StratifiedKFold, RepeatedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


# ============================================================================
# CONFIG
# ============================================================================
class Config:
    SEED = 42
    N_FOLDS = 5
    RECEIPTS_DIR = None
    TRAIN_PATH = "ed_cost_train.csv"
    TEST_PATH = "ed_cost_test.csv"
    PATIENTS_PATH = "patients.csv"
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# MODULE 1: ROBUST PDF EXTRACTION (PyMuPDF primary, pdfplumber fallback)
# ============================================================================

# Check available PDF libraries ONCE at import time
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz  # PyMuPDF
    _HAS_FITZ = True
except ImportError:
    pass
try:
    import pdfplumber
    _HAS_PDFPLUMBER = True
except ImportError:
    pass


def _extract_with_fitz(pdf_path):
    """Extract text from PDF using PyMuPDF (fitz)."""
    doc = fitz.open(pdf_path)
    text = doc[0].get_text()
    doc.close()
    return text


def _extract_with_pdfplumber(pdf_path):
    """Extract text from PDF using pdfplumber."""
    with pdfplumber.open(pdf_path) as pdf:
        return pdf.pages[0].extract_text()


def _parse_receipt_text(text):
    """
    Parse receipt text into structured data.
    Handles both pdfplumber format (tab-separated rows) and
    PyMuPDF format (each field on its own line).
    """
    if not text:
        return None, None, None

    lines = [l.strip() for l in text.split('\n') if l.strip()]

    # Extract header metadata
    zip3, insurance = None, None
    for line in lines:
        if 'ZIP3' in line:
            zm = re.search(r'ZIP3:\s*(\d+)', line)
            im = re.search(r'Insurance:\s*(\w+)', line)
            if zm: zip3 = zm.group(1)
            if im: insurance = im.group(1)

    # --- Strategy 1: Try single-line format (pdfplumber style) ---
    line_items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit_price, total = m.groups()
            line_items.append({
                'code': code,
                'description': desc.strip(),
                'qty': int(qty),
                'line_total': float(total.replace(',', ''))
            })

    # --- Strategy 2: If Strategy 1 found nothing, try multi-line (PyMuPDF style) ---
    if not line_items:
        # Find where data starts (after "Line Total ($)" header)
        data_start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line or 'line total' in line.lower():
                data_start = i + 1
                break

        if data_start is not None:
            remaining = lines[data_start:]
            # Filter out TOTAL line
            remaining = [l for l in remaining if not l.startswith('TOTAL')]

            # Group into chunks of 5: code, description, qty, unit_price, line_total
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                unit_str = remaining[i + 3].strip().replace(',', '')
                total_str = remaining[i + 4].strip().replace(',', '')

                # Validate: code should be alphanumeric, qty should be a digit
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        line_items.append({
                            'code': code,
                            'description': desc,
                            'qty': int(qty_str),
                            'line_total': float(total_str)
                        })
                    except ValueError:
                        pass
                i += 5

    return line_items, zip3, insurance


def extract_receipt_features(pdf_path):
    """Extract features from a single receipt PDF."""
    # Get text using best available library
    text = None
    if _HAS_FITZ:
        try:
            text = _extract_with_fitz(pdf_path)
        except Exception:
            pass
    if text is None and _HAS_PDFPLUMBER:
        try:
            text = _extract_with_pdfplumber(pdf_path)
        except Exception:
            pass
    if text is None:
        return None

    line_items, zip3, insurance = _parse_receipt_text(text)

    features = {}
    if zip3:
        features['pdf_zip3'] = zip3
    if insurance:
        features['pdf_insurance'] = insurance

    if line_items and len(line_items) > 0:
        codes = [item['code'] for item in line_items]
        amounts = [item['line_total'] for item in line_items]
        total = sum(amounts) if amounts else 1.0

        # --- Core count features ---
        features['n_line_items'] = len(line_items)
        features['n_unique_codes'] = len(set(codes))

        # --- Amount distribution ---
        features['max_single_charge'] = max(amounts)
        features['mean_charge'] = np.mean(amounts)
        if total > 0:
            features['max_charge_pct'] = max(amounts) / total
            shares = [a / total for a in amounts]
            features['cost_hhi'] = sum(s ** 2 for s in shares)
        else:
            features['max_charge_pct'] = 1.0
            features['cost_hhi'] = 1.0

        # --- CPT code category counts ---
        n_em, n_proc, n_lab, n_crit, n_obs = 0, 0, 0, 0, 0
        cost_em, cost_proc, cost_crit = 0.0, 0.0, 0.0
        em_levels = []

        for item in line_items:
            code = item['code']
            cost = item['line_total']

            if code.startswith('G037'):
                n_obs += 1
            elif code in ('99291', '99292'):
                n_crit += 1
                cost_crit += cost
            elif code.startswith('99'):
                n_em += 1
                cost_em += cost
                if code.startswith('9928') and len(code) == 5:
                    em_levels.append(int(code[-1]))
            elif code[0].isdigit():
                code_num = int(code)
                if 80000 <= code_num <= 89999:
                    n_lab += 1
                elif 10000 <= code_num <= 69999:
                    n_proc += 1
                    cost_proc += cost

        features['n_em_codes'] = n_em
        features['n_procedure_codes'] = n_proc
        features['n_lab_codes'] = n_lab
        features['n_critical_care'] = n_crit
        features['has_critical_care'] = int(n_crit > 0)
        features['has_observation'] = int(n_obs > 0)

        # Cost allocation ratios (more stable than raw costs)
        if total > 0:
            features['pct_cost_em'] = cost_em / total
            features['pct_cost_procedure'] = cost_proc / total
            features['pct_cost_critical'] = cost_crit / total
        else:
            features['pct_cost_em'] = 0
            features['pct_cost_procedure'] = 0
            features['pct_cost_critical'] = 0

        features['max_em_level'] = max(em_levels) if em_levels else 0

    return features if features else None


def extract_all_receipts(patient_ids, receipts_dir):
    """Extract features from all receipt PDFs."""
    if receipts_dir is None or not os.path.exists(receipts_dir):
        print(f"  WARNING: Receipts directory not found: {receipts_dir}")
        return None

    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  ERROR: Neither PyMuPDF nor pdfplumber installed!")
        print("  Run: pip install pymupdf   (recommended)")
        print("  Or:  pip install pdfplumber")
        return None

    backend = "PyMuPDF" if _HAS_FITZ else "pdfplumber"
    print(f"  Using PDF backend: {backend}")

    all_features = {}
    found, missing, failed = 0, 0, 0

    for i, pid in enumerate(patient_ids):
        pdf_path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(pdf_path):
            feats = extract_receipt_features(pdf_path)
            if feats:
                all_features[pid] = feats
                found += 1
            else:
                failed += 1
        else:
            missing += 1

        if (i + 1) % 1000 == 0:
            print(f"  Processed {i + 1}/{len(patient_ids)} PDFs "
                  f"(found={found}, missing={missing}, failed={failed})")

    print(f"  Final: {found} extracted, {missing} not found, {failed} failed")

    if all_features:
        df = pd.DataFrame.from_dict(all_features, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# MODULE 2: FEATURE ENGINEERING (Lean — fewer, more robust features)
# ============================================================================
def engineer_features(df, pdf_df=None, patients_df=None):
    """
    Build features. Philosophy: fewer, less correlated features = less overfit.
    """
    feat = df.copy()

    # --- Encode chronic condition ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)

    # --- Cost transforms (pick ONE good transform, not all of them) ---
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['log_prior_cost'] = np.log1p(feat['prior_ed_cost_5y_usd'])

    # --- Ratio features ---
    feat['cost_per_visit'] = (
        feat['prior_ed_cost_5y_usd'] / feat['prior_ed_visits_5y'].clip(lower=1)
    )
    feat['log_cost_per_visit'] = np.log1p(feat['cost_per_visit'])

    # --- Visit features ---
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # --- Key interaction (top feature from v1) ---
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']
    feat['chronic_x_visits'] = feat['chronic_encoded'] * feat['prior_ed_visits_5y']

    # --- Simple projection ---
    feat['annual_cost'] = feat['prior_ed_cost_5y_usd'] / 5.0

    # --- One-hot chronic ---
    feat['is_HF'] = (feat['primary_chronic'] == 'HF').astype(int)
    feat['is_DiabetesComp'] = (feat['primary_chronic'] == 'DiabetesComp').astype(int)

    # --- Merge patients.csv demographics ---
    if patients_df is not None:
        patients_df = patients_df.copy()
        if 'age' in patients_df.columns:
            patients_df['age'] = patients_df['age'].astype(float)
        if 'sex' in patients_df.columns:
            patients_df['sex_encoded'] = (patients_df['sex'] == 'M').astype(int)
        if 'insurance' in patients_df.columns:
            ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
            patients_df['insurance_encoded'] = patients_df['insurance'].map(ins_map).fillna(-1).astype(float)
        if 'zip3' in patients_df.columns:
            patients_df['zip_region'] = patients_df['zip3'].astype(str).str[0].astype(float)

        # Drop raw string columns before merge
        drop_cols = ['sex', 'insurance', 'zip3']
        patients_clean = patients_df.drop(columns=[c for c in drop_cols if c in patients_df.columns])
        feat = feat.merge(patients_clean, on='patient_id', how='left')

    # --- Merge PDF features ---
    if pdf_df is not None:
        pdf_numeric = pdf_df.copy()
        # Encode PDF categorical features
        if 'pdf_insurance' in pdf_numeric.columns:
            ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
            pdf_numeric['pdf_ins_encoded'] = pdf_numeric['pdf_insurance'].map(ins_map).fillna(-1)
        if 'pdf_zip3' in pdf_numeric.columns:
            pdf_numeric['pdf_zip_region'] = pdf_numeric['pdf_zip3'].astype(str).str[0].astype(float)

        drop_cols = ['pdf_insurance', 'pdf_zip3']
        pdf_numeric = pdf_numeric.drop(columns=[c for c in drop_cols if c in pdf_numeric.columns])
        feat = feat.merge(pdf_numeric, on='patient_id', how='left')

    return feat


def get_feature_columns(df):
    """Get only numeric feature columns."""
    exclude = {
        'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
        'sex', 'insurance', 'zip3', 'pdf_insurance', 'pdf_zip3',
        'cost_bin', 'strat_col',
    }
    return [
        c for c in df.columns
        if c not in exclude and df[c].dtype in ('int64', 'float64', 'int32', 'float32', 'bool')
    ]


# ============================================================================
# MODULE 3: MODELS (Stronger regularization)
# ============================================================================
def train_lightgbm(X_tr, y_tr, X_vl, y_vl, feat_cols, fold=0):
    params = {
        'objective': 'mae',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,        # slower learning
        'num_leaves': 20,             # simpler trees (was 31)
        'max_depth': 5,               # shallower (was 6)
        'min_child_samples': 30,      # more samples per leaf (was 20)
        'feature_fraction': 0.7,      # more dropout
        'bagging_fraction': 0.7,      # more dropout
        'bagging_freq': 5,
        'lambda_l1': 1.0,             # 10x stronger L1 (was 0.1)
        'lambda_l2': 1.0,             # 10x stronger L2 (was 0.1)
        'verbose': -1,
        'seed': Config.SEED + fold,
        'n_jobs': -1,
    }
    dtrain = lgb.Dataset(X_tr[feat_cols], label=y_tr)
    dval = lgb.Dataset(X_vl[feat_cols], label=y_vl, reference=dtrain)
    model = lgb.train(
        params, dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]  # patience 100
    )
    return model


def train_xgboost(X_tr, y_tr, X_vl, y_vl, feat_cols, fold=0):
    params = {
        'objective': 'reg:absoluteerror',
        'eval_metric': 'mae',
        'learning_rate': 0.03,
        'max_depth': 5,
        'min_child_weight': 30,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'reg_alpha': 1.0,
        'reg_lambda': 1.0,
        'seed': Config.SEED + fold,
        'nthread': -1,
        'verbosity': 0,
    }
    dtrain = xgb.DMatrix(X_tr[feat_cols], label=y_tr)
    dval = xgb.DMatrix(X_vl[feat_cols], label=y_vl)
    model = xgb.train(
        params, dtrain,
        num_boost_round=3000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False
    )
    return model


def train_catboost(X_tr, y_tr, X_vl, y_vl, feat_cols, fold=0):
    model = CatBoostRegressor(
        loss_function='MAE',
        eval_metric='MAE',
        iterations=3000,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=5,                # stronger (was 3)
        min_data_in_leaf=30,
        random_seed=Config.SEED + fold,
        verbose=0,
        early_stopping_rounds=100,
    )
    model.fit(X_tr[feat_cols], y_tr, eval_set=(X_vl[feat_cols], y_vl))
    return model


# ============================================================================
# MODULE 4: TRAINING PIPELINE
# ============================================================================
def run_kfold_ensemble(train_df, test_df, feature_cols):
    y = train_df['ed_cost_next3y_usd'].values

    # Stratified folds
    train_df = train_df.copy()
    train_df['cost_bin'] = pd.qcut(y, q=5, labels=False)
    train_df['strat_col'] = (
        train_df['primary_chronic'].astype(str) + '_' +
        train_df['cost_bin'].astype(str)
    )
    le = LabelEncoder()
    strat_labels = le.fit_transform(train_df['strat_col'])

    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)

    oof_lgb = np.zeros(len(train_df))
    oof_xgb = np.zeros(len(train_df))
    oof_cat = np.zeros(len(train_df))
    test_lgb = np.zeros(len(test_df))
    test_xgb = np.zeros(len(test_df))
    test_cat = np.zeros(len(test_df))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(train_df, strat_labels)):
        print(f"\n--- FOLD {fold + 1}/{Config.N_FOLDS} ---")

        X_tr, X_vl = train_df.iloc[train_idx], train_df.iloc[val_idx]
        y_tr, y_vl = y[train_idx], y[val_idx]

        # LightGBM
        lgb_model = train_lightgbm(X_tr, y_tr, X_vl, y_vl, feature_cols, fold)
        oof_lgb[val_idx] = lgb_model.predict(X_vl[feature_cols])
        test_lgb += lgb_model.predict(test_df[feature_cols]) / Config.N_FOLDS

        # XGBoost
        xgb_model = train_xgboost(X_tr, y_tr, X_vl, y_vl, feature_cols, fold)
        oof_xgb[val_idx] = xgb_model.predict(xgb.DMatrix(X_vl[feature_cols]))
        test_xgb += xgb_model.predict(xgb.DMatrix(test_df[feature_cols])) / Config.N_FOLDS

        # CatBoost
        cat_model = train_catboost(X_tr, y_tr, X_vl, y_vl, feature_cols, fold)
        oof_cat[val_idx] = cat_model.predict(X_vl[feature_cols])
        test_cat += cat_model.predict(test_df[feature_cols]) / Config.N_FOLDS

        mae_lgb = mean_absolute_error(y_vl, oof_lgb[val_idx])
        mae_xgb = mean_absolute_error(y_vl, oof_xgb[val_idx])
        mae_cat = mean_absolute_error(y_vl, oof_cat[val_idx])
        mae_ens = mean_absolute_error(y_vl, (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3)

        fold_scores.append({'fold': fold + 1, 'lgb': mae_lgb, 'xgb': mae_xgb,
                            'cat': mae_cat, 'ensemble': mae_ens})
        print(f"  LGB={mae_lgb:.2f}  XGB={mae_xgb:.2f}  CAT={mae_cat:.2f}  ENS={mae_ens:.2f}")

    # --- FIXED equal-weight ensemble (no optimization = no overfitting) ---
    oof_ens = (oof_lgb + oof_xgb + oof_cat) / 3.0
    test_pred = (test_lgb + test_xgb + test_cat) / 3.0
    cv_mae = mean_absolute_error(y, oof_ens)

    print(f"\n{'=' * 60}")
    print(f"OOF MAE — LGB:      {mean_absolute_error(y, oof_lgb):.2f}")
    print(f"OOF MAE — XGB:      {mean_absolute_error(y, oof_xgb):.2f}")
    print(f"OOF MAE — CatBoost: {mean_absolute_error(y, oof_cat):.2f}")
    print(f"OOF MAE — Equal Ensemble: {cv_mae:.2f}")

    # Also check: would a slight CatBoost tilt help?
    # But DON'T use this for final prediction — just report it
    for w_cat in [0.4, 0.5, 0.6]:
        w_other = (1.0 - w_cat) / 2.0
        blend = w_other * oof_lgb + w_other * oof_xgb + w_cat * oof_cat
        print(f"  (Info only) w_cat={w_cat:.1f}: OOF MAE={mean_absolute_error(y, blend):.2f}")

    # Clip to training range with small buffer
    y_min, y_max = y.min(), y.max()
    test_pred = np.clip(test_pred, y_min * 0.5, y_max * 1.2)

    # Clean up
    train_df.drop(columns=['cost_bin', 'strat_col'], inplace=True, errors='ignore')

    return test_pred, cv_mae, fold_scores


# ============================================================================
# MODULE 5: MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2: ED Cost Forecasting — v2 (Anti-Overfitting)")
    print("=" * 70)

    # --- Detect paths ---
    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.TRAIN_PATH = os.path.join(base, 'ed_cost_train.csv')
            Config.TEST_PATH = os.path.join(base, 'ed_cost_test.csv')
            Config.PATIENTS_PATH = os.path.join(base, 'patients.csv')
            for rc in [os.path.join(base, 'receipts_pdf'),
                       os.path.join(base, '..', 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
                    break
            break

    print(f"\nPaths: train={Config.TRAIN_PATH}, receipts={Config.RECEIPTS_DIR or 'NOT FOUND'}")
    print(f"PDF backends: PyMuPDF={_HAS_FITZ}, pdfplumber={_HAS_PDFPLUMBER}")

    # --- Load data ---
    print("\n[1/5] Loading data...")
    train = pd.read_csv(Config.TRAIN_PATH)
    test = pd.read_csv(Config.TEST_PATH)
    print(f"  Train: {train.shape[0]} rows | Test: {test.shape[0]} rows")

    patients_df = None
    if os.path.exists(Config.PATIENTS_PATH):
        patients_df = pd.read_csv(Config.PATIENTS_PATH)
        print(f"  Patients: {patients_df.shape[0]} rows, cols={patients_df.columns.tolist()}")

    # --- Extract PDF features ---
    print("\n[2/5] Extracting PDF features...")
    all_pids = sorted(set(train['patient_id'].tolist() + test['patient_id'].tolist()))
    pdf_features = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    if pdf_features is not None:
        n_pdf_cols = len([c for c in pdf_features.columns if c != 'patient_id'])
        print(f"  PDF features: {n_pdf_cols} columns for {len(pdf_features)} patients")
    else:
        print("  No PDF features extracted — proceeding without them.")

    # --- Feature engineering ---
    print("\n[3/5] Engineering features...")
    train_feat = engineer_features(train, pdf_features, patients_df)
    test_feat = engineer_features(test, pdf_features, patients_df)

    feature_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feature_cols)}): {feature_cols}")

    # Fill NaN + ensure numeric
    for col in feature_cols:
        median_val = train_feat[col].median() if not train_feat[col].isna().all() else 0
        train_feat[col] = pd.to_numeric(train_feat[col], errors='coerce').fillna(median_val)
        test_feat[col] = pd.to_numeric(test_feat[col], errors='coerce').fillna(median_val)

    # --- Train ---
    print("\n[4/5] Training ensemble (equal weights, strong regularization)...")
    test_pred, cv_mae, fold_scores = run_kfold_ensemble(train_feat, test_feat, feature_cols)

    # --- Submit ---
    print(f"\n[5/5] Generating submission...")
    submission = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    submission.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'=' * 70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Predictions: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"Saved: {Config.OUTPUT_PATH}")
    print(f"{'=' * 70}")

    for fs in fold_scores:
        print(f"  Fold {fs['fold']}: LGB={fs['lgb']:.2f}  XGB={fs['xgb']:.2f}  "
              f"CAT={fs['cat']:.2f}  ENS={fs['ensemble']:.2f}")

    return submission, cv_mae


if __name__ == '__main__':
    submission, cv_mae = main()

# Iteration 3
- 472.0431

In [2]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v3 (Precision Feature Engineering)
================================================================================
Diagnosis from v2 (CV=462, Test=475, gap=13):
  ✓ PDF extraction works (2319/4000 patients)
  ✓ Anti-overfit regularization helped
  ✗ 35 features for 2000 samples — too many, many redundant
  ✗ pdf_ins_encoded / pdf_zip_region duplicate patients.csv data
  ✗ 42% of PDF features median-filled without indicator → noise
  ✗ Equal ensemble weights dilute CatBoost (best model by ~12 MAE)

v3 Changes:
  1. Curated 18 features (was 35) — drop all redundant/correlated
  2. `has_pdf` indicator — let trees know when PDF features are imputed
  3. PDF fill strategy: use -1 sentinel (not median) for tree models
  4. Mild CatBoost weight (0.2/0.2/0.6) — structural prior, not OOF-tuned
  5. Tighter regularization on PDF features
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings
from collections import Counter

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


# ============================================================================
# CONFIG
# ============================================================================
class Config:
    SEED = 42
    N_FOLDS = 5
    # Mild CatBoost tilt — structural prior, not tuned on OOF
    W_LGB = 0.20
    W_XGB = 0.20
    W_CAT = 0.60
    RECEIPTS_DIR = None
    TRAIN_PATH = "ed_cost_train.csv"
    TEST_PATH = "ed_cost_test.csv"
    PATIENTS_PATH = "patients.csv"
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (same as v2, proven to work)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz
    _HAS_FITZ = True
except ImportError:
    pass
try:
    import pdfplumber
    _HAS_PDFPLUMBER = True
except ImportError:
    pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path)
            text = doc[0].get_text()
            doc.close()
            return text
        except Exception:
            pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except Exception:
            pass
    return None


def _parse_receipt(text):
    """Parse receipt into line items. Handles both single-line and multi-line formats."""
    if not text:
        return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]

    # Strategy 1: single-line format (pdfplumber)
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, _, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'total': float(total.replace(',', ''))})

    # Strategy 2: multi-line format (PyMuPDF)
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1
                break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'total': float(total_str)})
                    except ValueError:
                        pass
                i += 5
    return items


def extract_receipt_features(pdf_path):
    """Extract ONLY the non-redundant features from a receipt."""
    text = _get_pdf_text(pdf_path)
    if not text:
        return None

    items = _parse_receipt(text)
    if not items:
        return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    # --- Core features (non-redundant with tabular data) ---
    features = {}
    features['n_line_items'] = len(items)
    features['n_unique_codes'] = len(set(codes))

    # Cost concentration (new info: HOW cost is distributed across procedures)
    if total > 0:
        shares = [a / total for a in amounts]
        features['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        features['cost_hhi'] = 1.0

    # CPT category breakdown
    n_em, n_proc, n_lab, n_crit, n_obs = 0, 0, 0, 0, 0
    cost_proc = 0.0
    em_levels = []

    for it in items:
        code, cost = it['code'], it['total']
        if code.startswith('G037'):
            n_obs += 1
        elif code in ('99291', '99292'):
            n_crit += 1
        elif code.startswith('99'):
            n_em += 1
            if code.startswith('9928') and len(code) == 5:
                em_levels.append(int(code[-1]))
        elif code[0].isdigit():
            cn = int(code)
            if 80000 <= cn <= 89999:
                n_lab += 1
            elif 10000 <= cn <= 69999:
                n_proc += 1
                cost_proc += cost

    features['n_em_codes'] = n_em
    features['n_procedure_codes'] = n_proc
    features['n_critical_care'] = n_crit
    features['has_critical_care'] = int(n_crit > 0)
    features['max_em_level'] = max(em_levels) if em_levels else 0
    features['pct_cost_procedure'] = cost_proc / total if total > 0 else 0

    return features


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        print(f"  Receipts dir not found: {receipts_dir}")
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  No PDF library! pip install pymupdf")
        return None

    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0

    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            f = extract_receipt_features(path)
            if f:
                all_feats[pid] = f
                found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i + 1}/{len(patient_ids)} done (found={found}, missing={missing})")

    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# FEATURE ENGINEERING (Curated — 18 features, zero redundancy)
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None):
    feat = df.copy()

    # --- Chronic encoding ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)

    # --- Cost transforms (ONE good transform each) ---
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # --- Key interaction ---
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']

    # --- Demographics from patients.csv ---
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left'
        )

    # --- PDF features (ONLY non-redundant ones) ---
    if pdf_df is not None:
        # Add has_pdf indicator BEFORE merge
        pdf_ids = set(pdf_df['patient_id'])
        feat['has_pdf'] = feat['patient_id'].isin(pdf_ids).astype(int)

        # Select only the curated PDF columns
        pdf_cols = ['patient_id', 'n_line_items', 'n_unique_codes', 'cost_hhi',
                    'n_em_codes', 'n_procedure_codes', 'n_critical_care',
                    'has_critical_care', 'max_em_level', 'pct_cost_procedure']
        pdf_keep = pdf_df[[c for c in pdf_cols if c in pdf_df.columns]]
        feat = feat.merge(pdf_keep, on='patient_id', how='left')

        # Fill missing PDF features with -1 sentinel (tree-friendly, not median)
        pdf_feature_cols = [c for c in pdf_keep.columns if c != 'patient_id']
        for c in pdf_feature_cols:
            feat[c] = feat[c].fillna(-1)
    else:
        # No PDFs at all — add placeholder
        feat['has_pdf'] = 0

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'cost_bin', 'strat_col'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32', 'bool')]


# ============================================================================
# MODELS (same strong regularization as v2)
# ============================================================================
def train_lgb(X_tr, y_tr, X_vl, y_vl, cols, fold=0):
    params = {
        'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
        'learning_rate': 0.03, 'num_leaves': 20, 'max_depth': 5,
        'min_child_samples': 30, 'feature_fraction': 0.7,
        'bagging_fraction': 0.7, 'bagging_freq': 5,
        'lambda_l1': 1.0, 'lambda_l2': 1.0,
        'verbose': -1, 'seed': Config.SEED + fold, 'n_jobs': -1,
    }
    dtrain = lgb.Dataset(X_tr[cols], label=y_tr)
    dval = lgb.Dataset(X_vl[cols], label=y_vl, reference=dtrain)
    return lgb.train(params, dtrain, num_boost_round=3000,
                     valid_sets=[dval],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])


def train_xgb(X_tr, y_tr, X_vl, y_vl, cols, fold=0):
    params = {
        'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
        'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 30,
        'subsample': 0.7, 'colsample_bytree': 0.7,
        'reg_alpha': 1.0, 'reg_lambda': 1.0,
        'seed': Config.SEED + fold, 'nthread': -1, 'verbosity': 0,
    }
    dtrain = xgb.DMatrix(X_tr[cols], label=y_tr)
    dval = xgb.DMatrix(X_vl[cols], label=y_vl)
    return xgb.train(params, dtrain, num_boost_round=3000,
                     evals=[(dval, 'val')], early_stopping_rounds=100,
                     verbose_eval=False)


def train_cat(X_tr, y_tr, X_vl, y_vl, cols, fold=0):
    model = CatBoostRegressor(
        loss_function='MAE', eval_metric='MAE', iterations=3000,
        learning_rate=0.03, depth=5, l2_leaf_reg=5, min_data_in_leaf=30,
        random_seed=Config.SEED + fold, verbose=0, early_stopping_rounds=100,
    )
    model.fit(X_tr[cols], y_tr, eval_set=(X_vl[cols], y_vl))
    return model


# ============================================================================
# TRAINING PIPELINE
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values

    # Stratified folds
    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])

    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)

    oof = {m: np.zeros(len(train_df)) for m in ['lgb', 'xgb', 'cat']}
    test_preds = {m: np.zeros(len(test_df)) for m in ['lgb', 'xgb', 'cat']}
    fold_scores = []

    for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
        print(f"\n--- FOLD {fold + 1}/{Config.N_FOLDS} ---")
        X_tr, X_vl = train_df.iloc[ti], train_df.iloc[vi]
        y_tr, y_vl = y[ti], y[vi]

        # LightGBM
        m = train_lgb(X_tr, y_tr, X_vl, y_vl, feat_cols, fold)
        oof['lgb'][vi] = m.predict(X_vl[feat_cols])
        test_preds['lgb'] += m.predict(test_df[feat_cols]) / Config.N_FOLDS

        # XGBoost
        m = train_xgb(X_tr, y_tr, X_vl, y_vl, feat_cols, fold)
        oof['xgb'][vi] = m.predict(xgb.DMatrix(X_vl[feat_cols]))
        test_preds['xgb'] += m.predict(xgb.DMatrix(test_df[feat_cols])) / Config.N_FOLDS

        # CatBoost
        m = train_cat(X_tr, y_tr, X_vl, y_vl, feat_cols, fold)
        oof['cat'][vi] = m.predict(X_vl[feat_cols])
        test_preds['cat'] += m.predict(test_df[feat_cols]) / Config.N_FOLDS

        # Fold score
        ens = (Config.W_LGB * oof['lgb'][vi] + Config.W_XGB * oof['xgb'][vi] +
               Config.W_CAT * oof['cat'][vi])
        s = {name: mean_absolute_error(y_vl, oof[name][vi]) for name in ['lgb', 'xgb', 'cat']}
        s['ens'] = mean_absolute_error(y_vl, ens)
        s['fold'] = fold + 1
        fold_scores.append(s)
        print(f"  LGB={s['lgb']:.2f}  XGB={s['xgb']:.2f}  CAT={s['cat']:.2f}  ENS={s['ens']:.2f}")

    # Overall
    oof_ens = Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] + Config.W_CAT * oof['cat']
    test_final = Config.W_LGB * test_preds['lgb'] + Config.W_XGB * test_preds['xgb'] + Config.W_CAT * test_preds['cat']
    cv = mean_absolute_error(y, oof_ens)

    print(f"\n{'='*60}")
    for name in ['lgb', 'xgb', 'cat']:
        print(f"OOF MAE — {name.upper():8s}: {mean_absolute_error(y, oof[name]):.2f}")
    print(f"OOF MAE — ENSEMBLE: {cv:.2f} (w={Config.W_LGB}/{Config.W_XGB}/{Config.W_CAT})")

    # Report equal-weight for comparison
    eq = mean_absolute_error(y, (oof['lgb'] + oof['xgb'] + oof['cat']) / 3)
    print(f"OOF MAE — Equal 1/3: {eq:.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv, fold_scores


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2: ED Cost Forecasting — v3 (Precision Features)")
    print("=" * 70)

    # Detect paths
    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.TRAIN_PATH = os.path.join(base, 'ed_cost_train.csv')
            Config.TEST_PATH = os.path.join(base, 'ed_cost_test.csv')
            Config.PATIENTS_PATH = os.path.join(base, 'patients.csv')
            for rc in [os.path.join(base, 'receipts_pdf'),
                       os.path.join(base, '..', 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
                    break
            break

    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")
    print(f"PDF: PyMuPDF={_HAS_FITZ}, pdfplumber={_HAS_PDFPLUMBER}")

    # Load
    print("\n[1/5] Loading data...")
    train = pd.read_csv(Config.TRAIN_PATH)
    test = pd.read_csv(Config.TEST_PATH)
    patients = None
    if os.path.exists(Config.PATIENTS_PATH):
        patients = pd.read_csv(Config.PATIENTS_PATH)
        print(f"  Patients: {len(patients)} rows")

    # PDF
    print("\n[2/5] Extracting PDF features...")
    all_pids = sorted(set(train['patient_id'].tolist() + test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    # Features
    print("\n[3/5] Building features...")
    train_feat = build_features(train, pdf_df, patients)
    test_feat = build_features(test, pdf_df, patients)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    # Fill remaining NaN (demographics from patients.csv)
    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    # Train
    print(f"\n[4/5] Training (weights: LGB={Config.W_LGB}, XGB={Config.W_XGB}, CAT={Config.W_CAT})...")
    test_pred, cv_mae, fold_scores = run_ensemble(train_feat, test_feat, feat_cols)

    # Submit
    print(f"\n[5/5] Saving submission...")
    sub = pd.DataFrame({'patient_id': test['patient_id'],
                        'ed_cost_next3y_usd': np.round(test_pred, 2)})
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    for fs in fold_scores:
        print(f"  Fold {fs['fold']}: LGB={fs['lgb']:.2f} XGB={fs['xgb']:.2f} "
              f"CAT={fs['cat']:.2f} ENS={fs['ens']:.2f}")
    print(f"{'='*70}")
    return sub, cv_mae


if __name__ == '__main__':
    main()

AgentDS Challenge 2: ED Cost Forecasting — v3 (Precision Features)
Receipts: .\receipts_pdf
PDF: PyMuPDF=True, pdfplumber=False

[1/5] Loading data...
  Patients: 4000 rows

[2/5] Extracting PDF features...
  PDF backend: PyMuPDF
  1000/4000 done (found=234, missing=766)
  2000/4000 done (found=1234, missing=766)
  3000/4000 done (found=2234, missing=766)
  4000/4000 done (found=2319, missing=1681)
  Final: 2319 extracted, 1681 not found

[3/5] Building features...
  Features (21): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'chronic_x_sqrt_cost', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region', 'has_pdf', 'n_line_items', 'n_unique_codes', 'cost_hhi', 'n_em_codes', 'n_procedure_codes', 'n_critical_care', 'has_critical_care', 'max_em_level', 'pct_cost_procedure']

[4/5] Training (weights: LGB=0.2, XGB=0.2, CAT=0.6)...

--- FOLD 1/5 ---
Training until validation scores don't improve for 100 rounds
Early stop

# Iteration 4
- 485.3003

In [5]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v4 (Cross-Challenge Features)
================================================================================
The hint says "Try incorporating relevant information outside this table!"

Cross-challenge data sources (all joinable via patient_id):
  1. admissions_train/test.csv → charlson_band, los_days, acuity, ed_visits_6m
  2. stays_train/test.csv → unit_type, admission_reason
  3. discharge_notes.json → clinical severity keywords
  4. vitals_timeseries.json → physiological baselines & trends

Changes from v3 (CV=457, Test=472):
  - Add ~20 cross-challenge features (esp. charlson_band, the #1 cost predictor)
  - Add discharge note keyword features
  - Add vitals summary statistics
  - 10-fold CV for stability
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import json
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


# ============================================================================
# CONFIG
# ============================================================================
class Config:
    SEED = 42
    N_FOLDS = 10
    W_LGB = 0.20
    W_XGB = 0.20
    W_CAT = 0.60
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (proven from v3)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, _, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def extract_receipt_features(pdf_path):
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}
    f['n_line_items'] = len(items)
    f['n_unique_codes'] = len(set(codes))

    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    n_em, n_proc, n_crit = 0, 0, 0
    cost_proc = 0.0
    em_levels = []
    has_obs = 0

    for it in items:
        code, cost = it['code'], it['total']
        if code.startswith('G037'):
            has_obs = 1
        elif code in ('99291', '99292'):
            n_crit += 1
        elif code.startswith('99'):
            n_em += 1
            if code.startswith('9928') and len(code) == 5:
                em_levels.append(int(code[-1]))
        elif code[0].isdigit():
            try:
                cn = int(code)
                if 10000 <= cn <= 69999:
                    n_proc += 1
                    cost_proc += cost
            except ValueError: pass

    f['n_procedure_codes'] = n_proc
    f['has_critical_care'] = int(n_crit > 0)
    f['has_observation'] = has_obs
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['pct_cost_procedure'] = cost_proc / total if total > 0 else 0

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# CROSS-CHALLENGE: ADMISSIONS FEATURES
# ============================================================================
def load_admissions_features(base_dir):
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Admissions data: NOT FOUND")
        return None, None

    adm = pd.concat(dfs, ignore_index=True)
    print(f"  Admissions: {len(adm)} rows, cols={adm.columns.tolist()}")

    agg = adm.groupby('patient_id').agg(
        n_admissions=('patient_id', 'count'),
        charlson_max=('charlson_band', 'max'),
        charlson_mean=('charlson_band', 'mean'),
        total_los=('los_days', 'sum'),
        mean_los=('los_days', 'mean'),
        max_los=('los_days', 'max'),
        pct_emergent=('acuity_emergent', 'mean'),
        max_ed_visits_6m=('ed_visits_6m', 'max'),
        mean_ed_visits_6m=('ed_visits_6m', 'mean'),
    ).reset_index()

    if 'discharge_weekday' in adm.columns:
        weekend_pct = adm.groupby('patient_id')['discharge_weekday'].apply(
            lambda x: (x.isin([6, 7])).mean()
        ).reset_index(name='pct_weekend_discharge')
        agg = agg.merge(weekend_pct, on='patient_id', how='left')

    if 'primary_dx' in adm.columns:
        dx_nunique = adm.groupby('patient_id')['primary_dx'].nunique().reset_index(
            name='n_dx_types')
        agg = agg.merge(dx_nunique, on='patient_id', how='left')

    print(f"  Admissions features: {len(agg)} patients, {len(agg.columns)-1} features")
    return agg, adm  # Return raw too for notes mapping


# ============================================================================
# CROSS-CHALLENGE: STAYS FEATURES
# ============================================================================
def load_stays_features(base_dir):
    dfs = []
    for fname in ['stays_train.csv', 'stays_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Stays data: NOT FOUND")
        return None, None

    stays = pd.concat(dfs, ignore_index=True)
    print(f"  Stays: {len(stays)} rows, cols={stays.columns.tolist()}")

    agg = stays.groupby('patient_id').agg(
        n_stays=('patient_id', 'count'),
    ).reset_index()

    if 'unit_type' in stays.columns:
        stepdown = stays.groupby('patient_id')['unit_type'].apply(
            lambda x: int('stepdown' in x.values)
        ).reset_index(name='has_stepdown')
        agg = agg.merge(stepdown, on='patient_id', how='left')

    if 'admission_reason' in stays.columns:
        n_reasons = stays.groupby('patient_id')['admission_reason'].nunique().reset_index(
            name='n_admission_reasons')
        agg = agg.merge(n_reasons, on='patient_id', how='left')

        copd = stays.groupby('patient_id')['admission_reason'].apply(
            lambda x: int('COPD' in x.values)
        ).reset_index(name='has_copd_stay')
        agg = agg.merge(copd, on='patient_id', how='left')

    print(f"  Stays features: {len(agg)} patients, {len(agg.columns)-1} features")
    return agg, stays  # Return raw too for vitals mapping


# ============================================================================
# CROSS-CHALLENGE: DISCHARGE NOTES FEATURES
# ============================================================================
def load_discharge_notes_features(base_dir, raw_adm_df=None):
    path = os.path.join(base_dir, 'discharge_notes.json')
    if not os.path.exists(path):
        print("  Discharge notes: NOT FOUND")
        return None

    with open(path, 'r') as f:
        notes = json.load(f)
    print(f"  Discharge notes: {len(notes)} notes loaded")

    aid_to_pid = {}
    if raw_adm_df is not None:
        for _, row in raw_adm_df.iterrows():
            aid_to_pid[row['admission_id']] = row['patient_id']

    high_severity = ['icu', 'critical', 'intubat', 'ventilat', 'unstable', 'sepsis',
                     'shock', 'arrest', 'emergent', 'acute', 'severe', 'deteriorat']
    moderate_severity = ['readmi', 'complicat', 'infection', 'fever', 'tachycard',
                         'hypotens', 'decompens', 'exacerbat']
    positive_keywords = ['improv', 'stable', 'ambulat', 'tolerat', 'discharg',
                         'home', 'recover', 'resolv', 'independent']

    patient_scores = {}
    for note_entry in notes:
        aid = note_entry.get('admission_id')
        text = note_entry.get('note', '').lower()
        pid = aid_to_pid.get(aid)
        if pid is None:
            continue
        if pid not in patient_scores:
            patient_scores[pid] = {'high': 0, 'moderate': 0, 'positive': 0,
                                   'note_len': 0, 'n_notes': 0}
        patient_scores[pid]['n_notes'] += 1
        patient_scores[pid]['note_len'] += len(text)
        for kw in high_severity:
            if kw in text: patient_scores[pid]['high'] += 1
        for kw in moderate_severity:
            if kw in text: patient_scores[pid]['moderate'] += 1
        for kw in positive_keywords:
            if kw in text: patient_scores[pid]['positive'] += 1

    if not patient_scores:
        return None

    rows = []
    for pid, sc in patient_scores.items():
        n = max(sc['n_notes'], 1)
        rows.append({
            'patient_id': pid,
            'note_high_severity': sc['high'] / n,
            'note_moderate_severity': sc['moderate'] / n,
            'note_positive': sc['positive'] / n,
            'note_severity_ratio': (sc['high'] + sc['moderate']) / max(sc['positive'], 1),
            'mean_note_len': sc['note_len'] / n,
        })
    df = pd.DataFrame(rows)
    print(f"  Note features: {len(df)} patients, {len(df.columns)-1} features")
    return df


# ============================================================================
# CROSS-CHALLENGE: VITALS FEATURES
# ============================================================================
def load_vitals_features(base_dir, raw_stays_df=None):
    path = os.path.join(base_dir, 'vitals_timeseries.json')
    if not os.path.exists(path):
        print("  Vitals data: NOT FOUND")
        return None

    with open(path, 'r') as f:
        vitals = json.load(f)
    print(f"  Vitals: {len(vitals)} stay records loaded")

    sid_to_pid = {}
    if raw_stays_df is not None:
        for _, row in raw_stays_df.iterrows():
            sid_to_pid[row['stay_id']] = row['patient_id']

    patient_vitals = {}
    for entry in vitals:
        sid = entry.get('stay_id')
        pid = sid_to_pid.get(sid)
        if pid is None: continue
        if pid not in patient_vitals:
            patient_vitals[pid] = {'hr': [], 'sbp': [], 'dbp': [], 'temp': [], 'rr': []}
        for day in entry.get('days', []):
            for key, store_key in [('hr','hr'),('sbp','sbp'),('dbp','dbp'),('temp_c','temp'),('rr','rr')]:
                val = day.get(key)
                if val is not None and isinstance(val, (int, float)):
                    patient_vitals[pid][store_key].append(float(val))

    if not patient_vitals:
        return None

    rows = []
    for pid, v in patient_vitals.items():
        row = {'patient_id': pid}
        for vital, vals in v.items():
            if vals:
                arr = np.array(vals)
                row[f'{vital}_mean'] = arr.mean()
                row[f'{vital}_std'] = arr.std() if len(arr) > 1 else 0
                if len(arr) >= 6:
                    row[f'{vital}_trend'] = arr[-3:].mean() - arr[:3].mean()
                else:
                    row[f'{vital}_trend'] = 0

        hr = np.array(v.get('hr', []))
        sbp = np.array(v.get('sbp', []))
        temp = np.array(v.get('temp', []))
        rr = np.array(v.get('rr', []))

        n_abnormal = 0
        if len(hr): n_abnormal += int(((hr > 100) | (hr < 60)).sum())
        if len(sbp): n_abnormal += int(((sbp > 140) | (sbp < 90)).sum())
        if len(temp): n_abnormal += int((temp > 38.0).sum())
        if len(rr): n_abnormal += int((rr > 20).sum())

        total_readings = sum(len(v[k]) for k in v)
        row['n_abnormal_vitals'] = n_abnormal
        row['pct_abnormal_vitals'] = n_abnormal / max(total_readings, 1)
        rows.append(row)

    df = pd.DataFrame(rows)
    print(f"  Vitals features: {len(df)} patients, {len(df.columns)-1} features")
    return df


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, adm_df=None,
                   stays_df=None, notes_df=None, vitals_df=None):
    feat = df.copy()

    # Core features
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']

    # Demographics
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

    # Admissions (THE KEY!)
    if adm_df is not None:
        feat['has_admission'] = feat['patient_id'].isin(adm_df['patient_id']).astype(int)
        feat = feat.merge(adm_df, on='patient_id', how='left')
        for c in [col for col in adm_df.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)
        if 'charlson_max' in feat.columns:
            feat['charlson_x_cost'] = feat['charlson_max'].clip(0) * feat['sqrt_prior_cost']
            feat['charlson_x_visits'] = feat['charlson_max'].clip(0) * feat['prior_ed_visits_5y']

    # Stays
    if stays_df is not None:
        feat['has_stay'] = feat['patient_id'].isin(stays_df['patient_id']).astype(int)
        feat = feat.merge(stays_df, on='patient_id', how='left')
        for c in [col for col in stays_df.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)

    # Discharge notes
    if notes_df is not None:
        feat['has_notes'] = feat['patient_id'].isin(notes_df['patient_id']).astype(int)
        feat = feat.merge(notes_df, on='patient_id', how='left')
        for c in [col for col in notes_df.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)

    # Vitals
    if vitals_df is not None:
        feat['has_vitals'] = feat['patient_id'].isin(vitals_df['patient_id']).astype(int)
        feat = feat.merge(vitals_df, on='patient_id', how='left')
        for c in [col for col in vitals_df.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)

    # PDF
    if pdf_df is not None:
        feat['has_pdf'] = feat['patient_id'].isin(pdf_df['patient_id']).astype(int)
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        for c in [col for col in pdf_df.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)
    else:
        feat['has_pdf'] = 0

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'cost_bin', 'strat_col',
               'primary_dx', 'unit_type', 'admission_reason',
               'admission_id', 'stay_id', 'note', 'readmit_30d',
               'discharge_ready_day11', 'acuity_emergent',
               'discharge_weekday', 'charlson_band', 'los_days', 'ed_visits_6m'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32', 'bool', 'uint8')]


# ============================================================================
# MODELS
# ============================================================================
def train_lgb(X_tr, y_tr, X_vl, y_vl, cols, fold=0):
    params = {
        'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
        'learning_rate': 0.03, 'num_leaves': 24, 'max_depth': 5,
        'min_child_samples': 25, 'feature_fraction': 0.7,
        'bagging_fraction': 0.7, 'bagging_freq': 5,
        'lambda_l1': 1.0, 'lambda_l2': 1.0,
        'verbose': -1, 'seed': Config.SEED + fold, 'n_jobs': -1,
    }
    dtrain = lgb.Dataset(X_tr[cols], label=y_tr)
    dval = lgb.Dataset(X_vl[cols], label=y_vl, reference=dtrain)
    return lgb.train(params, dtrain, num_boost_round=5000,
                     valid_sets=[dval],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])


def train_xgb(X_tr, y_tr, X_vl, y_vl, cols, fold=0):
    params = {
        'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
        'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 25,
        'subsample': 0.7, 'colsample_bytree': 0.7,
        'reg_alpha': 1.0, 'reg_lambda': 1.0,
        'seed': Config.SEED + fold, 'nthread': -1, 'verbosity': 0,
    }
    dtrain = xgb.DMatrix(X_tr[cols], label=y_tr)
    dval = xgb.DMatrix(X_vl[cols], label=y_vl)
    return xgb.train(params, dtrain, num_boost_round=5000,
                     evals=[(dval, 'val')], early_stopping_rounds=100,
                     verbose_eval=False)


def train_cat(X_tr, y_tr, X_vl, y_vl, cols, fold=0):
    model = CatBoostRegressor(
        loss_function='MAE', eval_metric='MAE', iterations=5000,
        learning_rate=0.03, depth=5, l2_leaf_reg=5, min_data_in_leaf=25,
        random_seed=Config.SEED + fold, verbose=0, early_stopping_rounds=100,
    )
    model.fit(X_tr[cols], y_tr, eval_set=(X_vl[cols], y_vl))
    return model


# ============================================================================
# TRAINING PIPELINE
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])

    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)

    oof = {m: np.zeros(len(train_df)) for m in ['lgb', 'xgb', 'cat']}
    test_preds = {m: np.zeros(len(test_df)) for m in ['lgb', 'xgb', 'cat']}

    for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
        X_tr, X_vl = train_df.iloc[ti], train_df.iloc[vi]
        y_tr, y_vl = y[ti], y[vi]

        m = train_lgb(X_tr, y_tr, X_vl, y_vl, feat_cols, fold)
        oof['lgb'][vi] = m.predict(X_vl[feat_cols])
        test_preds['lgb'] += m.predict(test_df[feat_cols]) / Config.N_FOLDS

        m = train_xgb(X_tr, y_tr, X_vl, y_vl, feat_cols, fold)
        oof['xgb'][vi] = m.predict(xgb.DMatrix(X_vl[feat_cols]))
        test_preds['xgb'] += m.predict(xgb.DMatrix(test_df[feat_cols])) / Config.N_FOLDS

        m = train_cat(X_tr, y_tr, X_vl, y_vl, feat_cols, fold)
        oof['cat'][vi] = m.predict(X_vl[feat_cols])
        test_preds['cat'] += m.predict(test_df[feat_cols]) / Config.N_FOLDS

        ens_vl = Config.W_LGB*oof['lgb'][vi] + Config.W_XGB*oof['xgb'][vi] + Config.W_CAT*oof['cat'][vi]
        print(f"  Fold {fold+1:2d}: CAT={mean_absolute_error(y_vl, oof['cat'][vi]):.2f}  "
              f"ENS={mean_absolute_error(y_vl, ens_vl):.2f}")

    oof_ens = Config.W_LGB*oof['lgb'] + Config.W_XGB*oof['xgb'] + Config.W_CAT*oof['cat']
    test_final = Config.W_LGB*test_preds['lgb'] + Config.W_XGB*test_preds['xgb'] + Config.W_CAT*test_preds['cat']
    cv = mean_absolute_error(y, oof_ens)

    print(f"\n{'='*60}")
    for name in ['lgb', 'xgb', 'cat']:
        print(f"OOF MAE — {name.upper():8s}: {mean_absolute_error(y, oof[name]):.2f}")
    print(f"OOF MAE — Ensemble ({Config.W_LGB}/{Config.W_XGB}/{Config.W_CAT}): {cv:.2f}")
    eq = mean_absolute_error(y, (oof['lgb'] + oof['xgb'] + oof['cat']) / 3)
    print(f"OOF MAE — Equal 1/3: {eq:.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v4 (Cross-Challenge Features)")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")
    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")

    # [1] Core data
    print("\n[1/7] Loading core data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    # [2] Admissions
    print("\n[2/7] Loading admissions data...")
    adm_result = load_admissions_features(bd)
    if adm_result[0] is not None:
        adm_df, raw_adm = adm_result
    else:
        adm_df, raw_adm = None, None

    # [3] Stays
    print("\n[3/7] Loading stays data...")
    stays_result = load_stays_features(bd)
    if stays_result[0] is not None:
        stays_df, raw_stays = stays_result
    else:
        stays_df, raw_stays = None, None

    # [4] Discharge notes
    print("\n[4/7] Loading discharge notes...")
    notes_df = load_discharge_notes_features(bd, raw_adm)

    # [5] Vitals
    print("\n[5/7] Loading vitals data...")
    vitals_df = load_vitals_features(bd, raw_stays)

    # [6] PDFs
    print("\n[6/7] Extracting PDF features...")
    all_pids = sorted(set(train['patient_id'].tolist() + test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    # [7] Build & train
    print("\n[7/7] Building features & training...")
    train_feat = build_features(train, pdf_df, patients, adm_df, stays_df, notes_df, vitals_df)
    test_feat = build_features(test, pdf_df, patients, adm_df, stays_df, notes_df, vitals_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n  Training {Config.N_FOLDS}-fold ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    sub = pd.DataFrame({'patient_id': test['patient_id'],
                        'ed_cost_next3y_usd': np.round(test_pred, 2)})
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v4 (Cross-Challenge Features)
Base dir: .
Receipts: .\receipts_pdf

[1/7] Loading core data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/7] Loading admissions data...
  Admissions: 10000 rows, cols=['admission_id', 'patient_id', 'primary_dx', 'los_days', 'acuity_emergent', 'charlson_band', 'ed_visits_6m', 'discharge_weekday', 'readmit_30d']
  Admissions features: 4000 patients, 11 features

[3/7] Loading stays data...
  Stays: 2000 rows, cols=['stay_id', 'patient_id', 'unit_type', 'admission_reason', 'discharge_ready_day11']
  Stays features: 2000 patients, 4 features

[4/7] Loading discharge notes...
  Discharge notes: 10000 notes loaded
  Note features: 4000 patients, 5 features

[5/7] Loading vitals data...
  Vitals: 2000 stay records loaded
  Vitals features: 2000 patients, 17 features

[6/7] Extracting PDF features...
  PDF backend: PyMuPDF
  1000/4000 (found=234, missing=766)
  2000/4000 (found=1234, missing=766)
  3000/4000 (found=2234, missing=766)
 

# Iteration 5
- 480.4429

In [7]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v5 (Surgical Precision)
================================================================================
POST-MORTEM of v4 failure (CV=471, Test=485 — WORSE than v3):
  × 63 features for 2000 samples = massive overfit
  × Kitchen sink cross-challenge (stays, vitals, notes = 50% missing = noise)
  × LGB/XGB dragged down CatBoost (best model)
  × -1 sentinel for missing values (suboptimal)

v5 PHILOSOPHY: "Less is more. Surgical, not shotgun."
  1. CatBoost ONLY (consistently best by 10+ MAE over LGB/XGB)
  2. 3 diverse CatBoost configs → feature-set + param diversity
  3. v3 proven base (21 feat, CV=457) + surgical admissions (3 feat)
  4. Admissions: charlson_max, pct_readmitted, n_admissions (FULL coverage)
  5. pct_readmitted = mean(readmit_30d) — DIRECT complexity measure
  6. NaN for missing PDF features (native CatBoost handling)
  7. DROP all: stays, vitals, discharge notes (50% coverage = noise)
  8. ~25 total features
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


# ============================================================================
# CONFIG
# ============================================================================
class Config:
    SEED = 42
    N_FOLDS = 5
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (proven from v3)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, _, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def extract_receipt_features(pdf_path):
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}
    f['n_line_items'] = len(items)
    f['n_unique_codes'] = len(set(codes))

    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    n_em, n_proc, n_crit = 0, 0, 0
    cost_proc = 0.0
    em_levels = []
    has_obs = 0

    for it in items:
        code, cost = it['code'], it['total']
        if code.startswith('G037'):
            has_obs = 1
        elif code in ('99291', '99292'):
            n_crit += 1
        elif code.startswith('99'):
            n_em += 1
            if code.startswith('9928') and len(code) == 5:
                em_levels.append(int(code[-1]))
        elif code[0].isdigit():
            try:
                cn = int(code)
                if 10000 <= cn <= 69999:
                    n_proc += 1
                    cost_proc += cost
            except ValueError: pass

    f['n_procedure_codes'] = n_proc
    f['has_critical_care'] = int(n_crit > 0)
    f['has_observation'] = has_obs
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['pct_cost_procedure'] = cost_proc / total if total > 0 else 0

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# SURGICAL CROSS-CHALLENGE: ADMISSIONS ONLY
# ============================================================================
def load_admissions_surgical(base_dir):
    """
    Extract ONLY the highest-value admissions features per patient.
    Key insight: Use readmit_30d (Challenge 1 target) as a feature —
    patients with frequent readmissions have higher future ED costs.
    """
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None

    adm = pd.concat(dfs, ignore_index=True)
    print(f"  Admissions: {len(adm)} rows for "
          f"{adm['patient_id'].nunique()} patients")

    # Core aggregates (ONLY the most predictive)
    agg = adm.groupby('patient_id').agg(
        n_admissions=('patient_id', 'count'),
        charlson_max=('charlson_band', 'max'),
        mean_los=('los_days', 'mean'),
        max_ed_visits_6m=('ed_visits_6m', 'max'),
    ).reset_index()

    # KEY FEATURE: readmission rate (from rows that have it)
    # admissions_train has readmit_30d, admissions_test doesn't
    if 'readmit_30d' in adm.columns:
        # Compute per-patient readmission rate (only from rows with the label)
        has_label = adm.dropna(subset=['readmit_30d'])
        if len(has_label) > 0:
            readmit_rate = has_label.groupby('patient_id')['readmit_30d'].mean()\
                .reset_index(name='pct_readmitted')
            agg = agg.merge(readmit_rate, on='patient_id', how='left')
            # Patients without readmission data: leave as NaN (native handling)
            print(f"  Readmission rate: available for "
                  f"{len(readmit_rate)} patients")

    print(f"  Admissions features: {[c for c in agg.columns if c != 'patient_id']}")
    return agg


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, adm_df=None):
    feat = df.copy()

    # --- Core features (proven in v3) ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']

    # --- Demographics (from patients.csv) ---
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(np.nan).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

    # --- Surgical admissions features ---
    if adm_df is not None:
        feat = feat.merge(adm_df, on='patient_id', how='left')
        # NaN stays NaN — CatBoost handles natively
        # Key interactions
        if 'charlson_max' in feat.columns:
            # Only compute interaction where charlson is not NaN
            feat['charlson_x_cost'] = feat['charlson_max'] * feat['sqrt_prior_cost']
            feat['charlson_x_visits'] = feat['charlson_max'] * feat['prior_ed_visits_5y']

    # --- PDF features (leave NaN for missing — native handling) ---
    if pdf_df is not None:
        feat['has_pdf'] = feat['patient_id'].isin(pdf_df['patient_id']).astype(int)
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        # NaN stays NaN for CatBoost native handling
    else:
        feat['has_pdf'] = 0

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'cost_bin', 'strat_col',
               'admission_id', 'primary_dx', 'readmit_30d',
               'discharge_weekday', 'acuity_emergent', 'charlson_band',
               'los_days', 'ed_visits_6m', 'stay_id', 'unit_type',
               'admission_reason', 'discharge_ready_day11', 'note'}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if df[c].dtype in ('int64', 'float64', 'int32', 'float32', 'bool', 'uint8'):
            cols.append(c)
    return cols


# ============================================================================
# DIVERSE CATBOOST ENSEMBLE
# ============================================================================
CATBOOST_CONFIGS = [
    {   # Config A: Shallow, strong regularization (most generalizable)
        'name': 'shallow_reg',
        'depth': 4, 'l2_leaf_reg': 8, 'min_data_in_leaf': 40,
        'learning_rate': 0.02, 'rsm': 0.8,
    },
    {   # Config B: Medium depth, moderate regularization
        'name': 'medium',
        'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
        'learning_rate': 0.03, 'rsm': 0.7,
    },
    {   # Config C: Medium with different subsample
        'name': 'medium_diverse',
        'depth': 5, 'l2_leaf_reg': 3, 'min_data_in_leaf': 30,
        'learning_rate': 0.025, 'rsm': 0.6, 'bagging_temperature': 0.5,
    },
]


def run_diverse_catboost(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    # Stratified folds
    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    # Storage for each config
    all_oof = {}
    all_test = {}

    for cfg in CATBOOST_CONFIGS:
        name = cfg.pop('name')
        oof = np.zeros(n_train)
        test_pred = np.zeros(n_test)

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]

            model = CatBoostRegressor(
                loss_function='MAE', eval_metric='MAE',
                iterations=5000, early_stopping_rounds=100,
                random_seed=Config.SEED + fold + hash(name) % 1000,
                verbose=0,
                **cfg
            )
            model.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof[vi] = model.predict(X_vl)
            test_pred += model.predict(test_df[feat_cols]) / Config.N_FOLDS

        mae = mean_absolute_error(y, oof)
        all_oof[name] = oof
        all_test[name] = test_pred
        print(f"  {name:20s}: OOF MAE = {mae:.2f}")
        cfg['name'] = name  # restore

    # Equal blend of all configs
    n_configs = len(CATBOOST_CONFIGS)
    oof_blend = sum(all_oof.values()) / n_configs
    test_blend = sum(all_test.values()) / n_configs
    cv_mae = mean_absolute_error(y, oof_blend)
    print(f"\n  Diverse blend ({n_configs} configs): OOF MAE = {cv_mae:.2f}")

    # Also report per-fold for diagnostics
    for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
        fold_mae = mean_absolute_error(y[vi], oof_blend[vi])
        print(f"    Fold {fold+1}: {fold_mae:.2f}")

    test_blend = np.clip(test_blend, 0, None)
    return test_blend, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v5 (Surgical Precision)")
    print("=" * 70)

    # Auto-detect
    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")

    # [1] Core data
    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    # [2] Surgical admissions (ONLY high-value features)
    print("\n[2/5] Loading admissions (surgical)...")
    adm_df = load_admissions_surgical(bd)

    # [3] PDF receipts
    print("\n[3/5] Extracting PDF features...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    # [4] Build features
    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, adm_df)
    test_feat = build_features(test, pdf_df, patients, adm_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    # Ensure numeric (but keep NaN — CatBoost handles natively)
    for c in feat_cols:
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce')
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce')

    # [5] Train diverse CatBoost ensemble
    print(f"\n[5/5] Training diverse CatBoost ensemble...")
    test_pred, cv_mae = run_diverse_catboost(train_feat, test_feat, feat_cols)

    # Save
    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v5 (Surgical Precision)
Base dir: .

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading admissions (surgical)...
  Admissions: 10000 rows for 4000 patients
  Readmission rate: available for 2944 patients
  Admissions features: ['n_admissions', 'charlson_max', 'mean_los', 'max_ed_visits_6m', 'pct_readmitted']

[3/5] Extracting PDF features...
  PDF backend: PyMuPDF
  1000/4000 (found=234, missing=766)
  2000/4000 (found=1234, missing=766)
  3000/4000 (found=2234, missing=766)
  4000/4000 (found=2319, missing=1681)
  Final: 2319 extracted, 1681 not found

[4/5] Building features...
  Features (27): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'chronic_x_sqrt_cost', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region', 'n_admissions', 'charlson_max', 'mean_los', 'max_ed_visits_6m', 'pct_readmitted', 'charlson_x_cost', 'charlson_x_visits', 'has_pdf', 'n_line_items'

# Iteratoin 6
- 473.1824

In [9]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v6 (Variance Reduction)
================================================================================
Base: v3 (our proven best at Test=472)
Strategy: Don't add features. Reduce TEST VARIANCE.

Changes from v3:
  1. 7-fold CV (was 5) → more training data per model, better test avg
  2. 3-seed averaging → stabilizes test predictions
  3. Stronger LGB/XGB regularization → closes gap with CatBoost
  4. Keep v3's feature set exactly (21 features with PDF+patients)
  5. Keep 0.2/0.2/0.6 ensemble weights (proven)

Expected: Same CV range (~455-460) but closer TEST score (~465-468)
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


# ============================================================================
# CONFIG
# ============================================================================
class Config:
    SEED = 42
    N_FOLDS = 7          # 7-fold (was 5): more training data per model
    N_SEEDS = 3          # Seed averaging: reduces test variance
    W_LGB = 0.20
    W_XGB = 0.20
    W_CAT = 0.60
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (identical to v3)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, _, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def extract_receipt_features(pdf_path):
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}
    f['n_line_items'] = len(items)
    f['n_unique_codes'] = len(set(codes))

    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    n_em, n_proc, n_crit = 0, 0, 0
    cost_proc = 0.0
    em_levels = []
    has_obs = 0

    for it in items:
        code, cost = it['code'], it['total']
        if code.startswith('G037'):
            has_obs = 1
        elif code in ('99291', '99292'):
            n_crit += 1
        elif code.startswith('99'):
            n_em += 1
            if code.startswith('9928') and len(code) == 5:
                em_levels.append(int(code[-1]))
        elif code[0].isdigit():
            try:
                cn = int(code)
                if 10000 <= cn <= 69999:
                    n_proc += 1
                    cost_proc += cost
            except ValueError: pass

    f['n_procedure_codes'] = n_proc
    f['has_critical_care'] = int(n_crit > 0)
    f['has_observation'] = has_obs
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['pct_cost_procedure'] = cost_proc / total if total > 0 else 0

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# FEATURE ENGINEERING (identical to v3 — proven feature set)
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None):
    feat = df.copy()

    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']

    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

    if pdf_df is not None:
        feat['has_pdf'] = feat['patient_id'].isin(pdf_df['patient_id']).astype(int)
        pdf_cols = ['patient_id', 'n_line_items', 'n_unique_codes', 'cost_hhi',
                    'n_procedure_codes', 'has_critical_care', 'has_observation',
                    'max_em_level', 'pct_cost_procedure']
        pdf_keep = pdf_df[[c for c in pdf_cols if c in pdf_df.columns]]
        feat = feat.merge(pdf_keep, on='patient_id', how='left')
        for c in [col for col in pdf_keep.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)
    else:
        feat['has_pdf'] = 0

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'cost_bin', 'strat_col'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32', 'bool', 'uint8')]


# ============================================================================
# MODELS
# ============================================================================
def train_lgb(X_tr, y_tr, X_vl, y_vl, cols, seed=0):
    """LGB with STRONGER regularization than v3 (depth=4, l2=2, min=40)"""
    params = {
        'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
        'learning_rate': 0.02, 'num_leaves': 16, 'max_depth': 4,
        'min_child_samples': 40, 'feature_fraction': 0.7,
        'bagging_fraction': 0.7, 'bagging_freq': 5,
        'lambda_l1': 2.0, 'lambda_l2': 2.0,
        'verbose': -1, 'seed': seed, 'n_jobs': -1,
    }
    dtrain = lgb.Dataset(X_tr[cols], label=y_tr)
    dval = lgb.Dataset(X_vl[cols], label=y_vl, reference=dtrain)
    return lgb.train(params, dtrain, num_boost_round=5000,
                     valid_sets=[dval],
                     callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])


def train_xgb(X_tr, y_tr, X_vl, y_vl, cols, seed=0):
    """XGB with STRONGER regularization (depth=4, l2=2, min=40)"""
    params = {
        'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
        'learning_rate': 0.02, 'max_depth': 4, 'min_child_weight': 40,
        'subsample': 0.7, 'colsample_bytree': 0.7,
        'reg_alpha': 2.0, 'reg_lambda': 2.0,
        'seed': seed, 'nthread': -1, 'verbosity': 0,
    }
    dtrain = xgb.DMatrix(X_tr[cols], label=y_tr)
    dval = xgb.DMatrix(X_vl[cols], label=y_vl)
    return xgb.train(params, dtrain, num_boost_round=5000,
                     evals=[(dval, 'val')], early_stopping_rounds=150,
                     verbose_eval=False)


def train_cat(X_tr, y_tr, X_vl, y_vl, cols, seed=0):
    """CatBoost with v3 proven settings"""
    model = CatBoostRegressor(
        loss_function='MAE', eval_metric='MAE', iterations=5000,
        learning_rate=0.03, depth=5, l2_leaf_reg=5, min_data_in_leaf=25,
        random_seed=seed, verbose=0, early_stopping_rounds=100,
    )
    model.fit(X_tr[cols], y_tr, eval_set=(X_vl[cols], y_vl))
    return model


# ============================================================================
# TRAINING PIPELINE WITH SEED AVERAGING
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    # Stratified folds
    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    # Accumulate across seeds
    all_oof_lgb, all_oof_xgb, all_oof_cat = [], [], []
    all_test_lgb, all_test_xgb, all_test_cat = [], [], []

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")
        oof_lgb = np.zeros(n_train)
        oof_xgb = np.zeros(n_train)
        oof_cat = np.zeros(n_train)
        test_lgb = np.zeros(n_test)
        test_xgb = np.zeros(n_test)
        test_cat = np.zeros(n_test)

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr, X_vl = train_df.iloc[ti], train_df.iloc[vi]
            y_tr, y_vl = y[ti], y[vi]
            s = seed_idx * 100 + fold

            # LGB
            m = train_lgb(X_tr, y_tr, X_vl, y_vl, feat_cols, s)
            oof_lgb[vi] = m.predict(X_vl[feat_cols])
            test_lgb += m.predict(test_df[feat_cols]) / Config.N_FOLDS

            # XGB
            m = train_xgb(X_tr, y_tr, X_vl, y_vl, feat_cols, s)
            oof_xgb[vi] = m.predict(xgb.DMatrix(X_vl[feat_cols]))
            test_xgb += m.predict(xgb.DMatrix(test_df[feat_cols])) / Config.N_FOLDS

            # CatBoost
            m = train_cat(X_tr, y_tr, X_vl, y_vl, feat_cols, s)
            oof_cat[vi] = m.predict(X_vl[feat_cols])
            test_cat += m.predict(test_df[feat_cols]) / Config.N_FOLDS

        # Report this seed
        oof_ens = Config.W_LGB*oof_lgb + Config.W_XGB*oof_xgb + Config.W_CAT*oof_cat
        mae = mean_absolute_error(y, oof_ens)
        print(f"    LGB={mean_absolute_error(y, oof_lgb):.2f}  "
              f"XGB={mean_absolute_error(y, oof_xgb):.2f}  "
              f"CAT={mean_absolute_error(y, oof_cat):.2f}  "
              f"ENS={mae:.2f}")

        all_oof_lgb.append(oof_lgb)
        all_oof_xgb.append(oof_xgb)
        all_oof_cat.append(oof_cat)
        all_test_lgb.append(test_lgb)
        all_test_xgb.append(test_xgb)
        all_test_cat.append(test_cat)

    # Average across seeds
    avg_oof_lgb = np.mean(all_oof_lgb, axis=0)
    avg_oof_xgb = np.mean(all_oof_xgb, axis=0)
    avg_oof_cat = np.mean(all_oof_cat, axis=0)
    avg_test_lgb = np.mean(all_test_lgb, axis=0)
    avg_test_xgb = np.mean(all_test_xgb, axis=0)
    avg_test_cat = np.mean(all_test_cat, axis=0)

    # Final ensemble
    oof_final = Config.W_LGB*avg_oof_lgb + Config.W_XGB*avg_oof_xgb + \
                Config.W_CAT*avg_oof_cat
    test_final = Config.W_LGB*avg_test_lgb + Config.W_XGB*avg_test_xgb + \
                 Config.W_CAT*avg_test_cat
    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:      {mean_absolute_error(y, avg_oof_lgb):.2f}")
    print(f"  XGB:      {mean_absolute_error(y, avg_oof_xgb):.2f}")
    print(f"  CatBoost: {mean_absolute_error(y, avg_oof_cat):.2f}")
    print(f"  Ensemble: {cv_mae:.2f} "
          f"({Config.W_LGB}/{Config.W_XGB}/{Config.W_CAT})")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v6 (Variance Reduction)")
    print(f"  {Config.N_FOLDS}-fold CV × {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 3} models total")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")
    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")

    # Load data
    print("\n[1/4] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    # PDF extraction
    print("\n[2/4] Extracting PDF features...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    # Build features (v3 feature set)
    print("\n[3/4] Building features (v3 set)...")
    train_feat = build_features(train, pdf_df, patients)
    test_feat = build_features(test, pdf_df, patients)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    # Train
    print(f"\n[4/4] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    # Save
    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v6 (Variance Reduction)
  7-fold CV × 3 seeds = 63 models total
Base dir: .
Receipts: .\receipts_pdf

[1/4] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/4] Extracting PDF features...
  PDF backend: PyMuPDF
  1000/4000 (found=234, missing=766)
  2000/4000 (found=1234, missing=766)
  3000/4000 (found=2234, missing=766)
  4000/4000 (found=2319, missing=1681)
  Final: 2319 extracted, 1681 not found

[3/4] Building features (v3 set)...
  Features (20): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'chronic_x_sqrt_cost', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region', 'has_pdf', 'n_line_items', 'n_unique_codes', 'cost_hhi', 'n_procedure_codes', 'has_critical_care', 'has_observation', 'max_em_level', 'pct_cost_procedure']

[4/4] Training ensemble...

  --- Seed 1/3 ---
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[322]	val

# Iteration 7
- 478.6786

In [11]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v7 (Maximum Generalization)
================================================================================
Diagnosis from v3-v6 (stuck at Test=472-473):

FINDING 1: chronic_x_sqrt HURTS (+3.5 MAE). Ablation proves removal helps.
FINDING 2: LGB/XGB drag down CatBoost. Pure CatBoost is always best.
FINDING 3: PDF features cause half the CV-test gap.
           Without PDF: CV=470, gap~5. With PDF: CV=457, gap=15.
           8 PDF features add ~13 to CV but only ~3 to test → overfitting.
FINDING 4: 7-fold + 3-seed is the best variance reduction.

v7 RECIPE:
  ✗ DROP chronic_x_sqrt (proven overfit)
  ✗ DROP LGB/XGB (drag down ensemble)
  ✗ REDUCE PDF to top 3 features (n_line_items, cost_hhi, max_em_level)
  ✓ CatBoost-only with 7-fold × 3-seed averaging
  ✓ Keep patients.csv (age, sex, insurance, zip)
  ✓ ADD only charlson_max from admissions (1 feature, 100% coverage)
  ✓ Depth=4 (heavier regularization)
  
  ~14 total features (down from 21 in v3)
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (proven pipeline)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, _, total = m.groups()
            items.append({'code': code, 'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def extract_receipt_features(pdf_path):
    """Extract ONLY the top 3 non-overfit PDF features."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}
    # Feature 1: Complexity proxy
    f['n_line_items'] = len(items)

    # Feature 2: Cost concentration (unique info: single big charge vs many small)
    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    # Feature 3: E&M severity level (unique clinical acuity indicator)
    em_levels = []
    for it in items:
        code = it['code']
        if code.startswith('9928') and len(code) == 5:
            em_levels.append(int(code[-1]))
    f['max_em_level'] = max(em_levels) if em_levels else 0

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# SURGICAL ADMISSIONS: charlson_max ONLY
# ============================================================================
def load_charlson(base_dir):
    """Load ONLY charlson_max per patient from admissions data."""
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None
    adm = pd.concat(dfs, ignore_index=True)
    result = adm.groupby('patient_id')['charlson_band'].max().reset_index(
        name='charlson_max')
    print(f"  Charlson: {len(result)} patients (100% coverage)")
    return result


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, charlson_df=None):
    feat = df.copy()

    # Core features (NO chronic_x_sqrt — proven to hurt)
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # Demographics
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

    # Charlson (single cross-challenge feature)
    if charlson_df is not None:
        feat = feat.merge(charlson_df, on='patient_id', how='left')
        feat['charlson_max'] = feat['charlson_max'].fillna(-1)

    # PDF features (TOP 3 ONLY — reduced from 8)
    if pdf_df is not None:
        feat['has_pdf'] = feat['patient_id'].isin(pdf_df['patient_id']).astype(int)
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        for c in [col for col in pdf_df.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)
    else:
        feat['has_pdf'] = 0

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'cost_bin', 'strat_col'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# TRAINING: PURE CATBOOST + SEED AVERAGING
# ============================================================================
def run_catboost_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    all_oof = []
    all_test = []

    for seed_idx in range(Config.N_SEEDS):
        oof = np.zeros(n_train)
        test_pred = np.zeros(n_test)

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]

            model = CatBoostRegressor(
                loss_function='MAE', eval_metric='MAE',
                iterations=5000, learning_rate=0.03,
                depth=4, l2_leaf_reg=5, min_data_in_leaf=30,
                random_seed=seed_idx * 100 + fold,
                verbose=0, early_stopping_rounds=100,
            )
            model.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof[vi] = model.predict(X_vl)
            test_pred += model.predict(test_df[feat_cols]) / Config.N_FOLDS

        mae = mean_absolute_error(y, oof)
        all_oof.append(oof)
        all_test.append(test_pred)
        print(f"  Seed {seed_idx + 1}: OOF MAE = {mae:.2f}")

    # Average across seeds
    oof_avg = np.mean(all_oof, axis=0)
    test_avg = np.mean(all_test, axis=0)
    cv_mae = mean_absolute_error(y, oof_avg)

    print(f"\n  Seed-averaged ({Config.N_SEEDS} seeds): OOF MAE = {cv_mae:.2f}")

    # Per-fold diagnostics
    for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
        fold_mae = mean_absolute_error(y[vi], oof_avg[vi])
        print(f"    Fold {fold + 1}: {fold_mae:.2f}")

    test_avg = np.clip(test_avg, 0, None)
    return test_avg, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v7 (Maximum Generalization)")
    print(f"  CatBoost-only, {Config.N_FOLDS}-fold × {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")

    # Load data
    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    # Charlson only
    print("\n[2/5] Loading charlson from admissions...")
    charlson_df = load_charlson(bd)

    # PDF (reduced)
    print("\n[3/5] Extracting PDF features (top 3 only)...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    # Build features
    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, charlson_df)
    test_feat = build_features(test, pdf_df, patients, charlson_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    # Train
    print(f"\n[5/5] Training CatBoost ensemble...")
    test_pred, cv_mae = run_catboost_ensemble(train_feat, test_feat, feat_cols)

    # Save
    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v7 (Maximum Generalization)
  CatBoost-only, 7-fold × 3 seeds = 21 models
Base dir: .

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading charlson from admissions...
  Charlson: 4000 patients (100% coverage)

[3/5] Extracting PDF features (top 3 only)...
  PDF backend: PyMuPDF
  1000/4000 (found=234, missing=766)
  2000/4000 (found=1234, missing=766)
  3000/4000 (found=2234, missing=766)
  4000/4000 (found=2319, missing=1681)
  Final: 2319 extracted, 1681 not found

[4/5] Building features...
  Features (15): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region', 'charlson_max', 'has_pdf', 'n_line_items', 'cost_hhi', 'max_em_level']

[5/5] Training CatBoost ensemble...
  Seed 1: OOF MAE = 464.29
  Seed 2: OOF MAE = 462.32
  Seed 3: OOF MAE = 463.20

  Seed-averaged (3 seeds): OOF MAE = 462.38
    Fold 1: 427.20
    Fo

# Iteration 8
- 470.4049

In [13]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v8
================================================================================
v3 EXACT (our best test=472) + ONE safe addition:
  - CatBoost RMSE as 4th ensemble member (loss diversity, no new features)
  - 7-fold (more training data per model)
  - 3-seed averaging (stabilize test predictions)
  - v3 EXACT features, v3 EXACT hyperparameters
  
Weights: 0.1 LGB + 0.1 XGB + 0.4 CatMAE + 0.4 CatRMSE
(Shift weight from LGB/XGB toward CatBoost models)
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    # Shift weight toward CatBoost (proven best)
    W_LGB = 0.10
    W_XGB = 0.10
    W_CAT_MAE = 0.40
    W_CAT_RMSE = 0.40
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (v3 exact)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, _, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def extract_receipt_features(pdf_path):
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}
    f['n_line_items'] = len(items)
    f['n_unique_codes'] = len(set(codes))

    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    n_proc, n_crit = 0, 0
    cost_proc = 0.0
    em_levels = []
    has_obs = 0

    for it in items:
        code, cost = it['code'], it['total']
        if code.startswith('G037'):
            has_obs = 1
        elif code in ('99291', '99292'):
            n_crit += 1
        elif code.startswith('9928') and len(code) == 5:
            em_levels.append(int(code[-1]))
        elif code[0].isdigit():
            try:
                cn = int(code)
                if 10000 <= cn <= 69999:
                    n_proc += 1
                    cost_proc += cost
            except ValueError: pass

    f['n_procedure_codes'] = n_proc
    f['has_critical_care'] = int(n_crit > 0)
    f['has_observation'] = has_obs
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['pct_cost_procedure'] = cost_proc / total if total > 0 else 0

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# FEATURE ENGINEERING (v3 EXACT — proven at test=472)
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None):
    feat = df.copy()

    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']

    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

    if pdf_df is not None:
        feat['has_pdf'] = feat['patient_id'].isin(pdf_df['patient_id']).astype(int)
        pdf_cols = ['patient_id', 'n_line_items', 'n_unique_codes', 'cost_hhi',
                    'n_procedure_codes', 'has_critical_care', 'has_observation',
                    'max_em_level', 'pct_cost_procedure']
        pdf_keep = pdf_df[[c for c in pdf_cols if c in pdf_df.columns]]
        feat = feat.merge(pdf_keep, on='patient_id', how='left')
        for c in [col for col in pdf_keep.columns if col != 'patient_id']:
            feat[c] = feat[c].fillna(-1)
    else:
        feat['has_pdf'] = 0

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32', 'bool', 'uint8')]


# ============================================================================
# v3 EXACT MODEL CONFIGS
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 20, 'max_depth': 5,
    'min_child_samples': 30, 'feature_fraction': 0.7,
    'bagging_fraction': 0.7, 'bagging_freq': 5,
    'lambda_l1': 1.0, 'lambda_l2': 1.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 30,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 1.0, 'reg_lambda': 1.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}


# ============================================================================
# TRAINING
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    all_oof = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}
    all_test = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")

        oof = {k: np.zeros(n_train) for k in all_oof}
        tst = {k: np.zeros(n_test) for k in all_oof}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]
            X_te = test_df[feat_cols]
            s = seed_idx * 100 + fold

            # LGB (v3 exact)
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB (v3 exact)
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 3000, evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100, verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE (v3 exact)
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE (NEW — loss diversity)
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

        # Report this seed
        ens = (Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] +
               Config.W_CAT_MAE * oof['cat_mae'] + Config.W_CAT_RMSE * oof['cat_rmse'])
        print(f"    LGB={mean_absolute_error(y, oof['lgb']):.2f}  "
              f"XGB={mean_absolute_error(y, oof['xgb']):.2f}  "
              f"CAT_MAE={mean_absolute_error(y, oof['cat_mae']):.2f}  "
              f"CAT_RMSE={mean_absolute_error(y, oof['cat_rmse']):.2f}  "
              f"ENS={mean_absolute_error(y, ens):.2f}")

        for k in all_oof:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    # Average across seeds
    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    # Final ensemble
    oof_final = (Config.W_LGB * avg_oof['lgb'] +
                 Config.W_XGB * avg_oof['xgb'] +
                 Config.W_CAT_MAE * avg_oof['cat_mae'] +
                 Config.W_CAT_RMSE * avg_oof['cat_rmse'])
    test_final = (Config.W_LGB * avg_test['lgb'] +
                  Config.W_XGB * avg_test['xgb'] +
                  Config.W_CAT_MAE * avg_test['cat_mae'] +
                  Config.W_CAT_RMSE * avg_test['cat_rmse'])

    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:       {mean_absolute_error(y, avg_oof['lgb']):.2f}")
    print(f"  XGB:       {mean_absolute_error(y, avg_oof['xgb']):.2f}")
    print(f"  CAT MAE:   {mean_absolute_error(y, avg_oof['cat_mae']):.2f}")
    print(f"  CAT RMSE:  {mean_absolute_error(y, avg_oof['cat_rmse']):.2f}")
    print(f"  Ensemble:  {cv_mae:.2f} "
          f"({Config.W_LGB}/{Config.W_XGB}/{Config.W_CAT_MAE}/{Config.W_CAT_RMSE})")

    # Also report v3 weights for comparison
    v3_oof = (0.2 * avg_oof['lgb'] + 0.2 * avg_oof['xgb'] +
              0.6 * avg_oof['cat_mae'])
    print(f"  v3 weights (0.2/0.2/0.6 no RMSE): {mean_absolute_error(y, v3_oof):.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v8 (v3 exact + CatBoost RMSE)")
    print(f"  4 models × {Config.N_FOLDS}-fold × {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 4} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")
    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")

    print("\n[1/4] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/4] Extracting PDF features...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    print("\n[3/4] Building features (v3 exact set)...")
    train_feat = build_features(train, pdf_df, patients)
    test_feat = build_features(test, pdf_df, patients)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[4/4] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v8 (v3 exact + CatBoost RMSE)
  4 models × 7-fold × 3 seeds = 84 models
Base dir: .
Receipts: .\receipts_pdf

[1/4] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/4] Extracting PDF features...
  PDF backend: PyMuPDF
  1000/4000 (found=234, missing=766)
  2000/4000 (found=1234, missing=766)
  3000/4000 (found=2234, missing=766)
  4000/4000 (found=2319, missing=1681)
  Final: 2319 extracted, 1681 not found

[3/4] Building features (v3 exact set)...
  Features (20): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'chronic_x_sqrt_cost', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region', 'has_pdf', 'n_line_items', 'n_unique_codes', 'cost_hhi', 'n_procedure_codes', 'has_critical_care', 'has_observation', 'max_em_level', 'pct_cost_procedure']

[4/4] Training ensemble...

  --- Seed 1/3 ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration

# Iteration 9
- 456.3885 (MAE)

In [18]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v9 (Domain Expert + Complete PDFs)
================================================================================
Base: v8 architecture (Test=470.4, our proven best)
Key change: PDFs are now 100% complete → no more -1 fills, no has_pdf flag

DOMAIN-INFORMED PDF FEATURES:
  Each receipt is an itemized billing summary over 5 years.
  CPT/HCPCS codes reveal WHAT happened clinically:
  
  E&M Levels (99281-99285): Visit severity
    - Level 1-2: minor → low future cost
    - Level 4-5: high severity → high future cost
    - avg_em_level predicts persistence better than max alone
  
  Critical Care (99291-99292): ICU-level acuity → very high cost
  
  High-Acuity Procedures: intubation (31500), central line (36556),
    chest tube (32551) → life-threatening events
  
  Cost Mix: % E&M vs % procedure vs % lab tells us visit complexity
  
  Cost Intensity: cost_per_em = total / #visits → how expensive per visit

FEATURE BUDGET: ~20 (same as v8, but better quality)
  Core (7): prior_visits/cost, chronic, sqrt_cost, cpv, log_visits, chronic_x_sqrt
  Demo (5): age, sex, insurance, zip_region, ins_x_chronic
  PDF (9): n_unique_codes, cost_hhi, has_critical_care, max_em_level,
           avg_em_level, n_em_codes, cost_per_em, has_high_acuity, n_categories

ARCHITECTURE: v8 exact
  4 models × 7-fold × 3 seeds = 84 models
  Weights: 0.1 LGB + 0.1 XGB + 0.4 CatMAE + 0.4 CatRMSE
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    W_LGB = 0.10
    W_XGB = 0.10
    W_CAT_MAE = 0.40
    W_CAT_RMSE = 0.40
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION — DOMAIN-INFORMED CPT ANALYSIS
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    """Parse all line items from a receipt PDF."""
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []

    # Format A: tab-separated single-line
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'total': float(total.replace(',', ''))})

    # Format B: multi-line (PyMuPDF)
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def _categorize_code(code):
    """Clinical CPT categorization for ED cost prediction."""
    # E&M levels
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    # Critical care
    if code in ('99291', '99292'):
        return 'critical_care', 0
    # Observation
    if code.startswith('G037'):
        return 'observation', 0
    # High-acuity procedures
    if code in ('31500', '36556', '32551'):
        return 'high_acuity_proc', 0
    # Labs (80000-89999)
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999:
                return 'lab', 0
        except: pass
    # Imaging/Radiology (70000-79999)
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999:
                return 'imaging', 0
        except: pass
    # General procedures (10000-69999)
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999:
                return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features_v9(pdf_path):
    """Domain-informed feature extraction from ED billing receipt."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}

    # --- Basic complexity ---
    f['n_unique_codes'] = len(set(codes))

    # Cost concentration (HHI)
    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    # --- Clinical categorization ---
    em_levels = []
    em_costs = []
    categories_seen = set()
    has_crit = 0
    has_high_acuity = 0
    has_obs = 0
    has_imaging = 0
    cost_procedure = 0.0
    cost_lab = 0.0

    for it in items:
        cat, level = _categorize_code(it['code'])
        categories_seen.add(cat)

        if cat == 'em':
            em_levels.append(level)
            em_costs.append(it['total'])
        elif cat == 'critical_care':
            has_crit = 1
        elif cat == 'observation':
            has_obs = 1
        elif cat == 'high_acuity_proc':
            has_high_acuity = 1
            cost_procedure += it['total']
        elif cat == 'procedure':
            cost_procedure += it['total']
        elif cat == 'lab':
            cost_lab += it['total']
        elif cat == 'imaging':
            has_imaging = 1

    # --- E&M features (the clinical gold) ---
    f['n_em_codes'] = len(em_levels)
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0

    # Cost intensity per ED visit
    if len(em_levels) > 0:
        f['cost_per_em'] = total / len(em_levels)
    else:
        f['cost_per_em'] = total  # no E&M = single complex episode

    # --- Acuity indicators ---
    f['has_critical_care'] = has_crit
    f['has_high_acuity'] = has_high_acuity
    f['has_observation'] = has_obs

    # --- Cost mix ---
    em_total = sum(em_costs)
    f['pct_cost_em'] = em_total / total if total > 0 else 0
    f['pct_cost_procedure'] = cost_procedure / total if total > 0 else 0

    # --- Patient complexity ---
    f['n_categories'] = len(categories_seen)

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features_v9(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None):
    feat = df.copy()

    # --- Core features (v3 proven) ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['chronic_x_sqrt_cost'] = feat['chronic_encoded'] * feat['sqrt_prior_cost']

    # --- Demographics ---
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

        # Domain interaction: insurance × chronic
        # HF + self_pay = delayed care = higher costs
        # DiabetesComp + private = better management = lower costs
        if 'insurance_encoded' in feat.columns:
            feat['ins_x_chronic'] = feat['insurance_encoded'] * feat['chronic_encoded']

    # --- PDF features (domain-informed, v9) ---
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        # With complete PDFs: use median fill for rare missing (parsing failures)
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())
    
    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# v8 EXACT MODEL CONFIGS
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 20, 'max_depth': 5,
    'min_child_samples': 30, 'feature_fraction': 0.7,
    'bagging_fraction': 0.7, 'bagging_freq': 5,
    'lambda_l1': 1.0, 'lambda_l2': 1.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 30,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 1.0, 'reg_lambda': 1.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}


# ============================================================================
# TRAINING (v8 exact pipeline)
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    all_oof = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}
    all_test = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")

        oof = {k: np.zeros(n_train) for k in all_oof}
        tst = {k: np.zeros(n_test) for k in all_oof}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]
            X_te = test_df[feat_cols]
            s = seed_idx * 100 + fold

            # LGB
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 3000, evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100, verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE (loss diversity)
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

        ens = (Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] +
               Config.W_CAT_MAE * oof['cat_mae'] + Config.W_CAT_RMSE * oof['cat_rmse'])
        print(f"    LGB={mean_absolute_error(y, oof['lgb']):.2f}  "
              f"XGB={mean_absolute_error(y, oof['xgb']):.2f}  "
              f"CAT_MAE={mean_absolute_error(y, oof['cat_mae']):.2f}  "
              f"CAT_RMSE={mean_absolute_error(y, oof['cat_rmse']):.2f}  "
              f"ENS={mean_absolute_error(y, ens):.2f}")

        for k in all_oof:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    # Average across seeds
    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    oof_final = (Config.W_LGB * avg_oof['lgb'] +
                 Config.W_XGB * avg_oof['xgb'] +
                 Config.W_CAT_MAE * avg_oof['cat_mae'] +
                 Config.W_CAT_RMSE * avg_oof['cat_rmse'])
    test_final = (Config.W_LGB * avg_test['lgb'] +
                  Config.W_XGB * avg_test['xgb'] +
                  Config.W_CAT_MAE * avg_test['cat_mae'] +
                  Config.W_CAT_RMSE * avg_test['cat_rmse'])

    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:       {mean_absolute_error(y, avg_oof['lgb']):.2f}")
    print(f"  XGB:       {mean_absolute_error(y, avg_oof['xgb']):.2f}")
    print(f"  CAT MAE:   {mean_absolute_error(y, avg_oof['cat_mae']):.2f}")
    print(f"  CAT RMSE:  {mean_absolute_error(y, avg_oof['cat_rmse']):.2f}")
    print(f"  Ensemble:  {cv_mae:.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v9 (Domain Expert + Complete PDFs)")
    print(f"  4 models × {Config.N_FOLDS}-fold × {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 4} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")
    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")

    print("\n[1/4] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/4] Extracting PDF features (domain-informed v9)...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)
    if pdf_df is not None:
        print(f"  PDF features: {[c for c in pdf_df.columns if c != 'patient_id']}")
        n_total = len(all_pids)
        n_found = len(pdf_df)
        print(f"  Coverage: {n_found}/{n_total} ({100*n_found/n_total:.1f}%)")

    print("\n[3/4] Building features...")
    train_feat = build_features(train, pdf_df, patients)
    test_feat = build_features(test, pdf_df, patients)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[4/4] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v9 (Domain Expert + Complete PDFs)
  4 models × 7-fold × 3 seeds = 84 models
Base dir: .
Receipts: .\receipts_pdf

[1/4] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/4] Extracting PDF features (domain-informed v9)...
  PDF backend: PyMuPDF
  1000/4000 (found=1000, missing=0)
  2000/4000 (found=2000, missing=0)
  3000/4000 (found=3000, missing=0)
  4000/4000 (found=4000, missing=0)
  Final: 4000 extracted, 0 not found
  PDF features: ['n_unique_codes', 'cost_hhi', 'n_em_codes', 'max_em_level', 'avg_em_level', 'cost_per_em', 'has_critical_care', 'has_high_acuity', 'has_observation', 'pct_cost_em', 'pct_cost_procedure', 'n_categories']
  Coverage: 4000/4000 (100.0%)

[3/4] Building features...
  Features (24): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'chronic_x_sqrt_cost', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region', 'ins_x_chronic', 'n_unique_codes', 'cost_hhi'

# Iteration 10
- 455.4061

In [20]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v10 (Optimized Domain Expert)
================================================================================
Base: v9 (Test=456.4, CV=435.1) — our proven best

CHANGES FROM v9:
  1. WEIGHT SHIFT: 0.05/0.05/0.30/0.60 (was 0.10/0.10/0.40/0.40)
     CatRMSE was 6.8 pts better than CatMAE in v9. Give it dominance.
  
  2. NEW PDF FEATURES (3 domain-informed):
     n_high_em       — Count of level 4-5 E&M visits (high severity recurrence)
     n_procedures    — Count of surgical procedure codes (complexity)
     has_intubation  — 31500 (highest acuity ED procedure)
  
  3. CHARLSON from admissions (1 feature, 100% patient coverage)
     The #1 healthcare cost predictor in clinical medicine.
  
  4. DROP chronic_x_sqrt (ablation proves it hurts CatRMSE)

TOTAL: ~26 features
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    W_LGB = 0.05
    W_XGB = 0.05
    W_CAT_MAE = 0.30
    W_CAT_RMSE = 0.60
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION — DOMAIN-INFORMED CPT ANALYSIS
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    """Parse all line items from a receipt PDF."""
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []

    # Format A: tab-separated single-line
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'total': float(total.replace(',', ''))})

    # Format B: multi-line (PyMuPDF)
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def _categorize_code(code):
    """Clinical CPT categorization for ED cost prediction."""
    # E&M levels
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    # Critical care
    if code in ('99291', '99292'):
        return 'critical_care', 0
    # Observation
    if code.startswith('G037'):
        return 'observation', 0
    # High-acuity procedures
    if code in ('31500', '36556', '32551'):
        return 'high_acuity_proc', 0
    # Labs (80000-89999)
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999:
                return 'lab', 0
        except: pass
    # Imaging/Radiology (70000-79999)
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999:
                return 'imaging', 0
        except: pass
    # General procedures (10000-69999)
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999:
                return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features_v9(pdf_path):
    """Domain-informed feature extraction from ED billing receipt."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}

    # --- Basic complexity ---
    f['n_unique_codes'] = len(set(codes))

    # Cost concentration (HHI)
    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    # --- Clinical categorization ---
    em_levels = []
    em_costs = []
    categories_seen = set()
    has_crit = 0
    has_high_acuity = 0
    has_obs = 0
    has_imaging = 0
    has_intubation = 0
    cost_procedure = 0.0
    cost_lab = 0.0
    n_procedures = 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        categories_seen.add(cat)

        if cat == 'em':
            em_levels.append(level)
            em_costs.append(it['total'])
        elif cat == 'critical_care':
            has_crit = 1
        elif cat == 'observation':
            has_obs = 1
        elif cat == 'high_acuity_proc':
            has_high_acuity = 1
            if it['code'] == '31500':
                has_intubation = 1
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'procedure':
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'lab':
            cost_lab += it['total']
        elif cat == 'imaging':
            has_imaging = 1

    # --- E&M features (the clinical gold) ---
    f['n_em_codes'] = len(em_levels)
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['n_high_em'] = sum(1 for lv in em_levels if lv >= 4)  # NEW: high-severity visit count

    # Cost intensity per ED visit
    if len(em_levels) > 0:
        f['cost_per_em'] = total / len(em_levels)
    else:
        f['cost_per_em'] = total  # no E&M = single complex episode

    # --- Acuity indicators ---
    f['has_critical_care'] = has_crit
    f['has_high_acuity'] = has_high_acuity
    f['has_observation'] = has_obs
    f['has_intubation'] = has_intubation  # NEW: highest acuity procedure

    # --- Cost mix ---
    em_total = sum(em_costs)
    f['pct_cost_em'] = em_total / total if total > 0 else 0
    f['pct_cost_procedure'] = cost_procedure / total if total > 0 else 0

    # --- Patient complexity ---
    f['n_categories'] = len(categories_seen)
    f['n_procedures'] = n_procedures  # NEW: procedure count

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features_v9(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# SURGICAL CROSS-CHALLENGE: charlson_max ONLY
# ============================================================================
def load_charlson(base_dir):
    """Load ONLY charlson_max per patient from admissions."""
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None
    adm = pd.concat(dfs, ignore_index=True)
    result = adm.groupby('patient_id')['charlson_band'].max().reset_index(
        name='charlson_max')
    print(f"  Charlson: {len(result)} patients")
    return result


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, charlson_df=None):
    feat = df.copy()

    # --- Core features (DROP chronic_x_sqrt — hurts CatRMSE) ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # --- Demographics ---
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

        # Domain interaction: insurance × chronic
        # HF + self_pay = delayed care = higher costs
        # DiabetesComp + private = better management = lower costs
        if 'insurance_encoded' in feat.columns:
            feat['ins_x_chronic'] = feat['insurance_encoded'] * feat['chronic_encoded']

    # --- Charlson comorbidity (single cross-challenge feature) ---
    if charlson_df is not None:
        feat = feat.merge(charlson_df, on='patient_id', how='left')
        feat['charlson_max'] = feat['charlson_max'].fillna(feat['charlson_max'].median())

    # --- PDF features (domain-informed) ---
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        # With complete PDFs: use median fill for rare missing (parsing failures)
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())
    
    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'charlson_band'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# v8 EXACT MODEL CONFIGS
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 20, 'max_depth': 5,
    'min_child_samples': 30, 'feature_fraction': 0.7,
    'bagging_fraction': 0.7, 'bagging_freq': 5,
    'lambda_l1': 1.0, 'lambda_l2': 1.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 30,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 1.0, 'reg_lambda': 1.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}


# ============================================================================
# TRAINING (v8 exact pipeline)
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    all_oof = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}
    all_test = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")

        oof = {k: np.zeros(n_train) for k in all_oof}
        tst = {k: np.zeros(n_test) for k in all_oof}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]
            X_te = test_df[feat_cols]
            s = seed_idx * 100 + fold

            # LGB
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 3000, evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100, verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE (loss diversity)
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

        ens = (Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] +
               Config.W_CAT_MAE * oof['cat_mae'] + Config.W_CAT_RMSE * oof['cat_rmse'])
        print(f"    LGB={mean_absolute_error(y, oof['lgb']):.2f}  "
              f"XGB={mean_absolute_error(y, oof['xgb']):.2f}  "
              f"CAT_MAE={mean_absolute_error(y, oof['cat_mae']):.2f}  "
              f"CAT_RMSE={mean_absolute_error(y, oof['cat_rmse']):.2f}  "
              f"ENS={mean_absolute_error(y, ens):.2f}")

        for k in all_oof:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    # Average across seeds
    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    oof_final = (Config.W_LGB * avg_oof['lgb'] +
                 Config.W_XGB * avg_oof['xgb'] +
                 Config.W_CAT_MAE * avg_oof['cat_mae'] +
                 Config.W_CAT_RMSE * avg_oof['cat_rmse'])
    test_final = (Config.W_LGB * avg_test['lgb'] +
                  Config.W_XGB * avg_test['xgb'] +
                  Config.W_CAT_MAE * avg_test['cat_mae'] +
                  Config.W_CAT_RMSE * avg_test['cat_rmse'])

    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:       {mean_absolute_error(y, avg_oof['lgb']):.2f}")
    print(f"  XGB:       {mean_absolute_error(y, avg_oof['xgb']):.2f}")
    print(f"  CAT MAE:   {mean_absolute_error(y, avg_oof['cat_mae']):.2f}")
    print(f"  CAT RMSE:  {mean_absolute_error(y, avg_oof['cat_rmse']):.2f}")
    print(f"  Ensemble:  {cv_mae:.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v10 (Optimized Domain Expert)")
    print(f"  4 models × {Config.N_FOLDS}-fold × {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 4} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")
    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")

    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/5] Loading charlson from admissions...")
    charlson_df = load_charlson(bd)

    print("\n[3/5] Extracting PDF features (domain-informed v10)...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)
    if pdf_df is not None:
        print(f"  PDF features: {[c for c in pdf_df.columns if c != 'patient_id']}")
        n_total = len(all_pids)
        n_found = len(pdf_df)
        print(f"  Coverage: {n_found}/{n_total} ({100*n_found/n_total:.1f}%)")

    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, charlson_df)
    test_feat = build_features(test, pdf_df, patients, charlson_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[5/5] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v10 (Optimized Domain Expert)
  4 models × 7-fold × 3 seeds = 84 models
Base dir: .
Receipts: .\receipts_pdf

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading charlson from admissions...
  Charlson: 4000 patients

[3/5] Extracting PDF features (domain-informed v10)...
  PDF backend: PyMuPDF
  1000/4000 (found=1000, missing=0)
  2000/4000 (found=2000, missing=0)
  3000/4000 (found=3000, missing=0)
  4000/4000 (found=4000, missing=0)
  Final: 4000 extracted, 0 not found
  PDF features: ['n_unique_codes', 'cost_hhi', 'n_em_codes', 'max_em_level', 'avg_em_level', 'n_high_em', 'cost_per_em', 'has_critical_care', 'has_high_acuity', 'has_observation', 'has_intubation', 'pct_cost_em', 'pct_cost_procedure', 'n_categories', 'n_procedures']
  Coverage: 4000/4000 (100.0%)

[4/5] Building features...
  Features (27): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'age', 'sex_e

# Iteration 11
- 457.7386

In [22]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v11 (Price Level + Lean Features)
================================================================================
Base: v10 architecture (Test=455.4)

CRITICAL DISCOVERY: Unit costs in receipts encode REGIONAL PRICE LEVEL.
  Patient A (ZIP 850): 99284 -> $1,135/unit (expensive market)
  Patient B (ZIP 100): 99284 -> $8.67/unit (cheap market)
  Same procedure, 130x price difference!

  Total cost = price_level x utilization x acuity
  We had utilization (n_em_codes) and acuity (avg_em_level).
  We were MISSING price level -> now extracted from E&M unit costs.

OVERFITTING FIX: 27 features -> 17 features
  v10 gap: CV=431.5, Test=455.4 (24-point gap!)
  Aggressively drop redundant/weak PDF features.
  Stronger regularization (depth=4).

FEATURES (17):
  Core (6): visits, cost, chronic, sqrt_cost, cpv, log_visits
  Demo (4): age, sex, insurance, zip_region
  Cross (1): charlson_max
  PDF (6):  avg_em_level, n_em_codes, cost_hhi, has_critical_care,
            avg_em_unit_cost [NEW], mean_unit_cost [NEW]
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    W_LGB = 0.05
    W_XGB = 0.05
    W_CAT_MAE = 0.30
    W_CAT_RMSE = 0.60
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    """Parse line items including UNIT COSTS from receipt PDF."""
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []

    # Format A: single-line
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'unit_cost': float(unit.replace(',', '')),
                          'total': float(total.replace(',', ''))})

    # Format B: multi-line (PyMuPDF)
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                unit_str = remaining[i + 3].strip().replace(',', '')
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'unit_cost': float(unit_str),
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def _categorize_code(code):
    """Clinical CPT categorization."""
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    if code in ('99291', '99292'):
        return 'critical_care', 0
    if code.startswith('G037'):
        return 'observation', 0
    if code in ('31500', '36556', '32551'):
        return 'high_acuity_proc', 0
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999: return 'lab', 0
        except: pass
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999: return 'imaging', 0
        except: pass
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999: return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features_v11(pdf_path):
    """Lean domain features + price level extraction."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    total = sum(it['total'] for it in items) or 1.0
    f = {}

    # Cost concentration (HHI)
    shares = [it['total'] / total for it in items]
    f['cost_hhi'] = sum(s ** 2 for s in shares)

    # Clinical categorization
    em_levels = []
    em_unit_costs = []
    all_unit_costs = []
    has_crit = 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        all_unit_costs.append(it['unit_cost'])

        if cat == 'em':
            em_levels.append(level)
            em_unit_costs.append(it['unit_cost'])
        elif cat == 'critical_care':
            has_crit = 1

    # E&M clinical features (LEAN)
    f['n_em_codes'] = len(em_levels)
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['has_critical_care'] = has_crit

    # *** THE KEY DISCOVERY: Price Level from Unit Costs ***
    if em_unit_costs:
        f['avg_em_unit_cost'] = np.mean(em_unit_costs)
    else:
        f['avg_em_unit_cost'] = np.mean(all_unit_costs) if all_unit_costs else 0

    f['mean_unit_cost'] = np.mean(all_unit_costs) if all_unit_costs else 0

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features_v11(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# CHARLSON FROM ADMISSIONS
# ============================================================================
def load_charlson(base_dir):
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None
    adm = pd.concat(dfs, ignore_index=True)
    result = adm.groupby('patient_id')['charlson_band'].max().reset_index(
        name='charlson_max')
    print(f"  Charlson: {len(result)} patients")
    return result


# ============================================================================
# FEATURE ENGINEERING (LEAN — 17 features)
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, charlson_df=None):
    feat = df.copy()

    # Core (6)
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # Demographics (4)
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

    # Charlson (1)
    if charlson_df is not None:
        feat = feat.merge(charlson_df, on='patient_id', how='left')
        feat['charlson_max'] = feat['charlson_max'].fillna(feat['charlson_max'].median())

    # PDF features (6 — lean + price level)
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'charlson_band'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# MODEL CONFIGS — stronger regularization to close gap
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 16, 'max_depth': 4,
    'min_child_samples': 40, 'feature_fraction': 0.7,
    'bagging_fraction': 0.7, 'bagging_freq': 5,
    'lambda_l1': 2.0, 'lambda_l2': 2.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 40,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 2.0, 'reg_lambda': 2.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 4, 'l2_leaf_reg': 8, 'min_data_in_leaf': 35,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 4, 'l2_leaf_reg': 8, 'min_data_in_leaf': 35,
    'verbose': 0, 'early_stopping_rounds': 100,
}


# ============================================================================
# TRAINING
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    all_oof = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}
    all_test = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")

        oof = {k: np.zeros(n_train) for k in all_oof}
        tst = {k: np.zeros(n_test) for k in all_oof}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]
            X_te = test_df[feat_cols]
            s = seed_idx * 100 + fold

            # LGB
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 3000, evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100, verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

        ens = (Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] +
               Config.W_CAT_MAE * oof['cat_mae'] + Config.W_CAT_RMSE * oof['cat_rmse'])
        print(f"    LGB={mean_absolute_error(y, oof['lgb']):.2f}  "
              f"XGB={mean_absolute_error(y, oof['xgb']):.2f}  "
              f"CAT_MAE={mean_absolute_error(y, oof['cat_mae']):.2f}  "
              f"CAT_RMSE={mean_absolute_error(y, oof['cat_rmse']):.2f}  "
              f"ENS={mean_absolute_error(y, ens):.2f}")

        for k in all_oof:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    oof_final = (Config.W_LGB * avg_oof['lgb'] +
                 Config.W_XGB * avg_oof['xgb'] +
                 Config.W_CAT_MAE * avg_oof['cat_mae'] +
                 Config.W_CAT_RMSE * avg_oof['cat_rmse'])
    test_final = (Config.W_LGB * avg_test['lgb'] +
                  Config.W_XGB * avg_test['xgb'] +
                  Config.W_CAT_MAE * avg_test['cat_mae'] +
                  Config.W_CAT_RMSE * avg_test['cat_rmse'])

    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:       {mean_absolute_error(y, avg_oof['lgb']):.2f}")
    print(f"  XGB:       {mean_absolute_error(y, avg_oof['xgb']):.2f}")
    print(f"  CAT MAE:   {mean_absolute_error(y, avg_oof['cat_mae']):.2f}")
    print(f"  CAT RMSE:  {mean_absolute_error(y, avg_oof['cat_rmse']):.2f}")
    print(f"  Ensemble:  {cv_mae:.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v11 (Price Level + Lean Features)")
    print(f"  4 models x {Config.N_FOLDS}-fold x {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 4} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")

    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/5] Loading charlson from admissions...")
    charlson_df = load_charlson(bd)

    print("\n[3/5] Extracting PDF features (lean v11 + price level)...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)
    if pdf_df is not None:
        print(f"  PDF features: {[c for c in pdf_df.columns if c != 'patient_id']}")
        n_total = len(all_pids)
        n_found = len(pdf_df)
        print(f"  Coverage: {n_found}/{n_total} ({100*n_found/n_total:.1f}%)")

    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, charlson_df)
    test_feat = build_features(test, pdf_df, patients, charlson_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[5/5] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v11 (Price Level + Lean Features)
  4 models x 7-fold x 3 seeds = 84 models
Base dir: .

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading charlson from admissions...
  Charlson: 4000 patients

[3/5] Extracting PDF features (lean v11 + price level)...
  PDF backend: PyMuPDF
  1000/4000 (found=1000, missing=0)
  2000/4000 (found=2000, missing=0)
  3000/4000 (found=3000, missing=0)
  4000/4000 (found=4000, missing=0)
  Final: 4000 extracted, 0 not found
  PDF features: ['cost_hhi', 'n_em_codes', 'avg_em_level', 'has_critical_care', 'avg_em_unit_cost', 'mean_unit_cost']
  Coverage: 4000/4000 (100.0%)

[4/5] Building features...
  Features (17): ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'chronic_encoded', 'sqrt_prior_cost', 'cost_per_visit', 'log_visits', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region', 'charlson_max', 'cost_hhi', 'n_em_codes', 'avg_em_level', 'has_critical_care', 'avg_em_unit_cost', 'mean_unit_cost']

[5/5]

# Iteration 12
- 460.2910

In [25]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v12 (Full Cross-Challenge)
================================================================================
DIAGNOSIS: The model has extracted ALL signal from current features.
  R^2 = 0.90 on core features. CatBoost already captures nonlinearities.
  Stacking, per-condition models, log transforms → all NEUTRAL.
  The ceiling is FEATURES, not the model.

  To reach 400 MAE we need ~20% of residual variance explained.
  That requires ORTHOGONAL information = cross-challenge data.

WHAT'S NEW (vs v10 which was our best test at 455.4):
  1. FULL ADMISSIONS FEATURES (not just charlson_max!)
     - n_admissions:     hospitalizations (different from ED visits!)
     - total_los:        total inpatient days (severity)
     - mean_los:         average stay length
     - pct_readmitted:   readmission rate from readmit_30d (complexity!)
     - max_ed_visits_6m: RECENT ED utilization (recency signal)
     - pct_emergent:     emergency admission proportion
     - n_distinct_dx:    diagnosis diversity
     → These are 100% coverage AND orthogonal to ED features
  
  2. STAYS FEATURES (if available)
     - has_icu_stay:     ICU admission indicator
     - n_unit_types:     number of distinct unit types
     - max_stay_days:    longest single stay
     → Captures inpatient severity not in admissions
  
  3. ZIP3 TARGET ENCODING (within CV folds to avoid leakage)
     - zip3_cost_mean:   average cost per zip3 area
     → 100+ granular regions vs 9 coarse zip_region values
     → Directly captures regional cost variation

  4. v10's PROVEN PDF features kept intact

Architecture: 4-model ensemble (LGB/XGB/CatMAE/CatRMSE), 7-fold, 3-seed
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    W_LGB = 0.05
    W_XGB = 0.05
    W_CAT_MAE = 0.30
    W_CAT_RMSE = 0.60
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (v10 proven — domain-informed features)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'unit_cost': float(unit.replace(',', '')),
                          'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                unit_str = remaining[i + 3].strip().replace(',', '')
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'unit_cost': float(unit_str),
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def _categorize_code(code):
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    if code in ('99291', '99292'):
        return 'critical_care', 0
    if code.startswith('G037'):
        return 'observation', 0
    if code in ('31500', '36556', '32551'):
        return 'high_acuity_proc', 0
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999: return 'lab', 0
        except: pass
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999: return 'imaging', 0
        except: pass
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999: return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features_v10(pdf_path):
    """v10's proven domain-informed PDF features."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    total = sum(it['total'] for it in items) or 1.0
    f = {}

    # Cost concentration
    shares = [it['total'] / total for it in items]
    f['cost_hhi'] = sum(s ** 2 for s in shares)

    # Clinical categorization
    em_levels = []
    cats = set()
    n_crit = 0
    n_high_acuity = 0
    n_obs = 0
    n_intub = 0
    n_procedures = 0
    cost_em = 0
    cost_proc = 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        cats.add(cat)
        if cat == 'em':
            em_levels.append(level)
            cost_em += it['total']
        elif cat == 'critical_care':
            n_crit += 1
        elif cat == 'observation':
            n_obs += 1
        elif cat == 'high_acuity_proc':
            n_high_acuity += 1
            if it['code'] == '31500':
                n_intub += 1
        elif cat == 'procedure':
            n_procedures += 1
            cost_proc += it['total']

    f['n_unique_codes'] = len(set(it['code'] for it in items))
    f['n_em_codes'] = len(em_levels)
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['n_high_em'] = sum(1 for l in em_levels if l >= 4)
    f['cost_per_em'] = total / max(len(em_levels), 1)
    f['has_critical_care'] = int(n_crit > 0)
    f['has_high_acuity'] = int(n_high_acuity > 0)
    f['has_observation'] = int(n_obs > 0)
    f['has_intubation'] = int(n_intub > 0)
    f['pct_cost_em'] = cost_em / total if total > 0 else 0
    f['pct_cost_procedure'] = cost_proc / total if total > 0 else 0
    f['n_categories'] = len(cats)
    f['n_procedures'] = n_procedures

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features_v10(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# FULL ADMISSIONS FEATURES (the key change in v12!)
# ============================================================================
def load_admissions_full(base_dir):
    """Extract RICH admissions features — not just charlson_max."""
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None

    adm = pd.concat(dfs, ignore_index=True)
    print(f"  Admissions: {len(adm)} rows, {adm['patient_id'].nunique()} patients")
    print(f"  Columns: {list(adm.columns)}")

    # Build rich per-patient features
    agg = adm.groupby('patient_id').agg(
        n_admissions=('patient_id', 'count'),
        charlson_max=('charlson_band', 'max'),
        charlson_mean=('charlson_band', 'mean'),
    ).reset_index()

    # LOS features (if available)
    if 'los_days' in adm.columns:
        los = adm.groupby('patient_id')['los_days'].agg(['sum', 'mean', 'max'])
        los.columns = ['total_los', 'mean_los', 'max_los']
        agg = agg.merge(los.reset_index(), on='patient_id', how='left')
        print(f"  LOS features: total_los, mean_los, max_los")

    # ED visits in 6 months (recency signal!)
    if 'ed_visits_6m' in adm.columns:
        ed6m = adm.groupby('patient_id')['ed_visits_6m'].agg(['max', 'mean'])
        ed6m.columns = ['max_ed_visits_6m', 'mean_ed_visits_6m']
        agg = agg.merge(ed6m.reset_index(), on='patient_id', how='left')
        print(f"  ED recency: max_ed_visits_6m, mean_ed_visits_6m")

    # Emergency admissions
    if 'acuity_emergent' in adm.columns:
        emerg = adm.groupby('patient_id')['acuity_emergent'].mean().reset_index(
            name='pct_emergent')
        agg = agg.merge(emerg, on='patient_id', how='left')
        print(f"  Acuity: pct_emergent")

    # Readmission rate (from Challenge 1 target — DIRECT complexity measure!)
    if 'readmit_30d' in adm.columns:
        has_label = adm.dropna(subset=['readmit_30d'])
        if len(has_label) > 0:
            readmit = has_label.groupby('patient_id')['readmit_30d'].agg(
                ['mean', 'sum', 'count'])
            readmit.columns = ['pct_readmitted', 'n_readmissions', 'n_labeled_adm']
            agg = agg.merge(readmit.reset_index(), on='patient_id', how='left')
            print(f"  Readmission: pct_readmitted (from {len(has_label)} labeled rows)")

    # Diagnosis diversity
    if 'primary_dx' in adm.columns:
        dx_div = adm.groupby('patient_id')['primary_dx'].nunique().reset_index(
            name='n_distinct_dx')
        agg = agg.merge(dx_div, on='patient_id', how='left')
        print(f"  Diagnosis diversity: n_distinct_dx")

    feat_cols = [c for c in agg.columns if c != 'patient_id']
    print(f"  Total admissions features: {len(feat_cols)}: {feat_cols}")
    return agg


# ============================================================================
# STAYS FEATURES (if available)
# ============================================================================
def load_stays_features(base_dir):
    """Extract per-patient stays features if stays data exists."""
    dfs = []
    for fname in ['stays_train.csv', 'stays_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Stays: NOT FOUND")
        return None

    stays = pd.concat(dfs, ignore_index=True)
    print(f"  Stays: {len(stays)} rows, {stays['patient_id'].nunique()} patients")
    print(f"  Columns: {list(stays.columns)}")

    agg = stays.groupby('patient_id').agg(
        n_stays=('patient_id', 'count'),
    ).reset_index()

    # Unit type diversity (ICU vs general ward etc)
    if 'unit_type' in stays.columns:
        unit_div = stays.groupby('patient_id')['unit_type'].nunique().reset_index(
            name='n_unit_types')
        agg = agg.merge(unit_div, on='patient_id', how='left')

        # ICU indicator
        icu_keywords = ['icu', 'intensive', 'critical']
        stays['is_icu'] = stays['unit_type'].str.lower().apply(
            lambda x: int(any(k in str(x) for k in icu_keywords)))
        icu_flag = stays.groupby('patient_id')['is_icu'].max().reset_index(
            name='has_icu_stay')
        agg = agg.merge(icu_flag, on='patient_id', how='left')

    # Discharge readiness (Challenge 3 target!)
    if 'discharge_ready_day11' in stays.columns:
        has_label = stays.dropna(subset=['discharge_ready_day11'])
        if len(has_label) > 0:
            disc = has_label.groupby('patient_id')['discharge_ready_day11'].mean()
            disc = disc.reset_index(name='pct_discharge_ready')
            agg = agg.merge(disc, on='patient_id', how='left')
            print(f"  Discharge readiness: from {len(has_label)} labeled rows")

    feat_cols = [c for c in agg.columns if c != 'patient_id']
    print(f"  Stays features: {len(feat_cols)}: {feat_cols}")
    return agg


# ============================================================================
# ZIP3 TARGET ENCODING (within CV folds to prevent leakage)
# ============================================================================
def target_encode_zip3(train_df, test_df, patients_df, y_train, fold_indices=None):
    """
    Target-encode zip3 using global mean for test, fold-aware for train.
    This captures regional cost variation at 100+ granularity levels.
    """
    if patients_df is None or 'zip3' not in patients_df.columns:
        return None, None

    # Merge zip3
    train_zip = train_df[['patient_id']].merge(
        patients_df[['patient_id', 'zip3']], on='patient_id', how='left')
    test_zip = test_df[['patient_id']].merge(
        patients_df[['patient_id', 'zip3']], on='patient_id', how='left')

    # Global encoding for test (smoothed)
    global_mean = y_train.mean()
    zip_stats = pd.DataFrame({'zip3': train_zip['zip3'], 'target': y_train})
    zip_means = zip_stats.groupby('zip3')['target'].agg(['mean', 'count'])

    # Smoothing: blend zip mean with global mean based on sample size
    smoothing = 20  # regularization strength
    zip_means['smoothed'] = (zip_means['count'] * zip_means['mean'] +
                             smoothing * global_mean) / (zip_means['count'] + smoothing)

    test_encoded = test_zip['zip3'].map(zip_means['smoothed']).fillna(global_mean)

    n_zips = train_zip['zip3'].nunique()
    print(f"  ZIP3 target encoding: {n_zips} unique zip codes, smoothing={smoothing}")

    return train_zip['zip3'], test_encoded


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, adm_df=None, stays_df=None):
    feat = df.copy()

    # Core (6)
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # Demographics (5)
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        p['zip3_numeric'] = pd.to_numeric(p['zip3'], errors='coerce')
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 
               'zip_region', 'zip3_numeric']],
            on='patient_id', how='left')
        # Insurance x chronic interaction
        feat['ins_x_chronic'] = feat.get('insurance_encoded', 0) * feat.get('chronic_encoded', 0)

    # Full admissions (NEW in v12!)
    if adm_df is not None:
        feat = feat.merge(adm_df, on='patient_id', how='left')
        # Fill NaN with median for admissions features
        for c in [col for col in adm_df.columns if col != 'patient_id']:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    # Stays features (NEW in v12!)
    if stays_df is not None:
        feat = feat.merge(stays_df, on='patient_id', how='left')
        for c in [col for col in stays_df.columns if col != 'patient_id']:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    # PDF features (v10 proven set)
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'charlson_band',
               'readmit_30d', 'discharge_ready_day11',
               'admission_id', 'primary_dx', 'stay_id', 'unit_type',
               'admission_reason', 'discharge_weekday', 'note',
               'acuity_emergent', 'los_days', 'ed_visits_6m',
               'is_icu'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# MODEL CONFIGS (v10 proven hyperparameters)
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 20, 'max_depth': 5,
    'min_child_samples': 25, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5,
    'lambda_l1': 1.0, 'lambda_l2': 1.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 25,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 1.0, 'reg_lambda': 1.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}


# ============================================================================
# TRAINING (v10 architecture with ZIP3 target encoding)
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols, patients_df=None):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    # ZIP3 target encoding setup
    train_zip3, test_zip3_encoded = target_encode_zip3(
        train_df, test_df, patients_df, y)
    use_zip3 = train_zip3 is not None

    all_oof = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}
    all_test = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}

    # Determine feature columns including zip3 encoding
    feat_cols_full = feat_cols.copy()
    if use_zip3 and 'zip3_target_enc' not in feat_cols_full:
        feat_cols_full.append('zip3_target_enc')

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")

        oof = {k: np.zeros(n_train) for k in all_oof}
        tst = {k: np.zeros(n_test) for k in all_oof}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            # Fold-aware ZIP3 target encoding (prevent leakage!)
            if use_zip3:
                global_mean = y[ti].mean()
                zip_stats = pd.DataFrame({
                    'zip3': train_zip3.iloc[ti],
                    'target': y[ti]
                })
                zip_means = zip_stats.groupby('zip3')['target'].agg(['mean', 'count'])
                smoothing = 20
                zip_means['smoothed'] = (zip_means['count'] * zip_means['mean'] +
                                         smoothing * global_mean) / \
                                        (zip_means['count'] + smoothing)

                # Encode train validation fold
                train_df_copy = train_df.copy()
                train_df_copy['zip3_target_enc'] = train_zip3.map(
                    zip_means['smoothed']).fillna(global_mean)

                # Encode test
                test_df_copy = test_df.copy()
                test_df_copy['zip3_target_enc'] = test_zip3_encoded
            else:
                train_df_copy = train_df
                test_df_copy = test_df

            X_tr = train_df_copy[feat_cols_full].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df_copy[feat_cols_full].iloc[vi]
            y_vl = y[vi]
            X_te = test_df_copy[feat_cols_full]
            s = seed_idx * 100 + fold

            # LGB
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 3000, evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100, verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

        ens = (Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] +
               Config.W_CAT_MAE * oof['cat_mae'] + Config.W_CAT_RMSE * oof['cat_rmse'])
        print(f"    LGB={mean_absolute_error(y, oof['lgb']):.2f}  "
              f"XGB={mean_absolute_error(y, oof['xgb']):.2f}  "
              f"CAT_MAE={mean_absolute_error(y, oof['cat_mae']):.2f}  "
              f"CAT_RMSE={mean_absolute_error(y, oof['cat_rmse']):.2f}  "
              f"ENS={mean_absolute_error(y, ens):.2f}")

        for k in all_oof:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    oof_final = (Config.W_LGB * avg_oof['lgb'] +
                 Config.W_XGB * avg_oof['xgb'] +
                 Config.W_CAT_MAE * avg_oof['cat_mae'] +
                 Config.W_CAT_RMSE * avg_oof['cat_rmse'])
    test_final = (Config.W_LGB * avg_test['lgb'] +
                  Config.W_XGB * avg_test['xgb'] +
                  Config.W_CAT_MAE * avg_test['cat_mae'] +
                  Config.W_CAT_RMSE * avg_test['cat_rmse'])

    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:       {mean_absolute_error(y, avg_oof['lgb']):.2f}")
    print(f"  XGB:       {mean_absolute_error(y, avg_oof['xgb']):.2f}")
    print(f"  CAT MAE:   {mean_absolute_error(y, avg_oof['cat_mae']):.2f}")
    print(f"  CAT RMSE:  {mean_absolute_error(y, avg_oof['cat_rmse']):.2f}")
    print(f"  Ensemble:  {cv_mae:.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v12 (Full Cross-Challenge Integration)")
    print(f"  4 models x {Config.N_FOLDS}-fold x {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 4} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")

    print("\n[1/6] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/6] Loading FULL admissions features...")
    adm_df = load_admissions_full(bd)

    print("\n[3/6] Loading stays features...")
    stays_df = load_stays_features(bd)

    print("\n[4/6] Extracting PDF features (v10 domain set)...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)
    if pdf_df is not None:
        print(f"  PDF features: {[c for c in pdf_df.columns if c != 'patient_id']}")
        n_total = len(all_pids)
        n_found = len(pdf_df)
        print(f"  Coverage: {n_found}/{n_total} ({100*n_found/n_total:.1f}%)")

    print("\n[5/6] Building features...")
    train_feat = build_features(train, pdf_df, patients, adm_df, stays_df)
    test_feat = build_features(test, pdf_df, patients, adm_df, stays_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[6/6] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols, patients)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v12 (Full Cross-Challenge Integration)
  4 models x 7-fold x 3 seeds = 84 models
Base dir: .

[1/6] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/6] Loading FULL admissions features...
  Admissions: 10000 rows, 4000 patients
  Columns: ['admission_id', 'patient_id', 'primary_dx', 'los_days', 'acuity_emergent', 'charlson_band', 'ed_visits_6m', 'discharge_weekday', 'readmit_30d']
  LOS features: total_los, mean_los, max_los
  ED recency: max_ed_visits_6m, mean_ed_visits_6m
  Acuity: pct_emergent
  Readmission: pct_readmitted (from 5000 labeled rows)
  Diagnosis diversity: n_distinct_dx
  Total admissions features: 13: ['n_admissions', 'charlson_max', 'charlson_mean', 'total_los', 'mean_los', 'max_los', 'max_ed_visits_6m', 'mean_ed_visits_6m', 'pct_emergent', 'pct_readmitted', 'n_readmissions', 'n_labeled_adm', 'n_distinct_dx']

[3/6] Loading stays features...
  Stays: 2000 rows, 2000 patients
  Columns: ['stay_id', 'patient_id', 'unit_type', 'admis

# Iteration 13
- 460.9661

In [27]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v13 (Disciplined Cross-Challenge)
================================================================================
KEY LESSON FROM v10-v12:
  v10 (27 feat): CV=432, Test=455 ← BEST TEST
  v12 (44 feat): CV=435, Test=460 ← worse! 17 extra features = +5 test MAE

  WHY v12 FAILED: Added features with <100% coverage
    - pct_readmitted (50% admissions labeled), pct_discharge_ready (25%)
    - n_labeled_adm (encodes Challenge 1 train/test split!)
    - stays features (50% patient coverage)
    → Train/test distribution mismatch → gap widens → test degrades

v13 STRATEGY: v10 base + ONLY 100%-coverage additions
  1. v10's proven 27 features (core + demographics + charlson + PDF)
  2. NEW: 4 admissions features with 100% coverage:
     - n_admissions (hospitalization count — different from ED visits!)
     - total_los (total inpatient days — severity axis)
     - max_ed_visits_6m (RECENT ED use — trajectory signal)
     - pct_emergent (emergency admission rate — acuity)
  3. NEW: Target-encoded primary_dx (diagnosis-specific cost signal)
     - Each patient's admissions have diagnosis codes
     - Target encode: "patients with dx X tend to cost Y"
     - Within CV folds to prevent leakage, smoothed for small groups
  4. Stronger regularization: depth=4, l2=8, min_leaf=35
  5. NO stays features (50% coverage)
  6. NO readmit features (50% labeled)
  
  Total: ~33 features. Minimal, disciplined expansion.
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    W_LGB = 0.05
    W_XGB = 0.05
    W_CAT_MAE = 0.30
    W_CAT_RMSE = 0.60
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION (v10 proven pipeline — domain-informed features)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'unit_cost': float(unit.replace(',', '')),
                          'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                unit_str = remaining[i + 3].strip().replace(',', '')
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'unit_cost': float(unit_str),
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def _categorize_code(code):
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    if code in ('99291', '99292'):
        return 'critical_care', 0
    if code.startswith('G037'):
        return 'observation', 0
    if code in ('31500', '36556', '32551'):
        return 'high_acuity_proc', 0
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999: return 'lab', 0
        except: pass
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999: return 'imaging', 0
        except: pass
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999: return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features(pdf_path):
    """v10's proven domain-informed PDF features."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    total = sum(it['total'] for it in items) or 1.0
    f = {}

    shares = [it['total'] / total for it in items]
    f['cost_hhi'] = sum(s ** 2 for s in shares)

    em_levels = []
    cats = set()
    n_crit = 0
    n_high_acuity = 0
    n_obs = 0
    n_intub = 0
    n_procedures = 0
    cost_em = 0
    cost_proc = 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        cats.add(cat)
        if cat == 'em':
            em_levels.append(level)
            cost_em += it['total']
        elif cat == 'critical_care':
            n_crit += 1
        elif cat == 'observation':
            n_obs += 1
        elif cat == 'high_acuity_proc':
            n_high_acuity += 1
            if it['code'] == '31500':
                n_intub += 1
        elif cat == 'procedure':
            n_procedures += 1
            cost_proc += it['total']

    f['n_unique_codes'] = len(set(it['code'] for it in items))
    f['n_em_codes'] = len(em_levels)
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['n_high_em'] = sum(1 for l in em_levels if l >= 4)
    f['cost_per_em'] = total / max(len(em_levels), 1)
    f['has_critical_care'] = int(n_crit > 0)
    f['has_high_acuity'] = int(n_high_acuity > 0)
    f['has_observation'] = int(n_obs > 0)
    f['has_intubation'] = int(n_intub > 0)
    f['pct_cost_em'] = cost_em / total if total > 0 else 0
    f['pct_cost_procedure'] = cost_proc / total if total > 0 else 0
    f['n_categories'] = len(cats)
    f['n_procedures'] = n_procedures

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# ADMISSIONS: SELECTIVE 100%-COVERAGE FEATURES ONLY
# ============================================================================
def load_admissions_selective(base_dir):
    """
    Load ONLY features with 100% patient coverage from admissions.
    
    CRITICAL: Do NOT include:
    - readmit_30d features (50% labeled → distribution mismatch)
    - n_labeled_adm (encodes Challenge 1 train/test split)
    - Any feature derived from partial labels
    """
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None, None

    adm = pd.concat(dfs, ignore_index=True)
    n_patients = adm['patient_id'].nunique()
    print(f"  Admissions: {len(adm)} rows, {n_patients} patients")

    # ---- AGGREGATED FEATURES (100% coverage) ----
    agg = adm.groupby('patient_id').agg(
        n_admissions=('patient_id', 'count'),
        charlson_max=('charlson_band', 'max'),
        charlson_mean=('charlson_band', 'mean'),
    ).reset_index()

    # LOS (100% coverage — every admission has los_days)
    if 'los_days' in adm.columns:
        los = adm.groupby('patient_id')['los_days'].agg(['sum', 'mean', 'max'])
        los.columns = ['total_los', 'mean_los', 'max_los']
        agg = agg.merge(los.reset_index(), on='patient_id', how='left')

    # Recent ED use (100% coverage)
    if 'ed_visits_6m' in adm.columns:
        ed6m = adm.groupby('patient_id')['ed_visits_6m'].agg(['max', 'mean'])
        ed6m.columns = ['max_ed_visits_6m', 'mean_ed_visits_6m']
        agg = agg.merge(ed6m.reset_index(), on='patient_id', how='left')

    # Emergency admission rate (100% coverage)
    if 'acuity_emergent' in adm.columns:
        emerg = adm.groupby('patient_id')['acuity_emergent'].mean().reset_index(
            name='pct_emergent')
        agg = agg.merge(emerg, on='patient_id', how='left')

    # Diagnosis diversity (100% coverage)
    if 'primary_dx' in adm.columns:
        dx_div = adm.groupby('patient_id')['primary_dx'].nunique().reset_index(
            name='n_distinct_dx')
        agg = agg.merge(dx_div, on='patient_id', how='left')

    feat_cols = [c for c in agg.columns if c != 'patient_id']
    print(f"  Selected features ({len(feat_cols)}): {feat_cols}")
    print(f"  ALL 100% coverage ✓")

    # ---- PRIMARY_DX for target encoding ----
    dx_data = None
    if 'primary_dx' in adm.columns:
        # Get most frequent dx per patient (mode)
        dx_mode = adm.groupby('patient_id')['primary_dx'].agg(
            lambda x: x.value_counts().index[0]).reset_index(name='most_common_dx')
        dx_data = dx_mode
        n_unique_dx = adm['primary_dx'].nunique()
        print(f"  Primary DX: {n_unique_dx} unique codes → will target-encode")

    return agg, dx_data


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, adm_df=None):
    feat = df.copy()

    # Core (6 features)
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # Demographics (4 features)
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

    # Admissions — selective 100% coverage (up to 11 features)
    if adm_df is not None:
        feat = feat.merge(adm_df, on='patient_id', how='left')
        for c in [col for col in adm_df.columns if col != 'patient_id']:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    # PDF (v10's 14 domain features)
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'most_common_dx',
               'cost_bin', 'strat_col'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# MODEL CONFIGS — stronger regularization than v10
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 16, 'max_depth': 4,
    'min_child_samples': 35, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5,
    'lambda_l1': 2.0, 'lambda_l2': 2.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 35,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 2.0, 'reg_lambda': 2.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 5000, 'learning_rate': 0.03,
    'depth': 4, 'l2_leaf_reg': 8, 'min_data_in_leaf': 35,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 5000, 'learning_rate': 0.03,
    'depth': 4, 'l2_leaf_reg': 8, 'min_data_in_leaf': 35,
    'verbose': 0, 'early_stopping_rounds': 100,
}


# ============================================================================
# TARGET ENCODING for primary_dx (within CV folds)
# ============================================================================
def target_encode_dx(train_df, test_df, dx_data, y_train, kf, strat):
    """
    Target-encode primary_dx: "patients with this diagnosis tend to cost X"
    
    Uses fold-aware encoding to prevent leakage:
    - For each training fold, compute dx→cost mapping from other folds
    - For test, use global mapping from all training data
    - Smoothed with global mean to handle rare diagnoses
    """
    if dx_data is None:
        return None, None

    # Merge dx to train/test
    train_dx = train_df[['patient_id']].merge(dx_data, on='patient_id', how='left')
    test_dx = test_df[['patient_id']].merge(dx_data, on='patient_id', how='left')

    # Fill missing dx
    train_dx['most_common_dx'] = train_dx['most_common_dx'].fillna('UNKNOWN')
    test_dx['most_common_dx'] = test_dx['most_common_dx'].fillna('UNKNOWN')

    n_unique = train_dx['most_common_dx'].nunique()
    print(f"  DX target encoding: {n_unique} unique diagnosis codes")

    # Global encoding for test (smoothed)
    global_mean = y_train.mean()
    smoothing = 30  # stronger smoothing for potentially sparse categories

    dx_stats = pd.DataFrame({
        'dx': train_dx['most_common_dx'],
        'target': y_train
    })
    dx_means = dx_stats.groupby('dx')['target'].agg(['mean', 'count'])
    dx_means['smoothed'] = (dx_means['count'] * dx_means['mean'] +
                            smoothing * global_mean) / (dx_means['count'] + smoothing)

    test_encoded = test_dx['most_common_dx'].map(dx_means['smoothed']).fillna(global_mean)

    return train_dx['most_common_dx'], test_encoded


# ============================================================================
# TRAINING
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols, dx_data=None):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    # DX target encoding setup
    train_dx, test_dx_encoded = target_encode_dx(
        train_df, test_df, dx_data, y, kf, strat)
    use_dx = train_dx is not None

    feat_cols_full = feat_cols.copy()
    if use_dx and 'dx_target_enc' not in feat_cols_full:
        feat_cols_full.append('dx_target_enc')

    all_oof = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}
    all_test = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")

        oof = {k: np.zeros(n_train) for k in all_oof}
        tst = {k: np.zeros(n_test) for k in all_oof}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            # Fold-aware DX target encoding
            if use_dx:
                global_mean = y[ti].mean()
                smoothing = 30
                dx_df = pd.DataFrame({
                    'dx': train_dx.iloc[ti],
                    'target': y[ti]
                })
                dx_means = dx_df.groupby('dx')['target'].agg(['mean', 'count'])
                dx_means['smoothed'] = (dx_means['count'] * dx_means['mean'] +
                                        smoothing * global_mean) / \
                                       (dx_means['count'] + smoothing)

                train_df_copy = train_df.copy()
                train_df_copy['dx_target_enc'] = train_dx.map(
                    dx_means['smoothed']).fillna(global_mean)

                test_df_copy = test_df.copy()
                test_df_copy['dx_target_enc'] = test_dx_encoded
            else:
                train_df_copy = train_df
                test_df_copy = test_df

            X_tr = train_df_copy[feat_cols_full].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df_copy[feat_cols_full].iloc[vi]
            y_vl = y[vi]
            X_te = test_df_copy[feat_cols_full]
            s = seed_idx * 100 + fold

            # LGB
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 5000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 5000, evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100, verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

        ens = (Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] +
               Config.W_CAT_MAE * oof['cat_mae'] + Config.W_CAT_RMSE * oof['cat_rmse'])
        print(f"    LGB={mean_absolute_error(y, oof['lgb']):.2f}  "
              f"XGB={mean_absolute_error(y, oof['xgb']):.2f}  "
              f"CAT_MAE={mean_absolute_error(y, oof['cat_mae']):.2f}  "
              f"CAT_RMSE={mean_absolute_error(y, oof['cat_rmse']):.2f}  "
              f"ENS={mean_absolute_error(y, ens):.2f}")

        for k in all_oof:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    oof_final = (Config.W_LGB * avg_oof['lgb'] +
                 Config.W_XGB * avg_oof['xgb'] +
                 Config.W_CAT_MAE * avg_oof['cat_mae'] +
                 Config.W_CAT_RMSE * avg_oof['cat_rmse'])
    test_final = (Config.W_LGB * avg_test['lgb'] +
                  Config.W_XGB * avg_test['xgb'] +
                  Config.W_CAT_MAE * avg_test['cat_mae'] +
                  Config.W_CAT_RMSE * avg_test['cat_rmse'])

    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:       {mean_absolute_error(y, avg_oof['lgb']):.2f}")
    print(f"  XGB:       {mean_absolute_error(y, avg_oof['xgb']):.2f}")
    print(f"  CAT MAE:   {mean_absolute_error(y, avg_oof['cat_mae']):.2f}")
    print(f"  CAT RMSE:  {mean_absolute_error(y, avg_oof['cat_rmse']):.2f}")
    print(f"  Ensemble:  {cv_mae:.2f}")

    # Feature importance from last CatBoost RMSE
    print(f"\n  Top 15 feature importances (CatBoost RMSE):")
    imp = dict(zip(feat_cols_full, m.get_feature_importance()))
    for i, (feat, val) in enumerate(sorted(imp.items(), key=lambda x: -x[1])[:15]):
        print(f"    {i+1:2d}. {feat:25s}: {val:.1f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v13 (Disciplined Cross-Challenge)")
    print(f"  4 models x {Config.N_FOLDS}-fold x {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 4} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")

    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/5] Loading admissions (100% coverage only)...")
    adm_df, dx_data = load_admissions_selective(bd)

    print("\n[3/5] Extracting PDF features (v10 domain set)...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        print(f"  PDF features ({len(pdf_cols)}): {pdf_cols}")

    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, adm_df)
    test_feat = build_features(test, pdf_df, patients, adm_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[5/5] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols, dx_data)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v13 (Disciplined Cross-Challenge)
  4 models x 7-fold x 3 seeds = 84 models
Base dir: .

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading admissions (100% coverage only)...
  Admissions: 10000 rows, 4000 patients
  Selected features (10): ['n_admissions', 'charlson_max', 'charlson_mean', 'total_los', 'mean_los', 'max_los', 'max_ed_visits_6m', 'mean_ed_visits_6m', 'pct_emergent', 'n_distinct_dx']
  ALL 100% coverage ✓
  Primary DX: 3 unique codes → will target-encode

[3/5] Extracting PDF features (v10 domain set)...
  PDF backend: PyMuPDF
  1000/4000 (found=1000, missing=0)
  2000/4000 (found=2000, missing=0)
  3000/4000 (found=3000, missing=0)
  4000/4000 (found=4000, missing=0)
  Final: 4000 extracted, 0 not found
  PDF features (15): ['cost_hhi', 'n_unique_codes', 'n_em_codes', 'max_em_level', 'avg_em_level', 'n_high_em', 'cost_per_em', 'has_critical_care', 'has_high_acuity', 'has_observation', 'has_intubation', 'pct_cost_em', 'p

# Iteration 14
- 460.1811

In [29]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v14 (Cross-Challenge Intelligence)
================================================================================
Base: v10 (Test=455.4, CV=431.6) — proven best test score

KEY INSIGHT FROM v11-v13:
  Raw admissions features have <0.04 residual correlation → no new signal.
  Adding them widens CV-test gap → worse test score every time.
  primary_dx has only 3 values (= primary_chronic) → target encoding redundant.

v14 APPROACH: Use cross-challenge data DIFFERENTLY
  Instead of raw columns, use PREDICTED LABELS from other challenges:
  
  1. PREDICTED READMISSION PROBABILITY (cross-challenge model):
     - Train simple readmission model on admissions_train (has readmit_30d)
     - Predict P(readmit) for ALL admissions (train + test)
     - Aggregate per patient: mean/max predicted probability
     - ONE feature capturing nonlinear combination of all admissions info
     - 100% patient coverage, no distribution mismatch
  
  2. DISCHARGE NOTES TEXT FEATURES (if discharge_notes.json available):
     - Note length, word count, keyword extraction
     - Clinical severity indicators from unstructured text
     - Aggregate per patient across admissions
  
  3. SELECTIVE ADMISSIONS FEATURES (only highest-residual-signal):
     - discharge_weekday (highest residual corr at 0.04)
     - emergent_rate (2nd highest residual corr)
     - charlson_max (proven in v10)
  
  4. v10's exact PDF features + ensemble architecture preserved

TOTAL: v10's ~27 features + 2-5 cross-challenge features = ~29-32 features
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import warnings

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)


class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 3
    W_LGB = 0.05
    W_XGB = 0.05
    W_CAT_MAE = 0.30
    W_CAT_RMSE = 0.60
    RECEIPTS_DIR = None
    BASE_DIR = '.'
    OUTPUT_PATH = "submission.csv"


# ============================================================================
# PDF EXTRACTION — DOMAIN-INFORMED CPT ANALYSIS
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path); text = doc[0].get_text(); doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    """Parse all line items from a receipt PDF."""
    if not text: return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []

    # Format A: tab-separated single-line
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'total': float(total.replace(',', ''))})

    # Format B: multi-line (PyMuPDF)
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1; break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'total': float(total_str)})
                    except ValueError: pass
                i += 5
    return items


def _categorize_code(code):
    """Clinical CPT categorization for ED cost prediction."""
    # E&M levels
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    # Critical care
    if code in ('99291', '99292'):
        return 'critical_care', 0
    # Observation
    if code.startswith('G037'):
        return 'observation', 0
    # High-acuity procedures
    if code in ('31500', '36556', '32551'):
        return 'high_acuity_proc', 0
    # Labs (80000-89999)
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999:
                return 'lab', 0
        except: pass
    # Imaging/Radiology (70000-79999)
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999:
                return 'imaging', 0
        except: pass
    # General procedures (10000-69999)
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999:
                return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features_v9(pdf_path):
    """Domain-informed feature extraction from ED billing receipt."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items: return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0

    f = {}

    # --- Basic complexity ---
    f['n_unique_codes'] = len(set(codes))

    # Cost concentration (HHI)
    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    # --- Clinical categorization ---
    em_levels = []
    em_costs = []
    categories_seen = set()
    has_crit = 0
    has_high_acuity = 0
    has_obs = 0
    has_imaging = 0
    has_intubation = 0
    cost_procedure = 0.0
    cost_lab = 0.0
    n_procedures = 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        categories_seen.add(cat)

        if cat == 'em':
            em_levels.append(level)
            em_costs.append(it['total'])
        elif cat == 'critical_care':
            has_crit = 1
        elif cat == 'observation':
            has_obs = 1
        elif cat == 'high_acuity_proc':
            has_high_acuity = 1
            if it['code'] == '31500':
                has_intubation = 1
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'procedure':
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'lab':
            cost_lab += it['total']
        elif cat == 'imaging':
            has_imaging = 1

    # --- E&M features (the clinical gold) ---
    f['n_em_codes'] = len(em_levels)
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['n_high_em'] = sum(1 for lv in em_levels if lv >= 4)  # NEW: high-severity visit count

    # Cost intensity per ED visit
    if len(em_levels) > 0:
        f['cost_per_em'] = total / len(em_levels)
    else:
        f['cost_per_em'] = total  # no E&M = single complex episode

    # --- Acuity indicators ---
    f['has_critical_care'] = has_crit
    f['has_high_acuity'] = has_high_acuity
    f['has_observation'] = has_obs
    f['has_intubation'] = has_intubation  # NEW: highest acuity procedure

    # --- Cost mix ---
    em_total = sum(em_costs)
    f['pct_cost_em'] = em_total / total if total > 0 else 0
    f['pct_cost_procedure'] = cost_procedure / total if total > 0 else 0

    # --- Patient complexity ---
    f['n_categories'] = len(categories_seen)
    f['n_procedures'] = n_procedures  # NEW: procedure count

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features_v9(path)
            if feat: all_feats[pid] = feat; found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ============================================================================
# CROSS-CHALLENGE INTELLIGENCE
# ============================================================================
def load_cross_challenge_features(base_dir):
    """
    Extract cross-challenge features from admissions + discharge notes.
    
    Key insight: raw admissions columns have <0.04 residual correlation.
    Instead, we train a READMISSION MODEL and use its predicted probability.
    This captures nonlinear combinations that individual features miss.
    """
    dfs = []
    adm_train = None
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            if 'readmit_30d' in df.columns:
                adm_train = df.copy()
            dfs.append(df)
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None
    
    adm = pd.concat(dfs, ignore_index=True)
    n_patients = adm['patient_id'].nunique()
    print(f"  Admissions: {len(adm)} rows, {n_patients} patients")

    # ---- FEATURE 1: Charlson max (proven in v10) ----
    result = adm.groupby('patient_id')['charlson_band'].max().reset_index(
        name='charlson_max')
    
    # ---- FEATURE 2: Selective admissions (highest residual correlation) ----
    # discharge_weekday mean: highest residual corr (0.04)
    dw = adm.groupby('patient_id')['discharge_weekday'].mean().reset_index(
        name='discharge_wd_mean')
    result = result.merge(dw, on='patient_id', how='left')
    
    # emergent rate: 2nd highest residual corr (0.03)
    if 'acuity_emergent' in adm.columns:
        em = adm.groupby('patient_id')['acuity_emergent'].mean().reset_index(
            name='emergent_rate')
        result = result.merge(em, on='patient_id', how='left')
    
    # ---- FEATURE 3: Predicted readmission probability ----
    if adm_train is not None and 'readmit_30d' in adm_train.columns:
        print("  Training readmission model (cross-challenge)...")
        readmit_feat_cols = ['los_days', 'acuity_emergent', 'charlson_band',
                             'ed_visits_6m', 'discharge_weekday']
        # Only use columns that exist
        readmit_feat_cols = [c for c in readmit_feat_cols if c in adm_train.columns]
        
        has_label = adm_train.dropna(subset=['readmit_30d'])
        X_readmit = has_label[readmit_feat_cols].values
        y_readmit = has_label['readmit_30d'].values
        
        if len(X_readmit) > 100:
            from catboost import CatBoostClassifier
            readmit_model = CatBoostClassifier(
                iterations=500, learning_rate=0.05, depth=3,
                l2_leaf_reg=10, min_data_in_leaf=50,
                verbose=0, random_seed=42,
                auto_class_weights='Balanced'
            )
            readmit_model.fit(X_readmit, y_readmit)
            
            # Predict for ALL admissions
            X_all = adm[readmit_feat_cols].fillna(0).values
            readmit_probs = readmit_model.predict_proba(X_all)[:, 1]
            adm['readmit_prob'] = readmit_probs
            
            # Aggregate per patient
            rp = adm.groupby('patient_id')['readmit_prob'].agg(['mean', 'max'])
            rp.columns = ['mean_readmit_prob', 'max_readmit_prob']
            result = result.merge(rp.reset_index(), on='patient_id', how='left')
            
            print(f"  Readmit model: trained on {len(X_readmit)} admissions")
            print(f"  Predicted for {len(adm)} admissions")
            print(f"  Mean P(readmit): {readmit_probs.mean():.3f}")
        else:
            print(f"  Readmit model: insufficient labeled data ({len(X_readmit)})")
    else:
        print("  Readmit model: no labeled admissions data found")
    
    # ---- FEATURE 4: Discharge notes text features ----
    notes_path = os.path.join(base_dir, 'discharge_notes.json')
    if os.path.exists(notes_path):
        try:
            import json
            with open(notes_path, 'r') as f:
                notes = json.load(f)
            print(f"  Discharge notes: {len(notes)} notes found")
            
            # Build per-admission text features
            note_feats = {}
            severity_words = {'unstable', 'critical', 'severe', 'deteriorat',
                              'complicat', 'worsen', 'emergent', 'acute',
                              'readmit', 'failure', 'decline', 'poor'}
            recovery_words = {'stable', 'improv', 'recover', 'discharg',
                              'ambulat', 'tolerat', 'progress', 'better',
                              'resolv', 'independ'}
            
            for note_obj in notes:
                aid = note_obj.get('admission_id')
                text = note_obj.get('note', '')
                if not text:
                    continue
                text_lower = text.lower()
                words = text_lower.split()
                
                note_feats[aid] = {
                    'note_len': len(text),
                    'note_words': len(words),
                    'severity_score': sum(1 for w in words
                                         for kw in severity_words
                                         if kw in w),
                    'recovery_score': sum(1 for w in words
                                         for kw in recovery_words
                                         if kw in w),
                }
            
            if note_feats:
                nf_df = pd.DataFrame.from_dict(note_feats, orient='index')
                nf_df.index.name = 'admission_id'
                nf_df = nf_df.reset_index()
                
                # Net clinical tone: recovery - severity
                nf_df['clinical_tone'] = nf_df['recovery_score'] - nf_df['severity_score']
                
                # Merge with admissions to get patient_id
                adm_with_notes = adm[['admission_id', 'patient_id']].merge(
                    nf_df, on='admission_id', how='left')
                
                # Aggregate per patient
                note_agg = adm_with_notes.groupby('patient_id').agg(
                    mean_note_len=('note_len', 'mean'),
                    mean_severity=('severity_score', 'mean'),
                    mean_recovery=('recovery_score', 'mean'),
                    mean_clinical_tone=('clinical_tone', 'mean'),
                ).reset_index()
                
                result = result.merge(note_agg, on='patient_id', how='left')
                print(f"  Note features: mean_note_len, mean_severity, "
                      f"mean_recovery, mean_clinical_tone")
        except Exception as e:
            print(f"  Discharge notes: error loading ({e})")
    else:
        print("  Discharge notes: not found")
    
    feat_cols = [c for c in result.columns if c != 'patient_id']
    print(f"  Cross-challenge features ({len(feat_cols)}): {feat_cols}")
    return result


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, cross_df=None):
    feat = df.copy()

    # --- Core features (DROP chronic_x_sqrt — hurts CatRMSE) ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = feat['prior_ed_cost_5y_usd'] / \
                              feat['prior_ed_visits_5y'].clip(1)
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # --- Demographics ---
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')

        # Domain interaction: insurance × chronic
        if 'insurance_encoded' in feat.columns:
            feat['ins_x_chronic'] = feat['insurance_encoded'] * feat['chronic_encoded']

    # --- Cross-challenge features (readmit prob, charlson, notes, etc) ---
    if cross_df is not None:
        cross_cols = [c for c in cross_df.columns if c != 'patient_id']
        feat = feat.merge(cross_df, on='patient_id', how='left')
        for c in cross_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    # --- PDF features (domain-informed) ---
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        # With complete PDFs: use median fill for rare missing (parsing failures)
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())
    
    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'charlson_band',
               'admission_id', 'readmit_30d', 'primary_dx'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# v8 EXACT MODEL CONFIGS
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 20, 'max_depth': 5,
    'min_child_samples': 30, 'feature_fraction': 0.7,
    'bagging_fraction': 0.7, 'bagging_freq': 5,
    'lambda_l1': 1.0, 'lambda_l2': 1.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 30,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 1.0, 'reg_lambda': 1.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}


# ============================================================================
# TRAINING (v8 exact pipeline)
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)

    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = tmp['primary_chronic'].astype(str) + '_' + \
                   tmp['cost_bin'].astype(str)
    strat = LabelEncoder().fit_transform(tmp['strat'])
    kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                         random_state=Config.SEED)

    all_oof = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}
    all_test = {k: [] for k in ['lgb', 'xgb', 'cat_mae', 'cat_rmse']}

    for seed_idx in range(Config.N_SEEDS):
        print(f"\n  --- Seed {seed_idx + 1}/{Config.N_SEEDS} ---")

        oof = {k: np.zeros(n_train) for k in all_oof}
        tst = {k: np.zeros(n_test) for k in all_oof}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]
            X_te = test_df[feat_cols]
            s = seed_idx * 100 + fold

            # LGB
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 3000, evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100, verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE (loss diversity)
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

        ens = (Config.W_LGB * oof['lgb'] + Config.W_XGB * oof['xgb'] +
               Config.W_CAT_MAE * oof['cat_mae'] + Config.W_CAT_RMSE * oof['cat_rmse'])
        print(f"    LGB={mean_absolute_error(y, oof['lgb']):.2f}  "
              f"XGB={mean_absolute_error(y, oof['xgb']):.2f}  "
              f"CAT_MAE={mean_absolute_error(y, oof['cat_mae']):.2f}  "
              f"CAT_RMSE={mean_absolute_error(y, oof['cat_rmse']):.2f}  "
              f"ENS={mean_absolute_error(y, ens):.2f}")

        for k in all_oof:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

    # Average across seeds
    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    oof_final = (Config.W_LGB * avg_oof['lgb'] +
                 Config.W_XGB * avg_oof['xgb'] +
                 Config.W_CAT_MAE * avg_oof['cat_mae'] +
                 Config.W_CAT_RMSE * avg_oof['cat_rmse'])
    test_final = (Config.W_LGB * avg_test['lgb'] +
                  Config.W_XGB * avg_test['xgb'] +
                  Config.W_CAT_MAE * avg_test['cat_mae'] +
                  Config.W_CAT_RMSE * avg_test['cat_rmse'])

    cv_mae = mean_absolute_error(y, oof_final)

    print(f"\n{'='*60}")
    print(f"SEED-AVERAGED OOF MAE:")
    print(f"  LGB:       {mean_absolute_error(y, avg_oof['lgb']):.2f}")
    print(f"  XGB:       {mean_absolute_error(y, avg_oof['xgb']):.2f}")
    print(f"  CAT MAE:   {mean_absolute_error(y, avg_oof['cat_mae']):.2f}")
    print(f"  CAT RMSE:  {mean_absolute_error(y, avg_oof['cat_rmse']):.2f}")
    print(f"  Ensemble:  {cv_mae:.2f}")

    test_final = np.clip(test_final, 0, None)
    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("AgentDS Challenge 2 — v14 (Cross-Challenge Intelligence)")
    print(f"  4 models × {Config.N_FOLDS}-fold × {Config.N_SEEDS} seeds "
          f"= {Config.N_FOLDS * Config.N_SEEDS * 4} models")
    print("=" * 70)

    for base in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
        if os.path.exists(os.path.join(base, 'ed_cost_train.csv')):
            Config.BASE_DIR = base
            for rc in [os.path.join(base, 'receipts_pdf')]:
                if os.path.exists(rc):
                    Config.RECEIPTS_DIR = rc
            break

    bd = Config.BASE_DIR
    print(f"Base dir: {bd}")
    print(f"Receipts: {Config.RECEIPTS_DIR or 'NOT FOUND'}")

    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/5] Loading cross-challenge features...")
    cross_df = load_cross_challenge_features(bd)

    print("\n[3/5] Extracting PDF features (domain-informed v10)...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)
    if pdf_df is not None:
        print(f"  PDF features: {[c for c in pdf_df.columns if c != 'patient_id']}")
        n_total = len(all_pids)
        n_found = len(pdf_df)
        print(f"  Coverage: {n_found}/{n_total} ({100*n_found/n_total:.1f}%)")

    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, cross_df)
    test_feat = build_features(test, pdf_df, patients, cross_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[5/5] Training ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    print(f"\n{'='*70}")
    print(f"FINAL CV MAE: {cv_mae:.2f}")
    print(f"Pred: mean={test_pred.mean():.1f}, median={np.median(test_pred):.1f}")
    print(f"{'='*70}")


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v14 (Cross-Challenge Intelligence)
  4 models × 7-fold × 3 seeds = 84 models
Base dir: .
Receipts: .\receipts_pdf

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading cross-challenge features...
  Admissions: 10000 rows, 4000 patients
  Training readmission model (cross-challenge)...
  Readmit model: trained on 5000 admissions
  Predicted for 10000 admissions
  Mean P(readmit): 0.501
  Discharge notes: 10000 notes found
  Note features: mean_note_len, mean_severity, mean_recovery, mean_clinical_tone
  Cross-challenge features (9): ['charlson_max', 'discharge_wd_mean', 'emergent_rate', 'mean_readmit_prob', 'max_readmit_prob', 'mean_note_len', 'mean_severity', 'mean_recovery', 'mean_clinical_tone']

[3/5] Extracting PDF features (domain-informed v10)...
  PDF backend: PyMuPDF
  1000/4000 (found=1000, missing=0)
  2000/4000 (found=2000, missing=0)
  3000/4000 (found=3000, missing=0)
  4000/4000 (found=4000, missing=0)
  Final: 4000 extrac

# Iteration 15
- 452.4468

In [31]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v15 (Optimal Ensemble + Enhanced Features)
================================================================================
BASE: v10 (Test=455.4, CV=431.6) — proven best

IMPROVEMENTS OVER v10:
  1. OOF-OPTIMIZED ENSEMBLE WEIGHTS via scipy (not fixed)
     — v10 used fixed 0.05/0.05/0.30/0.60; optimize on OOF predictions
  2. ADDITIONAL ADMISSIONS FEATURES (100% coverage, ablation-validated):
     — pct_emergent (corr=0.182 with target, ablation: -1.8 MAE)
     — charlson_mean (corr=0.157, adds info beyond charlson_max)
  3. ENHANCED PDF FEATURES targeting top residual CPT codes:
     — has_cvc_36556, has_cpr_92950, has_artline_36620 (from residual analysis)
     — n_high_acuity_total: composite of all high-acuity indicators
     — pct_cost_critical: fraction of total cost from critical care codes
  4. 5-MODEL ENSEMBLE (add CatBoost depth=6 RMSE for structural diversity)
  5. 5 SEEDS (was 3) for better test stability
  6. CLIPPING: clip predictions to [0, 1.5 * train_max] to avoid outlier blowup

References (brief):
  - Healthcare cost prediction: GBTs dominate tabular (Rajkomar 2018)
  - MAE objective aligns w/ evaluation but RMSE adds diversity (Friedman 2001)
  - CPT code grouping: CMS HCPCS classification for clinical buckets
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import sys
import warnings
import joblib

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
SEED = 42
np.random.seed(SEED)

class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 5
    OUTPUT_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"

    # Auto-detect environment
    if os.path.exists(r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"):
        BASE_DIR = r"D:\AgentDs\agent_ds_healthcare"
    else:
        BASE_DIR = '.'
        for d in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
            if os.path.exists(os.path.join(d, 'ed_cost_train.csv')):
                BASE_DIR = d
                OUTPUT_PATH = os.path.join(d, 'submission.csv')
                break

    RECEIPTS_DIR = os.path.join(BASE_DIR, 'receipts_pdf')
    RECEIPTS_JOBLIB = os.path.join(BASE_DIR, 'receipts_parsed.joblib')


print("=" * 70)
print("AgentDS Challenge 2 — v15 (Optimal Ensemble + Enhanced Features)")
print(f"  Base dir: {Config.BASE_DIR}")
print(f"  N_FOLDS={Config.N_FOLDS}, N_SEEDS={Config.N_SEEDS}")
print(f"  5 models x {Config.N_FOLDS} folds x {Config.N_SEEDS} seeds = "
      f"{5 * Config.N_FOLDS * Config.N_SEEDS} total model fits")
print("=" * 70)


# ============================================================================
# PDF EXTRACTION — ENHANCED DOMAIN-INFORMED CPT ANALYSIS
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path)
            text = doc[0].get_text()
            doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    """Parse all line items from a receipt PDF."""
    if not text:
        return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []

    # Format A: tab-separated single-line (pdfplumber style)
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'unit_cost': float(unit.replace(',', '')),
                          'total': float(total.replace(',', ''))})

    # Format B: multi-line (PyMuPDF style)
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1
                break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                unit_str = remaining[i + 3].strip().replace(',', '')
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'unit_cost': float(unit_str),
                                      'total': float(total_str)})
                    except ValueError:
                        pass
                i += 5
    return items


def _categorize_code(code):
    """Clinical CPT categorization for ED cost prediction."""
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    if code in ('99291', '99292'):
        return 'critical_care', 0
    if code.startswith('G037'):
        return 'observation', 0
    if code in ('31500', '36556', '32551', '36620', '92950'):
        return 'high_acuity_proc', 0
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999: return 'lab', 0
        except: pass
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999: return 'imaging', 0
        except: pass
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999: return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features(pdf_path):
    """Enhanced domain-informed feature extraction from ED billing receipt.
    Targets top-residual CPT codes from prompt residual analysis."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items:
        return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0
    code_set = set(codes)

    f = {}

    # === Basic complexity ===
    f['n_unique_codes'] = len(set(codes))

    # Cost concentration (HHI)
    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    # === Clinical categorization ===
    em_levels = []
    em_costs = []
    categories_seen = set()
    has_crit = 0
    has_high_acuity = 0
    has_obs = 0
    has_imaging = 0
    cost_crit = 0.0
    cost_procedure = 0.0
    cost_lab = 0.0
    n_procedures = 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        categories_seen.add(cat)

        if cat == 'em':
            em_levels.append(level)
            em_costs.append(it['total'])
        elif cat == 'critical_care':
            has_crit = 1
            cost_crit += it['total']
        elif cat == 'observation':
            has_obs = 1
        elif cat == 'high_acuity_proc':
            has_high_acuity = 1
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'procedure':
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'lab':
            cost_lab += it['total']
        elif cat == 'imaging':
            has_imaging = 1

    # === E&M features ===
    f['n_em_codes'] = len(em_levels)
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['n_high_em'] = sum(1 for lv in em_levels if lv >= 4)

    # Cost intensity per ED visit
    if len(em_levels) > 0:
        f['cost_per_em'] = total / len(em_levels)
    else:
        f['cost_per_em'] = total

    # === Acuity indicators ===
    f['has_critical_care'] = has_crit
    f['has_high_acuity'] = has_high_acuity
    f['has_observation'] = has_obs

    # === TOP RESIDUAL CPT CODE INDICATORS (from prompt residual analysis) ===
    f['has_intub_31500'] = int('31500' in code_set)
    f['has_cvc_36556'] = int('36556' in code_set)
    f['has_cpr_92950'] = int('92950' in code_set)
    f['has_artline_36620'] = int('36620' in code_set)
    f['has_ct_head_70450'] = int('70450' in code_set)
    f['has_99285'] = int('99285' in code_set)
    f['has_ct_abdpel_74177'] = int('74177' in code_set)
    f['has_troponin_84484'] = int('84484' in code_set)
    f['has_obs_G0378'] = int('G0378' in code_set)

    # Composite: number of high-acuity indicators present
    f['n_high_acuity_total'] = (f['has_intub_31500'] + f['has_cvc_36556'] +
                                 f['has_cpr_92950'] + f['has_artline_36620'] +
                                 f['has_critical_care'])

    # === Cost mix ===
    em_total = sum(em_costs)
    f['pct_cost_em'] = em_total / total if total > 0 else 0
    f['pct_cost_procedure'] = cost_procedure / total if total > 0 else 0
    f['pct_cost_critical'] = cost_crit / total if total > 0 else 0

    # === Patient complexity ===
    f['n_categories'] = len(categories_seen)
    f['n_procedures'] = n_procedures

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    """Extract PDF features for all patients."""
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library available!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat:
                all_feats[pid] = feat
                found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"    {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


def load_receipts_joblib(joblib_path, patient_ids):
    """Load pre-parsed receipt features from joblib."""
    if not os.path.exists(joblib_path):
        return None
    print(f"  Loading receipts_parsed.joblib...")
    data = joblib.load(joblib_path)
    if isinstance(data, pd.DataFrame):
        print(f"  Loaded DataFrame: {data.shape}")
        return data
    elif isinstance(data, dict):
        df = pd.DataFrame.from_dict(data, orient='index')
        df.index.name = 'patient_id'
        df = df.reset_index()
        print(f"  Loaded dict -> DataFrame: {df.shape}")
        return df
    else:
        print(f"  Unknown joblib format: {type(data)}")
        return None


# ============================================================================
# CROSS-CHALLENGE FEATURES — ADMISSIONS (ablation-validated)
# ============================================================================
def load_admissions_features(base_dir):
    """Load ablation-validated admissions features."""
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            if 'readmit_30d' in df.columns:
                df = df.drop(columns=['readmit_30d'])
            dfs.append(df)
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None

    adm = pd.concat(dfs, ignore_index=True)
    result = adm.groupby('patient_id').agg(
        charlson_max=('charlson_band', 'max'),
        charlson_mean=('charlson_band', 'mean'),
        pct_emergent=('acuity_emergent', 'mean'),
    ).reset_index()
    print(f"  Admissions features: {len(result)} patients, "
          f"cols={list(result.columns[1:])}")
    return result


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, adm_df=None):
    feat = df.copy()

    # --- Core features ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = (feat['prior_ed_cost_5y_usd'] /
                               feat['prior_ed_visits_5y'].clip(1))
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['log_prior_cost'] = np.log1p(feat['prior_ed_cost_5y_usd'])

    # --- Demographics ---
    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')
        if 'insurance_encoded' in feat.columns:
            feat['ins_x_chronic'] = feat['insurance_encoded'] * feat['chronic_encoded']

    # --- Admissions (ablation-validated) ---
    if adm_df is not None:
        feat = feat.merge(adm_df, on='patient_id', how='left')
        for c in adm_df.columns:
            if c != 'patient_id' and feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    # --- PDF features ---
    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'charlson_band'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# MODEL CONFIGURATIONS
# ============================================================================
LGB_PARAMS = {
    'objective': 'mae', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 20, 'max_depth': 5,
    'min_child_samples': 30, 'feature_fraction': 0.7,
    'bagging_fraction': 0.7, 'bagging_freq': 5,
    'lambda_l1': 1.0, 'lambda_l2': 1.0,
    'verbose': -1, 'n_jobs': -1,
}

XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 30,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 1.0, 'reg_lambda': 1.0,
    'nthread': -1, 'verbosity': 0,
}

CAT_MAE_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'verbose': 0, 'early_stopping_rounds': 100,
}

CAT_RMSE_D6_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.025,
    'depth': 6, 'l2_leaf_reg': 8, 'min_data_in_leaf': 30,
    'verbose': 0, 'early_stopping_rounds': 100,
}

MODEL_NAMES = ['lgb', 'xgb', 'cat_mae', 'cat_rmse', 'cat_rmse_d6']


# ============================================================================
# TRAINING WITH OOF-OPTIMIZED ENSEMBLE
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)
    y_max = y.max()

    # Stratification
    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = (tmp['primary_chronic'].astype(str) + '_' +
                    tmp['cost_bin'].astype(str))
    strat = LabelEncoder().fit_transform(tmp['strat'])

    all_oof = {k: [] for k in MODEL_NAMES}
    all_test = {k: [] for k in MODEL_NAMES}

    for seed_idx in range(Config.N_SEEDS):
        kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                             random_state=Config.SEED + seed_idx * 7)

        oof = {k: np.zeros(n_train) for k in MODEL_NAMES}
        tst = {k: np.zeros(n_test) for k in MODEL_NAMES}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            X_tr = train_df[feat_cols].iloc[ti]
            y_tr = y[ti]
            X_vl = train_df[feat_cols].iloc[vi]
            y_vl = y[vi]
            X_te = test_df[feat_cols]
            s = seed_idx * 100 + fold

            # LGB
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_vl, label=y_vl, reference=dtrain)
            params = {**LGB_PARAMS, 'seed': s}
            m = lgb.train(params, dtrain, 3000, valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100),
                                     lgb.log_evaluation(0)])
            oof['lgb'][vi] = m.predict(X_vl)
            tst['lgb'] += m.predict(X_te) / Config.N_FOLDS

            # XGB
            dm_tr = xgb.DMatrix(X_tr, label=y_tr)
            dm_vl = xgb.DMatrix(X_vl, label=y_vl)
            params = {**XGB_PARAMS, 'seed': s}
            m = xgb.train(params, dm_tr, 3000,
                          evals=[(dm_vl, 'val')],
                          early_stopping_rounds=100,
                          verbose_eval=False)
            oof['xgb'][vi] = m.predict(dm_vl)
            tst['xgb'] += m.predict(xgb.DMatrix(X_te)) / Config.N_FOLDS

            # CatBoost MAE
            m = CatBoostRegressor(**CAT_MAE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_mae'][vi] = m.predict(X_vl)
            tst['cat_mae'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE
            m = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse'][vi] = m.predict(X_vl)
            tst['cat_rmse'] += m.predict(X_te) / Config.N_FOLDS

            # CatBoost RMSE depth=6
            m = CatBoostRegressor(**CAT_RMSE_D6_PARAMS, random_seed=s)
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
            oof['cat_rmse_d6'][vi] = m.predict(X_vl)
            tst['cat_rmse_d6'] += m.predict(X_te) / Config.N_FOLDS

        for k in MODEL_NAMES:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

        seed_maes = {k: mean_absolute_error(y, oof[k]) for k in MODEL_NAMES}
        print(f"  Seed {seed_idx}: " +
              "  ".join(f"{k}={v:.1f}" for k, v in seed_maes.items()))

    # Average across seeds
    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    print(f"\n{'='*60}")
    print("SEED-AVERAGED OOF MAE:")
    for k in MODEL_NAMES:
        print(f"  {k:15s}: {mean_absolute_error(y, avg_oof[k]):.2f}")

    # === OOF-OPTIMIZED ENSEMBLE WEIGHTS ===
    def ensemble_mae(weights):
        pred = np.zeros(n_train, dtype=float)
        for i, k in enumerate(MODEL_NAMES):
            pred += weights[i] * avg_oof[k]
        return mean_absolute_error(y, pred)

    n_models = len(MODEL_NAMES)
    best_result = None
    for trial in range(30):
        if trial == 0:
            x0 = np.ones(n_models) / n_models
        else:
            x0 = np.random.dirichlet(np.ones(n_models))
        result = minimize(ensemble_mae, x0,
                          bounds=[(0, 1)] * n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                          method='SLSQP')
        if best_result is None or result.fun < best_result.fun:
            best_result = result

    opt_w = best_result.x

    print(f"\nOPTIMIZED ENSEMBLE WEIGHTS:")
    for k, w in zip(MODEL_NAMES, opt_w):
        print(f"  {k:15s}: {w:.4f}")

    oof_final = sum(w * avg_oof[k] for k, w in zip(MODEL_NAMES, opt_w))
    test_final = sum(w * avg_test[k] for k, w in zip(MODEL_NAMES, opt_w))
    cv_mae = mean_absolute_error(y, oof_final)
    print(f"\n  OPTIMIZED ENSEMBLE CV MAE: {cv_mae:.2f}")

    # Compare v10 fixed weights
    v10_oof = (0.05 * avg_oof['lgb'] + 0.05 * avg_oof['xgb'] +
               0.30 * avg_oof['cat_mae'] + 0.60 * avg_oof['cat_rmse'])
    v10_mae = mean_absolute_error(y, v10_oof)
    print(f"  v10 fixed weights CV MAE:  {v10_mae:.2f}")

    # Clip predictions
    test_final = np.clip(test_final, 0, y_max * 1.5)

    # === Feature importance ===
    try:
        m_last = CatBoostRegressor(**CAT_RMSE_PARAMS, random_seed=Config.SEED)
        m_last.fit(train_df[feat_cols], y, verbose=0)
        imp = pd.Series(m_last.get_feature_importance(), index=feat_cols)
        imp = imp.sort_values(ascending=False)
        print(f"\nTOP 15 FEATURE IMPORTANCES:")
        for fname, val in imp.head(15).items():
            print(f"  {fname:30s}: {val:.2f}")
    except Exception as e:
        print(f"  Feature importance error: {e}")

    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    bd = Config.BASE_DIR

    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/5] Loading admissions features (ablation-validated)...")
    adm_df = load_admissions_features(bd)

    print("\n[3/5] Loading receipt features...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = None

    # Try joblib first (preferred)
    pdf_df = load_receipts_joblib(Config.RECEIPTS_JOBLIB, all_pids)

    # If joblib doesn't have our enhanced features, re-extract from PDFs
    if pdf_df is not None:
        expected_cols = ['n_unique_codes', 'has_critical_care', 'has_intub_31500']
        if not all(c in pdf_df.columns for c in expected_cols):
            print("  Joblib lacks enhanced features, extracting from PDFs...")
            pdf_df = None

    if pdf_df is None:
        print("  Extracting from PDFs...")
        pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        n_total = len(all_pids)
        n_found = len(pdf_df)
        print(f"  PDF features ({len(pdf_cols)}): {pdf_cols}")
        print(f"  Coverage: {n_found}/{n_total} ({100*n_found/n_total:.1f}%)")
    else:
        print("  WARNING: No receipt features available!")

    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, adm_df)
    test_feat = build_features(test, pdf_df, patients, adm_df)

    feat_cols = get_feature_columns(train_feat)
    print(f"  Features ({len(feat_cols)}): {feat_cols}")

    for c in feat_cols:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[5/5] Training 5-model ensemble...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat, feat_cols)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    # === SANITY CHECKS ===
    print(f"\n{'='*70}")
    print("SANITY CHECKS:")
    print(f"  Submission shape:   {sub.shape}")
    print(f"  Submission columns: {list(sub.columns)}")
    print(f"  NaN predictions:    {sub['ed_cost_next3y_usd'].isna().sum()}")
    print(f"  Pred min:           {test_pred.min():.2f}")
    print(f"  Pred median:        {np.median(test_pred):.2f}")
    print(f"  Pred mean:          {test_pred.mean():.2f}")
    print(f"  Pred max:           {test_pred.max():.2f}")
    print(f"\n  FINAL CV MAE:       {cv_mae:.2f}")
    print(f"  Models:             5-model ensemble (LGB + XGB + CatMAE + CatRMSE + CatRMSE_d6)")
    print(f"  Ensemble:           OOF-optimized weights, {Config.N_SEEDS}-seed averaged")
    print(f"  Saved to:           {Config.OUTPUT_PATH}")
    print(f"\nPaste back CV MAE and these logs for next iteration.")
    print("=" * 70)


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v15 (Optimal Ensemble + Enhanced Features)
  Base dir: D:\AgentDs\agent_ds_healthcare
  N_FOLDS=7, N_SEEDS=5
  5 models x 7 folds x 5 seeds = 175 total model fits

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading admissions features (ablation-validated)...
  Admissions features: 4000 patients, cols=['charlson_max', 'charlson_mean', 'pct_emergent']

[3/5] Loading receipt features...
  Loading receipts_parsed.joblib...
  Loaded dict -> DataFrame: (3, 2)
  Joblib lacks enhanced features, extracting from PDFs...
  Extracting from PDFs...
  PDF backend: PyMuPDF
    1000/4000 (found=1000, missing=0)
    2000/4000 (found=2000, missing=0)
    3000/4000 (found=3000, missing=0)
    4000/4000 (found=4000, missing=0)
  Final: 4000 extracted, 0 not found
  PDF features (25): ['n_unique_codes', 'cost_hhi', 'n_em_codes', 'max_em_level', 'avg_em_level', 'n_high_em', 'cost_per_em', 'has_critical_care', 'has_high_acuity', 'has_observation', 'has_intu

# Iteration 16
- 452.2536

In [33]:
"""
================================================================================
AgentDS Challenge 2: ED Cost Forecasting — v16 (RSM Regularization + Feature Diversity)
================================================================================
BASE: v15 (CV=427.0, Test=452.4)

DIAGNOSIS FROM v15:
  - CV=427.0, Test=452.4 → 25.5 point gap (overfit)
  - Optimizer put 100% on CatBoost RMSE → LGB/XGB/CatMAE are dead weight
  - 40 features with 2000 samples: 19 low-importance PDF features (<1% each) = noise
  - Feature importance: 6 PDF features carry 16.2% importance, other 19 carry 4.5%

KEY CHANGES IN v16:
  1. RSM=0.8 (column subsampling per tree) — biggest single improvement
     Without PDFs: 462.20 → 461.17 (−1.0 MAE)
     With 40 PDF features, expect BIGGER gap reduction (more noise to filter)
  2. PRUNED feature set for Model B:
     Keep 6 high-importance PDF features, drop 19 noisy ones (40→20 features)
     Hypothesis: pruned model has smaller CV-test gap → better test score
  3. 3 CATBOOST RMSE VARIANTS (all different regularization):
     A: depth=5, rsm=0.8, FULL features (captures maximum signal)
     B: depth=4, rsm=0.8, PRUNED features (maximum generalization)
     C: depth=5, random_strength=2, FULL features (different reg axis)
  4. DROP LGB/XGB/CatMAE — consistently 0% in OOF optimizer
  5. 5 seeds × 7 folds = 105 total model fits (faster than v15's 175)
================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import sys
import warnings
import joblib

from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
SEED = 42
np.random.seed(SEED)

class Config:
    SEED = 42
    N_FOLDS = 7
    N_SEEDS = 5
    OUTPUT_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"

    # Auto-detect environment
    if os.path.exists(r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"):
        BASE_DIR = r"D:\AgentDs\agent_ds_healthcare"
    else:
        BASE_DIR = '.'
        for d in ['.', '/mnt/user-data/uploads', '../healthcare', 'healthcare']:
            if os.path.exists(os.path.join(d, 'ed_cost_train.csv')):
                BASE_DIR = d
                OUTPUT_PATH = os.path.join(d, 'submission.csv')
                break

    RECEIPTS_DIR = os.path.join(BASE_DIR, 'receipts_pdf')
    RECEIPTS_JOBLIB = os.path.join(BASE_DIR, 'receipts_parsed.joblib')

    # PDF features to KEEP (>= 1% importance in v15)
    PDF_KEEP_COLS = [
        'cost_per_em', 'cost_hhi', 'pct_cost_procedure',
        'n_high_acuity_total', 'pct_cost_critical', 'pct_cost_em',
    ]

print("=" * 70)
print("AgentDS Challenge 2 — v16 (RSM Regularization + Feature Diversity)")
print(f"  Base dir: {Config.BASE_DIR}")
print(f"  N_FOLDS={Config.N_FOLDS}, N_SEEDS={Config.N_SEEDS}")
print(f"  3 CatBoost variants × {Config.N_FOLDS} folds × {Config.N_SEEDS} seeds = "
      f"{3 * Config.N_FOLDS * Config.N_SEEDS} total model fits")
print("=" * 70)


# ============================================================================
# PDF EXTRACTION (same as v15)
# ============================================================================
_HAS_FITZ = False
_HAS_PDFPLUMBER = False
try:
    import fitz; _HAS_FITZ = True
except ImportError: pass
try:
    import pdfplumber; _HAS_PDFPLUMBER = True
except ImportError: pass


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        try:
            doc = fitz.open(pdf_path)
            text = doc[0].get_text()
            doc.close()
            return text
        except: pass
    if _HAS_PDFPLUMBER:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                return pdf.pages[0].extract_text()
        except: pass
    return None


def _parse_receipt(text):
    if not text:
        return []
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    items = []
    for line in lines:
        m = re.match(r'^(\w+)\s+(.+?)\s+(\d+)\s+([\d,]+\.\d+)\s+([\d,]+\.\d+)$', line)
        if m:
            code, desc, qty, unit, total = m.groups()
            items.append({'code': code, 'desc': desc.strip(),
                          'qty': int(qty),
                          'unit_cost': float(unit.replace(',', '')),
                          'total': float(total.replace(',', ''))})
    if not items:
        start = None
        for i, line in enumerate(lines):
            if 'Line Total' in line:
                start = i + 1
                break
        if start is not None:
            remaining = [l for l in lines[start:] if not l.startswith('TOTAL')]
            i = 0
            while i + 4 < len(remaining):
                code = remaining[i].strip()
                desc = remaining[i + 1].strip()
                qty_str = remaining[i + 2].strip()
                unit_str = remaining[i + 3].strip().replace(',', '')
                total_str = remaining[i + 4].strip().replace(',', '')
                if re.match(r'^\w+$', code) and qty_str.isdigit():
                    try:
                        items.append({'code': code, 'desc': desc,
                                      'qty': int(qty_str),
                                      'unit_cost': float(unit_str),
                                      'total': float(total_str)})
                    except ValueError:
                        pass
                i += 5
    return items


def _categorize_code(code):
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    if code in ('99291', '99292'):
        return 'critical_care', 0
    if code.startswith('G037'):
        return 'observation', 0
    if code in ('31500', '36556', '32551', '36620', '92950'):
        return 'high_acuity_proc', 0
    if code.startswith('8') and len(code) == 5:
        try:
            cn = int(code)
            if 80000 <= cn <= 89999: return 'lab', 0
        except: pass
    if code.startswith('7') and len(code) == 5:
        try:
            cn = int(code)
            if 70000 <= cn <= 79999: return 'imaging', 0
        except: pass
    if code[0].isdigit():
        try:
            cn = int(code)
            if 10000 <= cn <= 69999: return 'procedure', 0
        except: pass
    return 'other', 0


def extract_receipt_features(pdf_path):
    """Same as v15 — full feature extraction."""
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items:
        return None

    codes = [it['code'] for it in items]
    amounts = [it['total'] for it in items]
    total = sum(amounts) if amounts else 1.0
    code_set = set(codes)
    f = {}

    f['n_unique_codes'] = len(set(codes))
    if total > 0:
        shares = [a / total for a in amounts]
        f['cost_hhi'] = sum(s ** 2 for s in shares)
    else:
        f['cost_hhi'] = 1.0

    em_levels, em_costs = [], []
    categories_seen = set()
    has_crit, has_high_acuity, has_obs, has_imaging = 0, 0, 0, 0
    cost_crit, cost_procedure, cost_lab, n_procedures = 0.0, 0.0, 0.0, 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        categories_seen.add(cat)
        if cat == 'em':
            em_levels.append(level)
            em_costs.append(it['total'])
        elif cat == 'critical_care':
            has_crit = 1
            cost_crit += it['total']
        elif cat == 'observation':
            has_obs = 1
        elif cat == 'high_acuity_proc':
            has_high_acuity = 1
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'procedure':
            cost_procedure += it['total']
            n_procedures += 1
        elif cat == 'lab':
            cost_lab += it['total']
        elif cat == 'imaging':
            has_imaging = 1

    f['n_em_codes'] = len(em_levels)
    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['n_high_em'] = sum(1 for lv in em_levels if lv >= 4)
    f['cost_per_em'] = total / len(em_levels) if em_levels else total

    f['has_critical_care'] = has_crit
    f['has_high_acuity'] = has_high_acuity
    f['has_observation'] = has_obs

    f['has_intub_31500'] = int('31500' in code_set)
    f['has_cvc_36556'] = int('36556' in code_set)
    f['has_cpr_92950'] = int('92950' in code_set)
    f['has_artline_36620'] = int('36620' in code_set)
    f['has_ct_head_70450'] = int('70450' in code_set)
    f['has_99285'] = int('99285' in code_set)
    f['has_ct_abdpel_74177'] = int('74177' in code_set)
    f['has_troponin_84484'] = int('84484' in code_set)
    f['has_obs_G0378'] = int('G0378' in code_set)

    f['n_high_acuity_total'] = (f['has_intub_31500'] + f['has_cvc_36556'] +
                                 f['has_cpr_92950'] + f['has_artline_36620'] +
                                 f['has_critical_care'])

    em_total = sum(em_costs)
    f['pct_cost_em'] = em_total / total if total > 0 else 0
    f['pct_cost_procedure'] = cost_procedure / total if total > 0 else 0
    f['pct_cost_critical'] = cost_crit / total if total > 0 else 0

    f['n_categories'] = len(categories_seen)
    f['n_procedures'] = n_procedures

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  WARNING: No PDF library!")
        return None
    print(f"  PDF backend: {'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found, missing = 0, 0
    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feat = extract_receipt_features(path)
            if feat:
                all_feats[pid] = feat
                found += 1
        else:
            missing += 1
        if (i + 1) % 1000 == 0:
            print(f"    {i+1}/{len(patient_ids)} (found={found}, missing={missing})")
    print(f"  Final: {found} extracted, {missing} not found")
    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


def load_receipts_joblib(joblib_path, patient_ids):
    if not os.path.exists(joblib_path):
        return None
    print(f"  Loading receipts_parsed.joblib...")
    data = joblib.load(joblib_path)
    if isinstance(data, pd.DataFrame):
        print(f"  Loaded DataFrame: {data.shape}")
        return data
    elif isinstance(data, dict):
        df = pd.DataFrame.from_dict(data, orient='index')
        df.index.name = 'patient_id'
        df = df.reset_index()
        print(f"  Loaded dict -> DataFrame: {df.shape}")
        return df
    return None


# ============================================================================
# ADMISSIONS FEATURES
# ============================================================================
def load_admissions_features(base_dir):
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            if 'readmit_30d' in df.columns:
                df = df.drop(columns=['readmit_30d'])
            dfs.append(df)
    if not dfs:
        print("  Admissions: NOT FOUND")
        return None

    adm = pd.concat(dfs, ignore_index=True)
    result = adm.groupby('patient_id').agg(
        charlson_mean=('charlson_band', 'mean'),
        pct_emergent=('acuity_emergent', 'mean'),
    ).reset_index()
    print(f"  Admissions: {len(result)} patients, cols={list(result.columns[1:])}")
    return result


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def build_features(df, pdf_df=None, patients_df=None, adm_df=None):
    feat = df.copy()

    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['cost_per_visit'] = (feat['prior_ed_cost_5y_usd'] /
                               feat['prior_ed_visits_5y'].clip(1))
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])
    feat['log_prior_cost'] = np.log1p(feat['prior_ed_cost_5y_usd'])

    if patients_df is not None:
        p = patients_df.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left')
        feat['ins_x_chronic'] = feat['insurance_encoded'] * feat['chronic_encoded']

    if adm_df is not None:
        feat = feat.merge(adm_df, on='patient_id', how='left')
        for c in adm_df.columns:
            if c != 'patient_id' and feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        feat = feat.merge(pdf_df, on='patient_id', how='left')
        for c in pdf_cols:
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    return feat


def get_feature_columns(df):
    exclude = {'patient_id', 'primary_chronic', 'ed_cost_next3y_usd',
               'sex', 'insurance', 'zip3', 'charlson_band'}
    return [c for c in df.columns
            if c not in exclude
            and df[c].dtype in ('int64', 'float64', 'int32', 'float32',
                                'bool', 'uint8')]


# ============================================================================
# MODEL CONFIGURATIONS — 3 CATBOOST RMSE VARIANTS
# ============================================================================
# Model A: depth=5, rsm=0.8, FULL features
CAT_A_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'rsm': 0.8,  # KEY: column subsampling reduces overfit
    'verbose': 0, 'early_stopping_rounds': 100,
}

# Model B: depth=4, rsm=0.8, PRUNED features (max generalization)
CAT_B_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 4, 'l2_leaf_reg': 3, 'min_data_in_leaf': 20,
    'rsm': 0.8,
    'verbose': 0, 'early_stopping_rounds': 100,
}

# Model C: depth=5, random_strength=2, FULL features (different reg axis)
CAT_C_PARAMS = {
    'loss_function': 'RMSE', 'eval_metric': 'MAE',
    'iterations': 3000, 'learning_rate': 0.03,
    'depth': 5, 'l2_leaf_reg': 5, 'min_data_in_leaf': 25,
    'random_strength': 2,  # Adds randomness to tree scoring
    'verbose': 0, 'early_stopping_rounds': 100,
}

MODEL_CONFIGS = {
    'cat_A_d5_rsm08': CAT_A_PARAMS,
    'cat_B_d4_rsm08_pruned': CAT_B_PARAMS,
    'cat_C_d5_rs2': CAT_C_PARAMS,
}
MODEL_NAMES = list(MODEL_CONFIGS.keys())


# ============================================================================
# TRAINING
# ============================================================================
def run_ensemble(train_df, test_df, feat_cols_full, feat_cols_pruned):
    y = train_df['ed_cost_next3y_usd'].values
    n_train, n_test = len(train_df), len(test_df)
    y_max = y.max()

    # Which features each model uses
    model_feat_cols = {
        'cat_A_d5_rsm08': feat_cols_full,
        'cat_B_d4_rsm08_pruned': feat_cols_pruned,
        'cat_C_d5_rs2': feat_cols_full,
    }

    # Stratification
    tmp = train_df.copy()
    tmp['cost_bin'] = pd.qcut(y, q=5, labels=False)
    tmp['strat'] = (tmp['primary_chronic'].astype(str) + '_' +
                    tmp['cost_bin'].astype(str))
    strat = LabelEncoder().fit_transform(tmp['strat'])

    all_oof = {k: [] for k in MODEL_NAMES}
    all_test = {k: [] for k in MODEL_NAMES}

    for seed_idx in range(Config.N_SEEDS):
        kf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True,
                             random_state=Config.SEED + seed_idx * 7)

        oof = {k: np.zeros(n_train) for k in MODEL_NAMES}
        tst = {k: np.zeros(n_test) for k in MODEL_NAMES}

        for fold, (ti, vi) in enumerate(kf.split(tmp, strat)):
            s = seed_idx * 100 + fold

            for name in MODEL_NAMES:
                fc = model_feat_cols[name]
                X_tr = train_df[fc].iloc[ti]
                y_tr = y[ti]
                X_vl = train_df[fc].iloc[vi]
                y_vl = y[vi]
                X_te = test_df[fc]

                m = CatBoostRegressor(**MODEL_CONFIGS[name], random_seed=s)
                m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl))
                oof[name][vi] = m.predict(X_vl)
                tst[name] += m.predict(X_te) / Config.N_FOLDS

        for k in MODEL_NAMES:
            all_oof[k].append(oof[k])
            all_test[k].append(tst[k])

        seed_maes = {k: mean_absolute_error(y, oof[k]) for k in MODEL_NAMES}
        print(f"  Seed {seed_idx}: " +
              "  ".join(f"{k.split('_')[1]}_{k.split('_')[2]}={v:.1f}"
                        for k, v in seed_maes.items()))

    # Average across seeds
    avg_oof = {k: np.mean(v, axis=0) for k, v in all_oof.items()}
    avg_test = {k: np.mean(v, axis=0) for k, v in all_test.items()}

    print(f"\n{'='*60}")
    print("SEED-AVERAGED OOF MAE:")
    for k in MODEL_NAMES:
        fc = model_feat_cols[k]
        print(f"  {k:30s}: {mean_absolute_error(y, avg_oof[k]):.2f} "
              f"({len(fc)} features)")

    # === OOF-OPTIMIZED ENSEMBLE WEIGHTS ===
    def ensemble_mae(weights):
        pred = sum(weights[i] * avg_oof[k] for i, k in enumerate(MODEL_NAMES))
        return mean_absolute_error(y, pred)

    n_models = len(MODEL_NAMES)
    best_result = None
    for trial in range(50):
        if trial == 0:
            x0 = np.ones(n_models) / n_models
        else:
            x0 = np.random.dirichlet(np.ones(n_models))
        result = minimize(ensemble_mae, x0,
                          bounds=[(0, 1)] * n_models,
                          constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                          method='SLSQP')
        if best_result is None or result.fun < best_result.fun:
            best_result = result

    opt_w = best_result.x

    print(f"\nOPTIMIZED ENSEMBLE WEIGHTS:")
    for k, w in zip(MODEL_NAMES, opt_w):
        print(f"  {k:30s}: {w:.4f}")

    oof_final = sum(w * avg_oof[k] for k, w in zip(MODEL_NAMES, opt_w))
    test_final = sum(w * avg_test[k] for k, w in zip(MODEL_NAMES, opt_w))
    cv_mae = mean_absolute_error(y, oof_final)
    print(f"\n  OPTIMIZED ENSEMBLE CV MAE: {cv_mae:.2f}")

    # Report individual model MAEs for comparison
    for k in MODEL_NAMES:
        print(f"  {k}: {mean_absolute_error(y, avg_oof[k]):.2f}")

    # Clip predictions
    test_final = np.clip(test_final, 0, y_max * 1.5)

    # === Feature importance (Model A = main model) ===
    try:
        m_last = CatBoostRegressor(**CAT_A_PARAMS, random_seed=Config.SEED)
        m_last.fit(train_df[feat_cols_full], y, verbose=0)
        imp = pd.Series(m_last.get_feature_importance(), index=feat_cols_full)
        imp = imp.sort_values(ascending=False)
        print(f"\nTOP 15 FEATURE IMPORTANCES (Model A):")
        for fname, val in imp.head(15).items():
            print(f"  {fname:30s}: {val:.2f}")
    except Exception as e:
        print(f"  Feature importance error: {e}")

    return test_final, cv_mae


# ============================================================================
# MAIN
# ============================================================================
def main():
    bd = Config.BASE_DIR

    print("\n[1/5] Loading data...")
    train = pd.read_csv(os.path.join(bd, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(bd, 'ed_cost_test.csv'))
    print(f"  Train: {len(train)} | Test: {len(test)}")

    patients = None
    p_path = os.path.join(bd, 'patients.csv')
    if os.path.exists(p_path):
        patients = pd.read_csv(p_path)
        print(f"  Patients: {len(patients)}")

    print("\n[2/5] Loading admissions features...")
    adm_df = load_admissions_features(bd)

    print("\n[3/5] Loading receipt features...")
    all_pids = sorted(set(train['patient_id'].tolist() +
                          test['patient_id'].tolist()))
    pdf_df = None

    pdf_df = load_receipts_joblib(Config.RECEIPTS_JOBLIB, all_pids)
    if pdf_df is not None:
        expected_cols = ['n_unique_codes', 'has_critical_care', 'has_intub_31500']
        if not all(c in pdf_df.columns for c in expected_cols):
            print("  Joblib lacks enhanced features, extracting from PDFs...")
            pdf_df = None

    if pdf_df is None:
        print("  Extracting from PDFs...")
        pdf_df = extract_all_receipts(all_pids, Config.RECEIPTS_DIR)

    if pdf_df is not None:
        pdf_cols = [c for c in pdf_df.columns if c != 'patient_id']
        n_total = len(all_pids)
        n_found = len(pdf_df)
        print(f"  PDF features ({len(pdf_cols)}): {pdf_cols}")
        print(f"  Coverage: {n_found}/{n_total} ({100*n_found/n_total:.1f}%)")
    else:
        print("  WARNING: No receipt features available!")

    print("\n[4/5] Building features...")
    train_feat = build_features(train, pdf_df, patients, adm_df)
    test_feat = build_features(test, pdf_df, patients, adm_df)

    feat_cols_full = get_feature_columns(train_feat)
    print(f"  FULL features ({len(feat_cols_full)}): {feat_cols_full}")

    # PRUNED: base features + only high-importance PDF features
    pdf_keep = [c for c in Config.PDF_KEEP_COLS if c in train_feat.columns]
    base_non_pdf = [c for c in feat_cols_full
                    if c not in (pdf_df.columns if pdf_df is not None else [])]
    feat_cols_pruned = base_non_pdf + pdf_keep
    # Ensure all pruned cols exist
    feat_cols_pruned = [c for c in feat_cols_pruned if c in train_feat.columns]
    print(f"  PRUNED features ({len(feat_cols_pruned)}): {feat_cols_pruned}")

    # Clean features
    for c in feat_cols_full:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    print(f"\n[5/5] Training 3 CatBoost RMSE variants...")
    test_pred, cv_mae = run_ensemble(train_feat, test_feat,
                                      feat_cols_full, feat_cols_pruned)

    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_pred, 2)
    })
    sub.to_csv(Config.OUTPUT_PATH, index=False)

    # === SANITY CHECKS ===
    print(f"\n{'='*70}")
    print("SANITY CHECKS:")
    print(f"  Submission shape:   {sub.shape}")
    print(f"  Submission columns: {list(sub.columns)}")
    print(f"  NaN predictions:    {sub['ed_cost_next3y_usd'].isna().sum()}")
    print(f"  Pred min:           {test_pred.min():.2f}")
    print(f"  Pred median:        {np.median(test_pred):.2f}")
    print(f"  Pred mean:          {test_pred.mean():.2f}")
    print(f"  Pred max:           {test_pred.max():.2f}")
    print(f"\n  FINAL CV MAE:       {cv_mae:.2f}")
    print(f"  Models:             3 CatBoost RMSE variants (A=d5_rsm, B=d4_rsm_pruned, C=d5_rs2)")
    print(f"  Full features:      {len(feat_cols_full)}")
    print(f"  Pruned features:    {len(feat_cols_pruned)}")
    print(f"  Ensemble:           OOF-optimized weights, {Config.N_SEEDS}-seed averaged")
    print(f"  Saved to:           {Config.OUTPUT_PATH}")
    print(f"\nPaste back CV MAE and these logs for next iteration.")
    print("=" * 70)


if __name__ == '__main__':
    main()

AgentDS Challenge 2 — v16 (RSM Regularization + Feature Diversity)
  Base dir: D:\AgentDs\agent_ds_healthcare
  N_FOLDS=7, N_SEEDS=5
  3 CatBoost variants × 7 folds × 5 seeds = 105 total model fits

[1/5] Loading data...
  Train: 2000 | Test: 2000
  Patients: 4000

[2/5] Loading admissions features...
  Admissions: 4000 patients, cols=['charlson_mean', 'pct_emergent']

[3/5] Loading receipt features...
  Loading receipts_parsed.joblib...
  Loaded dict -> DataFrame: (3, 2)
  Joblib lacks enhanced features, extracting from PDFs...
  Extracting from PDFs...
  PDF backend: PyMuPDF
    1000/4000 (found=1000, missing=0)
    2000/4000 (found=2000, missing=0)
    3000/4000 (found=3000, missing=0)
    4000/4000 (found=4000, missing=0)
  Final: 4000 extracted, 0 not found
  PDF features (25): ['n_unique_codes', 'cost_hhi', 'n_em_codes', 'max_em_level', 'avg_em_level', 'n_high_em', 'cost_per_em', 'has_critical_care', 'has_high_acuity', 'has_observation', 'has_intub_31500', 'has_cvc_36556', 'has_c

# Iteration 17
- 

In [35]:
"""
====================================================================================================
CODE 21 | Gap-closer: fewer features, heavier regularization | goal: beat LB 449.0
====================================================================================================
Diagnosis: Code20 OOF=428.3 vs LB=449.0 → 21-point gap.
  Root cause: 44 receipt features + 49 total features on n=2000 → overfitting.

Changes from Code 20:
  1. Receipt features: 44 → 5 (n_line_items, cost_hhi, n_unique_codes,
     log1p_max_line_total, log1p_median_unit_price). All has_* flags DROPPED.
  2. Total features: 49 → ~22 (hard cap).
  3. Regularization: depth 4/3→4/3, RSM 0.8→0.65, l2_leaf_reg=0→5,
     min_data_in_leaf default→25, subsample 0.8→0.75.
  4. Shifts: computed and reported but NOT applied to submission (noise-level).
  5. Adversarial validation diagnostic added.
  6. Single ensemble: two CatBoost configs, no LGB/XGB.

Expected: OOF ≈ 435–442, LB ≈ 443–448 (gap target: ≤10).
====================================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import sys
import warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# CONFIG — edit paths here
# ---------------------------------------------------------------------------
BASE_DIR = r'D:\AgentDs\agent_ds_healthcare'
RECEIPTS_DIR = os.path.join(BASE_DIR, 'receipts_pdf')
OUTPUT_PATH = os.path.join(BASE_DIR, 'submission.csv')

N_SEEDS = 5
N_FOLDS = 7
VERBOSE = True

# ---------------------------------------------------------------------------
# PDF PARSING (minimal — PyMuPDF preferred)
# ---------------------------------------------------------------------------
try:
    import fitz  # PyMuPDF
    _HAS_FITZ = True
except ImportError:
    _HAS_FITZ = False

try:
    import pdfplumber
    _HAS_PDFPLUMBER = True
except ImportError:
    _HAS_PDFPLUMBER = False

LINE_ITEM_RE = re.compile(
    r'(\d{5})\s+'           # CPT/HCPCS code (5 digits)
    r'(.+?)\s+'             # description
    r'(\d+)\s+'             # qty
    r'([\d,]+\.\d{2})\s+'   # unit price
    r'([\d,]+\.\d{2})'      # line total
)


def _get_pdf_text(pdf_path):
    """Extract text from PDF using available backend."""
    if _HAS_FITZ:
        doc = fitz.open(pdf_path)
        text = '\n'.join(page.get_text() for page in doc)
        doc.close()
        return text
    elif _HAS_PDFPLUMBER:
        with pdfplumber.open(pdf_path) as pdf:
            return '\n'.join(p.extract_text() or '' for p in pdf.pages)
    return ''


def _parse_receipt(text):
    """Parse receipt text into line items."""
    items = []
    for m in LINE_ITEM_RE.finditer(text):
        code = m.group(1)
        qty = int(m.group(3))
        unit_price = float(m.group(4).replace(',', ''))
        line_total = float(m.group(5).replace(',', ''))
        items.append({
            'code': code,
            'qty': qty,
            'unit_price': unit_price,
            'total': line_total,
        })
    return items


def extract_receipt_features_lean(pdf_path):
    """
    LEAN receipt features — only 5 structural/aggregate features.
    No individual code flags (these overfit on n=2000).
    """
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items:
        return None

    total = sum(it['total'] for it in items) or 1.0
    unit_prices = [it['unit_price'] for it in items if it['unit_price'] > 0]
    codes = set(it['code'] for it in items)

    f = {}

    # 1. Number of line items (utilization breadth)
    f['n_line_items'] = len(items)

    # 2. Cost concentration HHI (structural)
    shares = [it['total'] / total for it in items]
    f['cost_hhi'] = sum(s ** 2 for s in shares)

    # 3. Number of unique CPT codes (complexity)
    f['n_unique_codes'] = len(codes)

    # 4. Max line total (severity signal)
    f['log1p_max_line_total'] = np.log1p(max(it['total'] for it in items))

    # 5. Median unit price (price level — key decomposition feature)
    f['log1p_median_unit_price'] = np.log1p(np.median(unit_prices)) if unit_prices else 0

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    """Extract lean receipt features for all patients."""
    if receipts_dir is None or not os.path.exists(receipts_dir):
        print(f"  [receipts] directory not found: {receipts_dir}")
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  [receipts] ERROR: no PDF library! pip install pymupdf")
        return None

    print(f"  [receipts] backend={'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found = 0

    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feats = extract_receipt_features_lean(path)
            if feats:
                all_feats[pid] = feats
                found += 1
        if (i + 1) % 1000 == 0:
            print(f"    {i+1}/{len(patient_ids)} processed (found={found})")

    print(f"  [receipts] extracted: {found}/{len(patient_ids)}")

    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ---------------------------------------------------------------------------
# ADMISSIONS AGGREGATES (lean — 4 features)
# ---------------------------------------------------------------------------
def build_admissions_features(base_dir):
    """Aggregate admissions data per patient. Keep only robust aggregates."""
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  [admissions] NOT FOUND")
        return None

    adm = pd.concat(dfs, ignore_index=True)

    # Drop the label column if present (avoid leakage)
    adm = adm.drop(columns=['readmit_30d'], errors='ignore')

    agg = adm.groupby('patient_id').agg(
        charlson_max=('charlson_band', 'max'),
        pct_emergent=('acuity_emergent', 'mean'),
        n_admissions=('admission_id', 'count'),
        max_ed_visits_6m=('ed_visits_6m', 'max'),
    ).reset_index()

    print(f"  [admissions] {len(agg)} patients, 4 features")
    return agg


# ---------------------------------------------------------------------------
# FEATURE ENGINEERING (~22 features total)
# ---------------------------------------------------------------------------
def build_features(df, patients, adm_feats, receipt_feats):
    """
    Target: ~22 features. Every feature must earn its keep.
    """
    feat = df.copy()

    # --- Core (4) ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)
    feat['prior_ed_visits_5y'] = feat['prior_ed_visits_5y']  # keep raw
    feat['prior_ed_cost_5y_usd'] = feat['prior_ed_cost_5y_usd']  # keep raw

    # --- Cost transforms (4) ---
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['log_prior_cost'] = np.log1p(feat['prior_ed_cost_5y_usd'])
    feat['prior_cost_cap20k'] = feat['prior_ed_cost_5y_usd'].clip(upper=20000)
    feat['cost_per_visit'] = (feat['prior_ed_cost_5y_usd']
                              / feat['prior_ed_visits_5y'].clip(1))

    # --- Visit transforms (1) ---
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # --- Baseline prediction (1) — strong standalone signal ---
    # Group mean by chronic condition (robust, no leakage if done on full data)
    # This is a "prior" that the model can adjust
    chronic_baseline = {0: 3300, 1: 4100, 2: 4500}  # approximate from EDA
    feat['baseline_chronic'] = feat['chronic_encoded'].map(chronic_baseline)

    # --- Demographics (4) ---
    if patients is not None:
        p = patients.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left'
        )

    # --- Admissions (4) ---
    if adm_feats is not None:
        feat = feat.merge(adm_feats, on='patient_id', how='left')
        for c in ['charlson_max', 'pct_emergent', 'n_admissions', 'max_ed_visits_6m']:
            feat[c] = feat[c].fillna(feat[c].median())

    # --- Receipt features (5, lean) ---
    if receipt_feats is not None:
        feat = feat.merge(receipt_feats, on='patient_id', how='left')
        receipt_cols = ['n_line_items', 'cost_hhi', 'n_unique_codes',
                        'log1p_max_line_total', 'log1p_median_unit_price']
        for c in receipt_cols:
            if c in feat.columns:
                feat[c] = feat[c].fillna(feat[c].median())

    return feat


# Feature columns — explicit list for reproducibility
FEATURE_COLS = [
    # Core (3)
    'chronic_encoded', 'prior_ed_visits_5y', 'prior_ed_cost_5y_usd',
    # Cost transforms (4)
    'sqrt_prior_cost', 'log_prior_cost', 'prior_cost_cap20k', 'cost_per_visit',
    # Visit transform (1)
    'log_visits',
    # Baseline (1)
    'baseline_chronic',
    # Demographics (4)
    'age', 'sex_encoded', 'insurance_encoded', 'zip_region',
    # Admissions (4)
    'charlson_max', 'pct_emergent', 'n_admissions', 'max_ed_visits_6m',
    # Receipt (5)
    'n_line_items', 'cost_hhi', 'n_unique_codes',
    'log1p_max_line_total', 'log1p_median_unit_price',
]


# ---------------------------------------------------------------------------
# ADVERSARIAL VALIDATION (diagnostic only)
# ---------------------------------------------------------------------------
def adversarial_validation(train_feat, test_feat, feat_cols):
    """
    Train classifier to distinguish train vs test.
    Features that rank high have distribution shift → risky.
    """
    from catboost import CatBoostClassifier
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import roc_auc_score

    X_train = train_feat[feat_cols].copy()
    X_test = test_feat[feat_cols].copy()

    X = pd.concat([X_train, X_test], ignore_index=True)
    y = np.array([0] * len(X_train) + [1] * len(X_test))

    clf = CatBoostClassifier(
        iterations=200, depth=4, learning_rate=0.05,
        verbose=0, random_seed=42
    )
    clf.fit(X, y)

    importances = clf.get_feature_importance()
    imp_df = pd.DataFrame({
        'feature': feat_cols,
        'adv_importance': importances
    }).sort_values('adv_importance', ascending=False)

    print("\n[adversarial validation] Feature importance (train vs test distinguishability):")
    print("  (High = distribution shift between train/test → overfit risk)")
    for _, row in imp_df.head(10).iterrows():
        flag = " ⚠️" if row['adv_importance'] > 10 else ""
        print(f"  {row['feature']:30s} {row['adv_importance']:6.2f}{flag}")

    # Quick AUC check
    preds = clf.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, preds)
    print(f"  Train-vs-test AUC: {auc:.4f} (0.50=identical, >0.60=concerning)")

    return imp_df


# ---------------------------------------------------------------------------
# TRAINING — CatBoost only, two configs, heavy regularization
# ---------------------------------------------------------------------------
def train_catboost_ensemble(train_feat, test_feat, feat_cols, target):
    """
    Two CatBoost configs:
      A: depth=4, broader (all features)
      B: depth=3, tighter (pruned features — top importance only)
    Both with heavy regularization.
    """
    from catboost import CatBoostRegressor, Pool
    from sklearn.model_selection import KFold

    # Pruned features for model B: drop the 5 least important from Code 20
    # Keep ~17 features
    pruned_cols = [c for c in feat_cols if c not in [
        'baseline_chronic', 'zip_region', 'sex_encoded',
        'n_unique_codes', 'log_visits',
    ]]

    models_config = {
        'A_d4': {
            'cols': feat_cols,
            'params': dict(
                iterations=2000,
                depth=4,
                learning_rate=0.04,
                l2_leaf_reg=5,
                rsm=0.65,
                subsample=0.75,
                min_data_in_leaf=25,
                loss_function='MAE',
                eval_metric='MAE',
                early_stopping_rounds=150,
                use_best_model=True,
                verbose=0,
                random_seed=None,  # set per seed
            ),
        },
        'B_d3': {
            'cols': pruned_cols,
            'params': dict(
                iterations=2000,
                depth=3,
                learning_rate=0.035,
                l2_leaf_reg=8,
                rsm=0.65,
                subsample=0.75,
                min_data_in_leaf=30,
                loss_function='MAE',
                eval_metric='MAE',
                early_stopping_rounds=150,
                use_best_model=True,
                verbose=0,
                random_seed=None,
            ),
        },
    }

    model_names = list(models_config.keys())
    print(f"\n[training] CatBoost CPU | heavy regularization | multi-seed | {N_FOLDS}-fold")
    print(f"Models: {model_names}")
    print(f"  A_d4: {len(feat_cols)} features, depth=4, rsm=0.65, l2=5, min_leaf=25")
    print(f"  B_d3: {len(pruned_cols)} features, depth=3, rsm=0.65, l2=8, min_leaf=30")
    print(f"Seeds={N_SEEDS}, Folds={N_FOLDS}")

    X_train = train_feat[feat_cols].values
    X_test = test_feat[feat_cols].values
    y = target.values

    # Storage
    oof_preds = {m: np.zeros(len(X_train)) for m in model_names}
    test_preds = {m: np.zeros(len(X_test)) for m in model_names}
    seed_oof_maes = {m: [] for m in model_names}
    best_iters = {m: [] for m in model_names}

    for seed_i in range(N_SEEDS):
        seed = 42 + seed_i * 7

        oof_this = {m: np.zeros(len(X_train)) for m in model_names}
        test_this = {m: np.zeros(len(X_test)) for m in model_names}

        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

        for fold_i, (tr_idx, va_idx) in enumerate(kf.split(X_train)):
            for mname, mcfg in models_config.items():
                cols = mcfg['cols']
                col_idx = [feat_cols.index(c) for c in cols]

                X_tr = X_train[np.ix_(tr_idx, col_idx)]
                X_va = X_train[np.ix_(va_idx, col_idx)]
                y_tr, y_va = y[tr_idx], y[va_idx]
                X_te = X_test[:, col_idx]

                params = mcfg['params'].copy()
                params['random_seed'] = seed

                model = CatBoostRegressor(**params)
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va))

                oof_this[mname][va_idx] = model.predict(X_va)
                test_this[mname] += model.predict(X_te) / N_FOLDS
                best_iters[mname].append(model.get_best_iteration())

        for mname in model_names:
            mae = np.mean(np.abs(oof_this[mname] - y))
            seed_oof_maes[mname].append(mae)
            oof_preds[mname] += oof_this[mname] / N_SEEDS
            test_preds[mname] += test_this[mname] / N_SEEDS

        seed_str = ' | '.join(f"{m}={seed_oof_maes[m][-1]:.2f}" for m in model_names)
        print(f"  Seed {seed_i+1}/{N_SEEDS} OOF MAE: {seed_str}")

    # Seed-aggregated OOF MAE (trimmed mean — drop best and worst)
    print("\n[seed-aggregated OOF MAE per model] (trimmed mean)")
    for mname in model_names:
        vals = sorted(seed_oof_maes[mname])
        if len(vals) >= 5:
            trimmed = vals[1:-1]  # drop best and worst
        else:
            trimmed = vals
        print(f"  {mname:15s}: {np.mean(trimmed):.2f}")

    # Median best iteration
    print("\n[median best_iteration] (ref)")
    for mname in model_names:
        print(f"  {mname:15s}: {int(np.median(best_iters[mname]))}")

    return oof_preds, test_preds, model_names, feat_cols, pruned_cols


# ---------------------------------------------------------------------------
# ENSEMBLE SEARCH
# ---------------------------------------------------------------------------
def ensemble_search(oof_preds, target, model_names):
    """Grid search over weights for two models."""
    y = target.values
    results = []

    for wA in np.arange(0.0, 1.05, 0.05):
        wB = 1.0 - wA
        if wB < -0.01:
            continue

        oof_blend = wA * oof_preds[model_names[0]] + wB * oof_preds[model_names[1]]

        # Per-fold MAE for stability metric
        from sklearn.model_selection import KFold
        fold_maes = []
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        for _, va_idx in kf.split(oof_blend):
            fold_maes.append(np.mean(np.abs(oof_blend[va_idx] - y[va_idx])))

        mean_mae = np.mean(fold_maes)
        std_mae = np.std(fold_maes)
        # Penalize complexity (deviation from equal weights)
        lam = abs(wA - 0.5)
        obj = mean_mae + 0.2 * std_mae + 6 * lam

        results.append({
            'wA': round(wA, 2), 'wB': round(wB, 2), 'lam': round(lam, 2),
            'mean_mae': mean_mae, 'std_mae': std_mae, 'obj': obj,
        })

    results.sort(key=lambda x: x['obj'])

    print("\n[ensemble search] Top 8 (obj=mean+0.2*std+6*lam):")
    for i, r in enumerate(results[:8]):
        print(f"  #{i+1:02d} obj={r['obj']:.3f} mean={r['mean_mae']:.3f} "
              f"std={r['std_mae']:.3f} | wA={r['wA']:.2f} wB={r['wB']:.2f} | lam={r['lam']:.2f}")

    best = results[0]
    print(f"\n[chosen ensemble] {best}")
    return best


# ---------------------------------------------------------------------------
# SHIFT TUNING (diagnostic — NOT applied to final submission)
# ---------------------------------------------------------------------------
def compute_shifts(oof_preds_blend, target, train_feat):
    """
    Compute potential chronic/flag shifts via cross-fitting.
    Report but DO NOT apply — the improvement is noise-level on n=2000.
    """
    y = target.values
    base_mae = np.mean(np.abs(oof_preds_blend - y))

    # Chronic shifts
    chronic_map_inv = {0: 'Pneumonia', 1: 'DiabetesComp', 2: 'HF'}
    print(f"\n[shift analysis] (DIAGNOSTIC ONLY — not applied to submission)")
    print(f"  Base OOF MAE: {base_mae:.3f}")

    best_cf, best_mae = 0.0, base_mae
    for cf in np.arange(0, 0.55, 0.05):
        shifted = oof_preds_blend.copy()
        for code, name in chronic_map_inv.items():
            mask = train_feat['chronic_encoded'].values == code
            residuals = y[mask] - shifted[mask]
            median_res = np.median(residuals)
            shifted[mask] += cf * median_res
        mae = np.mean(np.abs(shifted - y))
        if mae < best_mae:
            best_mae = mae
            best_cf = cf

    print(f"  Best chronic_factor={best_cf:.2f} → MAE={best_mae:.3f} (delta={best_mae - base_mae:.3f})")
    print(f"  ⚠️  Delta is {abs(best_mae - base_mae):.3f} — {'applied' if abs(best_mae - base_mae) > 1.0 else 'NOT applied (noise-level)'}")

    return best_cf, best_mae


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    print("=" * 100)
    print("CODE 21 | Gap-closer: 22 features, heavy reg, no shifts | goal: beat LB 449.0")
    print("=" * 100)

    # --- Load data ---
    print("\n[load] reading CSVs...")
    train = pd.read_csv(os.path.join(BASE_DIR, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(BASE_DIR, 'ed_cost_test.csv'))
    patients = pd.read_csv(os.path.join(BASE_DIR, 'patients.csv'))
    target = train['ed_cost_next3y_usd']

    print(f"[ed_cost_train] shape={train.shape}")
    print(f"[ed_cost_test] shape={test.shape}")
    print(f"[patients] shape={patients.shape}")

    print(f"\n[target stats]")
    print(target.describe().to_string())

    # --- Admissions ---
    print("\n[admissions] building aggregates...")
    adm_feats = build_admissions_features(BASE_DIR)

    # --- Receipts (lean — 5 features only) ---
    print("\n[receipts] building LEAN receipt features (5 only)...")
    all_pids = pd.concat([train['patient_id'], test['patient_id']]).unique()
    receipt_feats = extract_all_receipts(all_pids, RECEIPTS_DIR)
    if receipt_feats is not None:
        print(f"  receipt_feat shape: {receipt_feats.shape} | n_features={len(receipt_feats.columns)-1}")

    # --- Build features ---
    print("\n[features] building train/test feature frames...")
    train_feat = build_features(train, patients, adm_feats, receipt_feats)
    test_feat = build_features(test, patients, adm_feats, receipt_feats)

    # Filter to only available feature columns
    available_cols = [c for c in FEATURE_COLS if c in train_feat.columns]
    missing_cols = [c for c in FEATURE_COLS if c not in train_feat.columns]
    if missing_cols:
        print(f"  ⚠️  Missing features (skipped): {missing_cols}")
    print(f"  Feature count: {len(available_cols)}")

    # Fill any remaining NaNs
    for c in available_cols:
        med = train_feat[c].median()
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    # --- Adversarial validation ---
    adv_imp = adversarial_validation(train_feat, test_feat, available_cols)

    # --- Check for high-risk features ---
    high_risk = adv_imp[adv_imp['adv_importance'] > 15]['feature'].tolist()
    if high_risk:
        print(f"\n[WARNING] High adversarial importance features: {high_risk}")
        print("  Consider dropping these if LB gap remains large.")

    # --- Train ---
    oof_preds, test_preds, model_names, _, pruned_cols = train_catboost_ensemble(
        train_feat, test_feat, available_cols, target
    )

    # --- Ensemble ---
    best_ens = ensemble_search(oof_preds, target, model_names)

    wA = best_ens['wA']
    wB = best_ens['wB']
    oof_blend = wA * oof_preds[model_names[0]] + wB * oof_preds[model_names[1]]
    test_blend = wA * test_preds[model_names[0]] + wB * test_preds[model_names[1]]

    base_oof_mae = np.mean(np.abs(oof_blend - target.values))
    print(f"\n[base ensemble]")
    print(f"  OOF MAE: {base_oof_mae:.3f}")
    pq = np.percentile(test_blend, [0, 1, 5, 10, 50, 90, 95, 99, 100])
    print(f"  pred quantiles: {{0: {pq[0]:.1f}, 0.01: {pq[1]:.1f}, 0.05: {pq[2]:.1f}, "
          f"0.10: {pq[3]:.1f}, 0.50: {pq[4]:.1f}, 0.90: {pq[5]:.1f}, "
          f"0.95: {pq[6]:.1f}, 0.99: {pq[7]:.1f}, 1.0: {pq[8]:.1f}}}")

    # --- Shift analysis (diagnostic) ---
    compute_shifts(oof_blend, target, train_feat)

    # --- Feature importance ---
    print("\n[feature importance] fit reference model on full train...")
    from catboost import CatBoostRegressor
    ref_model = CatBoostRegressor(
        iterations=800, depth=4, learning_rate=0.04,
        l2_leaf_reg=5, rsm=0.65, subsample=0.75,
        min_data_in_leaf=25, loss_function='MAE',
        verbose=0, random_seed=42
    )
    ref_model.fit(train_feat[available_cols], target)
    imp = ref_model.get_feature_importance()
    imp_df = pd.DataFrame({
        'feature': available_cols,
        'importance': imp
    }).sort_values('importance', ascending=False)
    print(imp_df.head(12).to_string(index=False))

    # --- Submission (NO shifts applied) ---
    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_blend, 2)
    })

    # Sanity checks
    print(f"\n[SUBMISSION sanity checks]")
    print(f"submission shape: {sub.shape}")
    print(f"submission columns exactly: {sub.columns.tolist()}")
    print(f"any NaNs in preds: {sub['ed_cost_next3y_usd'].isna().any()}")
    preds = sub['ed_cost_next3y_usd']
    print(f"pred min/median/max: {preds.min():.2f} {preds.median():.2f} {preds.max():.2f}")

    sub.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved submission to: {OUTPUT_PATH}")

    print(f"\n{'='*80}")
    print(f"  CODE 21 SUMMARY")
    print(f"  Features: {len(available_cols)} (was 49 in Code 20)")
    print(f"  OOF MAE:  {base_oof_mae:.3f} (was 428.3 in Code 20)")
    print(f"  Target:   OOF ~435-442, LB ~443-448 (gap ≤10)")
    print(f"  Shifts:   NOT applied (noise-level on n=2000)")
    print(f"{'='*80}")
    print(f"\nPaste back: (1) LB MAE, (2) OOF MAE, (3) ensemble weights, (4) adv validation AUC")


if __name__ == '__main__':
    main()

CODE 21 | Gap-closer: 22 features, heavy reg, no shifts | goal: beat LB 449.0

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5)
[ed_cost_test] shape=(2000, 4)
[patients] shape=(4000, 5)

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building aggregates...
  [admissions] 4000 patients, 4 features

[receipts] building LEAN receipt features (5 only)...
  [receipts] backend=PyMuPDF
    1000/4000 processed (found=1000)
    2000/4000 processed (found=2000)
    3000/4000 processed (found=3000)
    4000/4000 processed (found=4000)
  [receipts] extracted: 4000/4000
  receipt_feat shape: (4000, 6) | n_features=5

[features] building train/test feature frames...
  Feature count: 22

[adversarial validation] Feature importance (train vs test distinguishability):
  (High = distribution shift between train/test → overfit risk)
  age             

# Iteration 18
- 459.4645

In [37]:
"""
====================================================================================================
CODE 22 | Code20 base + surgical pruning + LGB diversity | goal: beat LB 449.0
====================================================================================================
Lesson learned from Code 21 (LB 470.8): aggressive pruning DESTROYED signal.
  Code 20 (LB 449.0) is our best. Make SMALL targeted changes only.

Changes from Code 20:
  1. Receipt features: 44 → ~30 (drop only bottom-importance has_* flags, keep
     structural features that clearly help).
  2. Regularization: MILD bump only (rsm 0.8→0.75, l2_leaf_reg 0→2, subsample same).
     Depth stays 5/4. DO NOT go to depth 3.
  3. Add LightGBM as 3rd model for genuine ensemble diversity
     (different algo = decorrelated errors → reduces gap).
  4. Keep cross-fitted shifts (they helped in Code 20).
  5. Adversarial validation diagnostic (report only, don't auto-drop).
  6. 3-model ensemble with stability-penalized weight search.

Expected: OOF ~430-433, LB ~445-448 (gap ≤ 15).
====================================================================================================
"""

import pandas as pd
import numpy as np
import re
import os
import sys
import warnings
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# CONFIG — edit paths here
# ---------------------------------------------------------------------------
BASE_DIR = r'D:\AgentDs\agent_ds_healthcare'
RECEIPTS_DIR = os.path.join(BASE_DIR, 'receipts_pdf')
OUTPUT_PATH = os.path.join(BASE_DIR, 'submission.csv')

N_SEEDS = 5
N_FOLDS = 7
VERBOSE = True

# ---------------------------------------------------------------------------
# PDF PARSING — SAME as Code 20 (don't touch what works)
# ---------------------------------------------------------------------------
try:
    import fitz
    _HAS_FITZ = True
except ImportError:
    _HAS_FITZ = False

try:
    import pdfplumber
    _HAS_PDFPLUMBER = True
except ImportError:
    _HAS_PDFPLUMBER = False

LINE_ITEM_RE = re.compile(
    r'(\d{5})\s+'
    r'(.+?)\s+'
    r'(\d+)\s+'
    r'([\d,]+\.\d{2})\s+'
    r'([\d,]+\.\d{2})'
)


def _get_pdf_text(pdf_path):
    if _HAS_FITZ:
        doc = fitz.open(pdf_path)
        text = '\n'.join(page.get_text() for page in doc)
        doc.close()
        return text
    elif _HAS_PDFPLUMBER:
        with pdfplumber.open(pdf_path) as pdf:
            return '\n'.join(p.extract_text() or '' for p in pdf.pages)
    return ''


def _parse_receipt(text):
    items = []
    for m in LINE_ITEM_RE.finditer(text):
        code = m.group(1)
        qty = int(m.group(3))
        unit_price = float(m.group(4).replace(',', ''))
        line_total = float(m.group(5).replace(',', ''))
        items.append({
            'code': code, 'qty': qty,
            'unit_price': unit_price, 'total': line_total,
        })
    return items


def _categorize_code(code):
    """Categorize CPT/HCPCS code."""
    if code in ('99281', '99282', '99283', '99284', '99285'):
        return 'em', int(code[-1])
    if code in ('99291', '99292'):
        return 'critical_care', 0
    if code.startswith('G037'):
        return 'observation', 0
    if code in ('31500', '36556', '32551', '36620', '92950'):
        return 'high_acuity_proc', 0
    try:
        cn = int(code)
        if 80000 <= cn <= 89999: return 'lab', 0
        if 70000 <= cn <= 79999: return 'imaging', 0
        if 10000 <= cn <= 69999: return 'procedure', 0
    except ValueError:
        pass
    return 'other', 0


def extract_receipt_features(pdf_path):
    """
    Receipt features — MODERATE set.
    Keep structural/aggregate + clinically meaningful code groups.
    Drop only the rarest individual has_* flags.
    """
    text = _get_pdf_text(pdf_path)
    items = _parse_receipt(text)
    if not items:
        return None

    total = sum(it['total'] for it in items) or 1.0
    unit_prices = [it['unit_price'] for it in items if it['unit_price'] > 0]
    codes = [it['code'] for it in items]
    code_set = set(codes)

    f = {}

    # --- Structural features (keep all — these generalize) ---
    f['n_line_items'] = len(items)
    f['n_unique_codes'] = len(code_set)

    shares = [it['total'] / total for it in items]
    f['cost_hhi'] = sum(s ** 2 for s in shares)

    f['log1p_max_line_total'] = np.log1p(max(it['total'] for it in items))
    f['log1p_median_unit_price'] = np.log1p(np.median(unit_prices)) if unit_prices else 0
    f['log1p_mean_unit_price'] = np.log1p(np.mean(unit_prices)) if unit_prices else 0
    f['cost_per_code'] = total / max(len(code_set), 1)

    # --- Clinical categorization (aggregated, not individual flags) ---
    em_levels = []
    n_lab = 0
    n_imaging = 0
    n_proc = 0
    cost_em = 0
    cost_critical = 0

    for it in items:
        cat, level = _categorize_code(it['code'])
        if cat == 'em':
            em_levels.append(level)
            cost_em += it['total']
        elif cat == 'critical_care':
            cost_critical += it['total']
        elif cat == 'lab':
            n_lab += 1
        elif cat == 'imaging':
            n_imaging += 1
        elif cat == 'procedure':
            n_proc += 1

    f['max_em_level'] = max(em_levels) if em_levels else 0
    f['avg_em_level'] = np.mean(em_levels) if em_levels else 0
    f['n_em_codes'] = len(em_levels)
    f['pct_cost_em'] = cost_em / total
    f['has_critical_care'] = int(cost_critical > 0)
    f['pct_cost_critical'] = cost_critical / total

    # Category counts (aggregated — not individual code flags)
    f['n_lab_codes'] = n_lab
    f['n_imaging_codes'] = n_imaging
    f['n_procedure_codes'] = n_proc

    # --- HIGH-VALUE individual flags ONLY (keep top ones from Code 20 importance) ---
    # Only keep flags that had importance > 2.0 in Code 20
    f['has_99285'] = int('99285' in code_set)        # highest E&M level
    f['has_critical_care_code'] = int(
        bool(code_set & {'99291', '99292'})
    )
    f['has_obs_G0378'] = int(any(c.startswith('G037') for c in code_set))
    f['has_troponin_84484'] = int('84484' in code_set)  # cardiac workup
    f['has_ct_head_70450'] = int('70450' in code_set)    # head CT

    return f


def extract_all_receipts(patient_ids, receipts_dir):
    if receipts_dir is None or not os.path.exists(receipts_dir):
        print(f"  [receipts] directory not found: {receipts_dir}")
        return None
    if not _HAS_FITZ and not _HAS_PDFPLUMBER:
        print("  [receipts] ERROR: no PDF library!")
        return None

    print(f"  [receipts] backend={'PyMuPDF' if _HAS_FITZ else 'pdfplumber'}")
    all_feats = {}
    found = 0

    for i, pid in enumerate(patient_ids):
        path = os.path.join(receipts_dir, f"receipt_{pid}.pdf")
        if os.path.exists(path):
            feats = extract_receipt_features(path)
            if feats:
                all_feats[pid] = feats
                found += 1
        if (i + 1) % 1000 == 0:
            print(f"    {i+1}/{len(patient_ids)} processed (found={found})")

    print(f"  [receipts] extracted: {found}/{len(patient_ids)}")

    if all_feats:
        df = pd.DataFrame.from_dict(all_feats, orient='index')
        df.index.name = 'patient_id'
        return df.reset_index()
    return None


# ---------------------------------------------------------------------------
# ADMISSIONS AGGREGATES — same as Code 20
# ---------------------------------------------------------------------------
def build_admissions_features(base_dir):
    dfs = []
    for fname in ['admissions_train.csv', 'admissions_test.csv']:
        path = os.path.join(base_dir, fname)
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    if not dfs:
        print("  [admissions] NOT FOUND")
        return None

    adm = pd.concat(dfs, ignore_index=True)
    adm = adm.drop(columns=['readmit_30d'], errors='ignore')

    agg = adm.groupby('patient_id').agg(
        charlson_max=('charlson_band', 'max'),
        pct_emergent=('acuity_emergent', 'mean'),
        n_admissions=('admission_id', 'count'),
        max_ed_visits_6m=('ed_visits_6m', 'max'),
    ).reset_index()

    print(f"  [admissions] {len(agg)} patients, 4 features")
    return agg


# ---------------------------------------------------------------------------
# FEATURE ENGINEERING
# ---------------------------------------------------------------------------
def build_features(df, patients, adm_feats, receipt_feats):
    feat = df.copy()

    # --- Core ---
    chronic_map = {'Pneumonia': 0, 'DiabetesComp': 1, 'HF': 2}
    feat['chronic_encoded'] = feat['primary_chronic'].map(chronic_map)

    # --- Cost transforms (same as Code 20 — proven) ---
    feat['sqrt_prior_cost'] = np.sqrt(feat['prior_ed_cost_5y_usd'])
    feat['log_prior_cost'] = np.log1p(feat['prior_ed_cost_5y_usd'])
    feat['prior_cost_cap20k'] = feat['prior_ed_cost_5y_usd'].clip(upper=20000)
    feat['log_prior_cost_cap20k'] = np.log1p(feat['prior_cost_cap20k'])
    feat['cost_per_visit'] = (feat['prior_ed_cost_5y_usd']
                              / feat['prior_ed_visits_5y'].clip(1))
    feat['cost_per_code'] = feat['prior_ed_cost_5y_usd']  # placeholder, overwritten if receipts

    # --- Visit transforms ---
    feat['log_visits'] = np.log1p(feat['prior_ed_visits_5y'])

    # --- Baseline (Code 20 had this — keep it) ---
    # Chronic-stratified regression-to-mean baseline
    chronic_stats = {0: 3300, 1: 4100, 2: 4500}
    feat['baseline_next3y'] = feat['chronic_encoded'].map(chronic_stats)

    # --- Demographics ---
    if patients is not None:
        p = patients.copy()
        p['sex_encoded'] = (p['sex'] == 'M').astype(int)
        ins_map = {'private': 2, 'public': 1, 'self_pay': 0}
        p['insurance_encoded'] = p['insurance'].map(ins_map).fillna(-1).astype(float)
        p['zip_region'] = p['zip3'].astype(str).str[0].astype(float)
        feat = feat.merge(
            p[['patient_id', 'age', 'sex_encoded', 'insurance_encoded', 'zip_region']],
            on='patient_id', how='left'
        )

    # --- Admissions ---
    if adm_feats is not None:
        feat = feat.merge(adm_feats, on='patient_id', how='left')
        for c in ['charlson_max', 'pct_emergent', 'n_admissions', 'max_ed_visits_6m']:
            if c in feat.columns:
                feat[c] = feat[c].fillna(feat[c].median())

    # --- Receipts ---
    if receipt_feats is not None:
        feat = feat.merge(receipt_feats, on='patient_id', how='left')
        rcpt_cols = [c for c in receipt_feats.columns if c != 'patient_id']
        for c in rcpt_cols:
            if c in feat.columns:
                feat[c] = feat[c].fillna(feat[c].median())
        # Update cost_per_code with actual receipt data
        if 'n_unique_codes' in feat.columns:
            feat['cost_per_code'] = (feat['prior_ed_cost_5y_usd']
                                     / feat['n_unique_codes'].clip(1))

    return feat


# ---------------------------------------------------------------------------
# ADVERSARIAL VALIDATION (diagnostic)
# ---------------------------------------------------------------------------
def adversarial_validation(train_feat, test_feat, feat_cols):
    from catboost import CatBoostClassifier
    from sklearn.metrics import roc_auc_score

    X = pd.concat([train_feat[feat_cols], test_feat[feat_cols]], ignore_index=True)
    y = np.array([0] * len(train_feat) + [1] * len(test_feat))

    clf = CatBoostClassifier(
        iterations=200, depth=4, learning_rate=0.05,
        verbose=0, random_seed=42
    )
    clf.fit(X, y)

    importances = clf.get_feature_importance()
    imp_df = pd.DataFrame({
        'feature': feat_cols, 'adv_importance': importances
    }).sort_values('adv_importance', ascending=False)

    preds = clf.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, preds)

    print("\n[adversarial validation] Top features distinguishing train vs test:")
    for _, row in imp_df.head(10).iterrows():
        flag = " ⚠️" if row['adv_importance'] > 12 else ""
        print(f"  {row['feature']:30s} {row['adv_importance']:6.2f}{flag}")
    print(f"  Train-vs-test AUC: {auc:.4f} (0.50=identical, >0.65=concerning)")

    return imp_df, auc


# ---------------------------------------------------------------------------
# TRAINING — CatBoost (2 configs) + LightGBM (1 config)
# ---------------------------------------------------------------------------
def train_models(train_feat, test_feat, feat_cols, target):
    from catboost import CatBoostRegressor
    from sklearn.model_selection import KFold

    try:
        import lightgbm as lgb
        _HAS_LGB = True
    except ImportError:
        _HAS_LGB = False
        print("  [WARNING] LightGBM not available, using CatBoost-only")

    # Pruned cols for model B (drop bottom ~8 by expected importance)
    pruned_drop = ['baseline_next3y', 'zip_region', 'sex_encoded',
                   'log_visits', 'n_imaging_codes', 'pct_cost_critical',
                   'has_critical_care_code', 'has_obs_G0378']
    pruned_cols = [c for c in feat_cols if c not in pruned_drop]

    X_train_full = train_feat[feat_cols].values
    X_test_full = test_feat[feat_cols].values
    y = target.values

    # --- Model configs ---
    models_config = {}

    # Model A: CatBoost depth=5, full features (same as Code 20 A but RSM 0.75)
    models_config['A_cat_d5'] = {
        'type': 'catboost',
        'cols': feat_cols,
        'params': dict(
            iterations=2000, depth=5, learning_rate=0.04,
            l2_leaf_reg=2, rsm=0.75, subsample=0.8,
            min_data_in_leaf=15,
            loss_function='MAE', eval_metric='MAE',
            early_stopping_rounds=150, use_best_model=True,
            verbose=0, random_seed=None,
        ),
    }

    # Model B: CatBoost depth=4, pruned features (same as Code 20 B but RSM 0.75)
    models_config['B_cat_d4'] = {
        'type': 'catboost',
        'cols': pruned_cols,
        'params': dict(
            iterations=2000, depth=4, learning_rate=0.04,
            l2_leaf_reg=3, rsm=0.75, subsample=0.8,
            min_data_in_leaf=20,
            loss_function='MAE', eval_metric='MAE',
            early_stopping_rounds=150, use_best_model=True,
            verbose=0, random_seed=None,
        ),
    }

    # Model C: LightGBM — different algorithm family for diversity
    if _HAS_LGB:
        models_config['C_lgb'] = {
            'type': 'lgb',
            'cols': feat_cols,
            'params': dict(
                n_estimators=2000, max_depth=5, learning_rate=0.04,
                num_leaves=20,  # conservative
                reg_alpha=0.5, reg_lambda=2.0,
                colsample_bytree=0.75, subsample=0.8,
                subsample_freq=1, min_child_samples=25,
                objective='mae', metric='mae',
                verbose=-1, random_state=None,
                n_jobs=-1,
            ),
        }

    model_names = list(models_config.keys())
    print(f"\n[training] Models: {model_names}")
    for mname, mcfg in models_config.items():
        print(f"  {mname}: {mcfg['type']}, {len(mcfg['cols'])} features, "
              f"depth={mcfg['params'].get('depth', mcfg['params'].get('max_depth', '?'))}")
    print(f"Seeds={N_SEEDS}, Folds={N_FOLDS}")

    # Storage
    oof_preds = {m: np.zeros(len(y)) for m in model_names}
    test_preds = {m: np.zeros(len(X_test_full)) for m in model_names}
    seed_oof_maes = {m: [] for m in model_names}
    best_iters = {m: [] for m in model_names}

    for seed_i in range(N_SEEDS):
        seed = 42 + seed_i * 7

        oof_this = {m: np.zeros(len(y)) for m in model_names}
        test_this = {m: np.zeros(len(X_test_full)) for m in model_names}

        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

        for fold_i, (tr_idx, va_idx) in enumerate(kf.split(X_train_full)):
            for mname, mcfg in models_config.items():
                cols = mcfg['cols']
                col_idx = [feat_cols.index(c) for c in cols]

                X_tr = X_train_full[np.ix_(tr_idx, col_idx)]
                X_va = X_train_full[np.ix_(va_idx, col_idx)]
                y_tr, y_va = y[tr_idx], y[va_idx]
                X_te = X_test_full[:, col_idx]

                if mcfg['type'] == 'catboost':
                    params = mcfg['params'].copy()
                    params['random_seed'] = seed
                    model = CatBoostRegressor(**params)
                    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
                    oof_this[mname][va_idx] = model.predict(X_va)
                    test_this[mname] += model.predict(X_te) / N_FOLDS
                    best_iters[mname].append(model.get_best_iteration())

                elif mcfg['type'] == 'lgb':
                    params = mcfg['params'].copy()
                    params['random_state'] = seed
                    n_est = params.pop('n_estimators')
                    dtrain = lgb.Dataset(X_tr, y_tr)
                    dval = lgb.Dataset(X_va, y_va, reference=dtrain)

                    callbacks = [
                        lgb.early_stopping(150, verbose=False),
                        lgb.log_evaluation(0),
                    ]
                    model = lgb.train(
                        params, dtrain, num_boost_round=n_est,
                        valid_sets=[dval], callbacks=callbacks,
                    )
                    oof_this[mname][va_idx] = model.predict(X_va)
                    test_this[mname] += model.predict(X_te) / N_FOLDS
                    best_iters[mname].append(model.best_iteration)

        for mname in model_names:
            mae = np.mean(np.abs(oof_this[mname] - y))
            seed_oof_maes[mname].append(mae)
            oof_preds[mname] += oof_this[mname] / N_SEEDS
            test_preds[mname] += test_this[mname] / N_SEEDS

        seed_str = ' | '.join(f"{m}={seed_oof_maes[m][-1]:.2f}" for m in model_names)
        print(f"  Seed {seed_i+1}/{N_SEEDS} OOF MAE: {seed_str}")

    # Summary
    print("\n[seed-aggregated OOF MAE per model] (trimmed mean)")
    for mname in model_names:
        vals = sorted(seed_oof_maes[mname])
        trimmed = vals[1:-1] if len(vals) >= 5 else vals
        print(f"  {mname:15s}: {np.mean(trimmed):.2f}")

    print("\n[median best_iteration]")
    for mname in model_names:
        print(f"  {mname:15s}: {int(np.median(best_iters[mname]))}")

    return oof_preds, test_preds, model_names


# ---------------------------------------------------------------------------
# ENSEMBLE SEARCH — now 3 models
# ---------------------------------------------------------------------------
def ensemble_search(oof_preds, target, model_names):
    y = target.values
    results = []

    # Grid over 3 weights (step 0.05)
    n_models = len(model_names)
    if n_models == 3:
        for wA in np.arange(0.0, 1.05, 0.05):
            for wB in np.arange(0.0, 1.05 - wA, 0.05):
                wC = round(1.0 - wA - wB, 2)
                if wC < -0.01:
                    continue
                blend = (wA * oof_preds[model_names[0]]
                         + wB * oof_preds[model_names[1]]
                         + wC * oof_preds[model_names[2]])
                mae = np.mean(np.abs(blend - y))
                # Penalize deviation from equal weights
                lam = sum(abs(w - 1/3) for w in [wA, wB, wC]) / 3
                obj = mae + 4 * lam
                results.append({
                    'wA': round(wA, 2), 'wB': round(wB, 2), 'wC': round(wC, 2),
                    'mae': mae, 'lam': round(lam, 3), 'obj': obj,
                })
    else:
        # 2-model fallback
        for wA in np.arange(0.0, 1.05, 0.05):
            wB = round(1.0 - wA, 2)
            blend = wA * oof_preds[model_names[0]] + wB * oof_preds[model_names[1]]
            mae = np.mean(np.abs(blend - y))
            lam = abs(wA - 0.5)
            obj = mae + 6 * lam
            results.append({
                'wA': round(wA, 2), 'wB': round(wB, 2),
                'mae': mae, 'lam': round(lam, 3), 'obj': obj,
            })

    results.sort(key=lambda x: x['obj'])

    print("\n[ensemble search] Top 8:")
    for i, r in enumerate(results[:8]):
        weights = ' '.join(f"w{chr(65+j)}={r.get(f'w{chr(65+j)}', 0):.2f}"
                           for j in range(n_models))
        print(f"  #{i+1:02d} obj={r['obj']:.3f} mae={r['mae']:.3f} | {weights}")

    best = results[0]
    print(f"\n[chosen ensemble] {best}")
    return best


# ---------------------------------------------------------------------------
# SHIFT TUNING (cross-fitted, applied only if > 1.0 improvement)
# ---------------------------------------------------------------------------
def compute_and_apply_shifts(oof_blend, test_blend, target, train_feat, test_feat):
    y = target.values
    base_mae = np.mean(np.abs(oof_blend - y))

    chronic_map_inv = {0: 'Pneumonia', 1: 'DiabetesComp', 2: 'HF'}

    print(f"\n[shift tuning] cross-fitted")
    print(f"  Base OOF MAE: {base_mae:.3f}")

    # Grid search chronic_factor
    best_cf, best_mae = 0.0, base_mae
    results = []

    for cf in np.arange(0, 0.55, 0.05):
        shifted = oof_blend.copy()
        for code in range(3):
            mask = train_feat['chronic_encoded'].values == code
            residuals = y[mask] - shifted[mask]
            median_res = np.median(residuals)
            shifted[mask] += cf * median_res
        mae = np.mean(np.abs(shifted - y))
        results.append((cf, mae))
        if mae < best_mae:
            best_mae = mae
            best_cf = cf

    results.sort(key=lambda x: x[1])
    print("  Top 5:")
    for cf, mae in results[:5]:
        print(f"    cf={cf:.2f} → MAE={mae:.3f}")

    delta = base_mae - best_mae
    print(f"\n  Best: cf={best_cf:.2f}, delta={delta:.3f}")

    # Only apply if delta > 1.0 (meaningful, not noise)
    if delta > 1.0:
        print(f"  ✅ Applying shifts (delta={delta:.3f} > 1.0 threshold)")
        oof_shifted = oof_blend.copy()
        test_shifted = test_blend.copy()
        shift_report = {}

        for code, name in chronic_map_inv.items():
            # OOF
            mask_tr = train_feat['chronic_encoded'].values == code
            residuals = y[mask_tr] - oof_blend[mask_tr]
            median_res = np.median(residuals)
            shift = best_cf * median_res
            oof_shifted[mask_tr] += shift

            # Test
            mask_te = test_feat['chronic_encoded'].values == code
            test_shifted[mask_te] += shift
            shift_report[name] = shift

        final_mae = np.mean(np.abs(oof_shifted - y))
        print(f"  Shifts: {shift_report}")
        print(f"  Final OOF MAE: {final_mae:.3f}")
        return oof_shifted, test_shifted, final_mae
    else:
        print(f"  ❌ NOT applying shifts (delta={delta:.3f} ≤ 1.0 threshold)")
        return oof_blend, test_blend, base_mae


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    print("=" * 100)
    print("CODE 22 | Code20 base + surgical pruning + LGB diversity | goal: beat LB 449.0")
    print("=" * 100)

    # --- Load ---
    print("\n[load] reading CSVs...")
    train = pd.read_csv(os.path.join(BASE_DIR, 'ed_cost_train.csv'))
    test = pd.read_csv(os.path.join(BASE_DIR, 'ed_cost_test.csv'))
    patients = pd.read_csv(os.path.join(BASE_DIR, 'patients.csv'))
    target = train['ed_cost_next3y_usd']

    print(f"[ed_cost_train] shape={train.shape}")
    print(f"[ed_cost_test] shape={test.shape}")
    print(f"[patients] shape={patients.shape}")

    print(f"\n[target stats]")
    print(target.describe().to_string())

    # --- Admissions ---
    print("\n[admissions] building aggregates...")
    adm_feats = build_admissions_features(BASE_DIR)

    # --- Receipts (moderate — ~22 features) ---
    print("\n[receipts] building receipt features (moderate set)...")
    all_pids = pd.concat([train['patient_id'], test['patient_id']]).unique()
    receipt_feats = extract_all_receipts(all_pids, RECEIPTS_DIR)
    if receipt_feats is not None:
        rcpt_feat_cols = [c for c in receipt_feats.columns if c != 'patient_id']
        print(f"  receipt features: {len(rcpt_feat_cols)}: {rcpt_feat_cols}")

    # --- Build features ---
    print("\n[features] building train/test feature frames...")
    train_feat = build_features(train, patients, adm_feats, receipt_feats)
    test_feat = build_features(test, patients, adm_feats, receipt_feats)

    # Determine available features
    all_possible = [
        # Core (3)
        'chronic_encoded', 'prior_ed_visits_5y', 'prior_ed_cost_5y_usd',
        # Cost transforms (5)
        'sqrt_prior_cost', 'log_prior_cost', 'prior_cost_cap20k',
        'log_prior_cost_cap20k', 'cost_per_visit',
        # Visit (1)
        'log_visits',
        # Baseline (1)
        'baseline_next3y',
        # Demographics (4)
        'age', 'sex_encoded', 'insurance_encoded', 'zip_region',
        # Admissions (4)
        'charlson_max', 'pct_emergent', 'n_admissions', 'max_ed_visits_6m',
        # Receipt structural (7)
        'n_line_items', 'n_unique_codes', 'cost_hhi', 'cost_per_code',
        'log1p_max_line_total', 'log1p_median_unit_price', 'log1p_mean_unit_price',
        # Receipt clinical aggregated (6)
        'max_em_level', 'avg_em_level', 'n_em_codes', 'pct_cost_em',
        'has_critical_care', 'pct_cost_critical',
        # Receipt category counts (3)
        'n_lab_codes', 'n_imaging_codes', 'n_procedure_codes',
        # Receipt high-value flags (5)
        'has_99285', 'has_critical_care_code', 'has_obs_G0378',
        'has_troponin_84484', 'has_ct_head_70450',
    ]

    feat_cols = [c for c in all_possible if c in train_feat.columns]
    missing = [c for c in all_possible if c not in train_feat.columns]
    if missing:
        print(f"  ⚠️  Missing: {missing}")
    print(f"  Feature count: {len(feat_cols)}")

    # Fill NaNs
    for c in feat_cols:
        med = train_feat[c].median()
        train_feat[c] = pd.to_numeric(train_feat[c], errors='coerce').fillna(med)
        test_feat[c] = pd.to_numeric(test_feat[c], errors='coerce').fillna(med)

    # --- Adversarial validation ---
    adv_imp, adv_auc = adversarial_validation(train_feat, test_feat, feat_cols)

    # --- Train ---
    oof_preds, test_preds, model_names = train_models(
        train_feat, test_feat, feat_cols, target
    )

    # --- Ensemble ---
    best_ens = ensemble_search(oof_preds, target, model_names)

    # Build blended predictions
    oof_blend = sum(best_ens.get(f'w{chr(65+i)}', 0) * oof_preds[m]
                    for i, m in enumerate(model_names))
    test_blend = sum(best_ens.get(f'w{chr(65+i)}', 0) * test_preds[m]
                     for i, m in enumerate(model_names))

    base_oof_mae = np.mean(np.abs(oof_blend - target.values))
    print(f"\n[base ensemble] OOF MAE: {base_oof_mae:.3f}")

    # --- Shifts (conditional) ---
    oof_final, test_final, final_mae = compute_and_apply_shifts(
        oof_blend, test_blend, target, train_feat, test_feat
    )

    # --- Feature importance ---
    print("\n[feature importance] reference model on full train...")
    from catboost import CatBoostRegressor
    ref = CatBoostRegressor(
        iterations=800, depth=5, learning_rate=0.04,
        l2_leaf_reg=2, rsm=0.75, subsample=0.8,
        loss_function='MAE', verbose=0, random_seed=42
    )
    ref.fit(train_feat[feat_cols], target)
    imp = ref.get_feature_importance()
    imp_df = pd.DataFrame({'feature': feat_cols, 'importance': imp}).sort_values(
        'importance', ascending=False)
    print(imp_df.head(15).to_string(index=False))

    # --- Submission ---
    sub = pd.DataFrame({
        'patient_id': test['patient_id'],
        'ed_cost_next3y_usd': np.round(test_final, 2)
    })

    print(f"\n[SUBMISSION sanity checks]")
    print(f"shape: {sub.shape}")
    print(f"columns: {sub.columns.tolist()}")
    print(f"NaNs: {sub['ed_cost_next3y_usd'].isna().any()}")
    preds = sub['ed_cost_next3y_usd']
    print(f"min/median/max: {preds.min():.2f} {preds.median():.2f} {preds.max():.2f}")
    qs = np.percentile(preds, [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100])
    print(f"quantiles: 0%={qs[0]:.0f} 1%={qs[1]:.0f} 5%={qs[2]:.0f} "
          f"10%={qs[3]:.0f} 25%={qs[4]:.0f} 50%={qs[5]:.0f} "
          f"75%={qs[6]:.0f} 90%={qs[7]:.0f} 95%={qs[8]:.0f} "
          f"99%={qs[9]:.0f} 100%={qs[10]:.0f}")

    sub.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved to: {OUTPUT_PATH}")

    print(f"\n{'='*80}")
    print(f"  CODE 22 SUMMARY")
    print(f"  Features: {len(feat_cols)} (Code 20 had 49, Code 21 had 22)")
    print(f"  Models: {model_names}")
    print(f"  Base OOF MAE: {base_oof_mae:.3f}")
    print(f"  Final OOF MAE: {final_mae:.3f}")
    print(f"  Adv validation AUC: {adv_auc:.4f}")
    print(f"{'='*80}")
    print(f"\nPaste back: (1) LB MAE, (2) OOF MAE, (3) adv AUC, (4) ensemble weights")


if __name__ == '__main__':
    main()

CODE 22 | Code20 base + surgical pruning + LGB diversity | goal: beat LB 449.0

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5)
[ed_cost_test] shape=(2000, 4)
[patients] shape=(4000, 5)

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building aggregates...
  [admissions] 4000 patients, 4 features

[receipts] building receipt features (moderate set)...
  [receipts] backend=PyMuPDF
    1000/4000 processed (found=1000)
    2000/4000 processed (found=2000)
    3000/4000 processed (found=3000)
    4000/4000 processed (found=4000)
  [receipts] extracted: 4000/4000
  receipt features: 21: ['n_line_items', 'n_unique_codes', 'cost_hhi', 'log1p_max_line_total', 'log1p_median_unit_price', 'log1p_mean_unit_price', 'cost_per_code', 'max_em_level', 'avg_em_level', 'n_em_codes', 'pct_cost_em', 'has_critical_care', 'pct_cost_critical', 'n_lab_code

# New Evaluation

# Submission

In [38]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "D:/AgentDs/agent_ds_healthcare/submission.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 459.4645 (MAE)
✅ Validation passed
✅ Submission successful!
   📊 Score: 459.4645
   📏 Metric: MAE
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 3!
